# Test set analysis

In [ ]:
import os, json, requests
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install openai elasticsearch tqdm pandas numpy

In [ ]:
import json
import pandas as pd
from collections import Counter, defaultdict

# Φόρτωση δεδομένων
data = []
file_path = '/content/drive/MyDrive/Project/test_set/rag_taskAC.jsonl'
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.strip()))

df = pd.DataFrame(data)

# ΣΗΜΑΝΤΙΚΟ: Μετατροπή turn σε integer
df['turn_int'] = df['turn'].astype(int)

print("="*70)
print("ΣΥΝΟΛΙΚΗ ΑΝΑΛΥΣΗ TEST SET")
print("="*70)
print(f"Συνολικός αριθμός tasks: {len(data)}")
print(f"Συνολικός αριθμός conversations: {df['conversation_id'].nunique()}")
print(f"Μέσος όρος turns ανά conversation: {df.groupby('conversation_id')['turn_int'].max().mean():.2f}")

# ==================== ΑΝΑΛΥΣΗ ΑΝΑ COLLECTION ====================
print("\n" + "="*70)
print("ΑΝΑΛΥΣΗ ΑΝΑ COLLECTION (Domain)")
print("="*70)

collection_stats = df.groupby('Collection').agg({
    'task_id': 'count',
    'conversation_id': 'nunique',
    'turn_int': 'max'
})
collection_stats.columns = ['Tasks', 'Conversations', 'Max Turn']

# Υπολογισμός μέσου όρου turns ανά conversation ανά collection
avg_turns_per_collection = df.groupby(['Collection', 'conversation_id'])['turn_int'].max().groupby('Collection').mean()
collection_stats['Avg Turns/Conv'] = avg_turns_per_collection.round(2)

print(collection_stats)
print(f"\nΣυνολικά Collections: {df['Collection'].nunique()}")

# ==================== ΑΝΑΛΥΣΗ ΑΝΑ CONVERSATION ====================
print("\n" + "="*70)
print("ΑΝΑΛΥΣΗ ΑΝΑ CONVERSATION")
print("="*70)

conv_stats = df.groupby('conversation_id').agg({
    'task_id': 'count',
    'turn_int': 'max',
    'Collection': 'first'
}).rename(columns={'task_id': 'num_tasks', 'turn_int': 'max_turn', 'Collection': 'domain'})

print(f"Μέσος αριθμός tasks ανά conversation: {conv_stats['num_tasks'].mean():.2f}")
print(f"Min tasks σε conversation: {conv_stats['num_tasks'].min()}")
print(f"Max tasks σε conversation: {conv_stats['num_tasks'].max()}")
print(f"\nΜέσος αριθμός turns ανά conversation: {conv_stats['max_turn'].mean():.2f}")
print(f"Min turns: {conv_stats['max_turn'].min()}")
print(f"Max turns: {conv_stats['max_turn'].max()}")

# Top 10 conversations με περισσότερα turns
print("\n--- Top 10 Conversations με περισσότερα turns ---")
top_convs = conv_stats.nlargest(10, 'max_turn')[['domain', 'max_turn', 'num_tasks']]
print(top_convs.to_string())

# ==================== ΑΝΑΛΥΣΗ ΑΝΑ TURN ====================
print("\n" + "="*70)
print("ΑΝΑΛΥΣΗ ΑΝΑ TURN")
print("="*70)

turn_distribution = df['turn_int'].value_counts().sort_index()

print("Κατανομή tasks ανά turn:")
for turn, count in turn_distribution.items():
    percentage = (count / len(df)) * 100
    print(f"  Turn {turn}: {count} tasks ({percentage:.1f}%)")

# ==================== ΣΥΝΔΥΑΣΤΙΚΗ ΑΝΑΛΥΣΗ ====================
print("\n" + "="*70)
print("ΚΑΤΑΝΟΜΗ TASKS: COLLECTION x TURN")
print("="*70)

pivot_table = pd.crosstab(df['Collection'], df['turn_int'], margins=True, margins_name='Total')
print(pivot_table)

# ==================== ΛΕΠΤΟΜΕΡΗΣ ΑΝΑΛΥΣΗ ΑΝΑ COLLECTION ====================
print("\n" + "="*70)
print("ΛΕΠΤΟΜΕΡΗΣ ΑΝΑΛΥΣΗ ΑΝΑ COLLECTION")
print("="*70)

for collection in sorted(df['Collection'].unique()):
    coll_df = df[df['Collection'] == collection]
    print(f"\n{'='*50}")
    print(f"  {collection}")
    print(f"{'='*50}")
    print(f"  • Tasks: {len(coll_df)}")
    print(f"  • Conversations: {coll_df['conversation_id'].nunique()}")
    print(f"  • Avg turns/conversation: {coll_df.groupby('conversation_id')['turn_int'].max().mean():.2f}")
    print(f"  • Turn range: {coll_df['turn_int'].min()} - {coll_df['turn_int'].max()}")

    # Κατανομή turns
    turn_dist = coll_df['turn_int'].value_counts().sort_index()
    print(f"  • Turn distribution:")
    for t, c in turn_dist.items():
        print(f"      Turn {t}: {c} tasks")

# ==================== ΑΠΟΘΗΚΕΥΣΗ ΣΤΑΤΙΣΤΙΚΩΝ ====================
print("\n" + "="*70)
print("ΑΠΟΘΗΚΕΥΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ")
print("="*70)

collection_stats.to_csv('analysis_by_collection.csv')
conv_stats.to_csv('analysis_by_conversation.csv')
turn_distribution.to_frame('count').to_csv('analysis_by_turn.csv')
pivot_table.to_csv('analysis_collection_x_turn.csv')

# Λεπτομερής πληροφορία για κάθε conversation
detailed_conv_stats = df.groupby(['conversation_id', 'Collection']).agg({
    'task_id': 'count',
    'turn_int': list
}).rename(columns={'task_id': 'num_tasks', 'turn_int': 'turns_list'})
detailed_conv_stats.to_csv('detailed_conversation_analysis.csv')

print("✓ analysis_by_collection.csv")
print("✓ analysis_by_conversation.csv")
print("✓ analysis_by_turn.csv")
print("✓ analysis_collection_x_turn.csv")
print("✓ detailed_conversation_analysis.csv")

print(f"\n{'='*70}")
print("ΑΝΑΛΥΣΗ ΟΛΟΚΛΗΡΩΘΗΚΕ!")
print(f"{'='*70}")


#Minimal Query rewriting

In [ ]:
# ==============================================================================
# ELSER v1 — TEST SET RETRIEVAL (Task A)
# INPUT: rag_taskAC.jsonl
# OUTPUT: Top-100 predictions in format checker format
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from elasticsearch import Elasticsearch
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
INPUT_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"
OUTPUT_FILE = f"{BASE_PATH}/test_set/predictions_minimal_elser_top100.jsonl"
REWRITES_FILE = f"{BASE_PATH}/test_set/minimal_rewrites.json"

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

# ✅ ΔΙΟΡΘΩΜΕΝΟ: Index mapping για τα ΑΠΛΑ collection names
INDEX_MAP = {
    "clapnq": "clapnq_elser_v1",
    "fiqa": "fiqa_elser_v1",
    "govt": "govt_elser_v1",
    "ibmcloud": "cloud_elser_v1"
}

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. STRUCTURAL HISTORY (DAMO-NLP BASED)
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    """XML-structured conversation history"""
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['speaker'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['speaker'] == 'agent' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

# ----------------------------
# 4. LOAD TEST SET
# ----------------------------
def load_test_set(path: str):
    """Load rag_taskAC.jsonl"""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# ----------------------------
# 5. REWRITER
# ----------------------------
class ELSERStructuralRewriter:
    def __init__(self):
        self.name = "elser_minimal_optimized"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "Given the following conversation, please reword the final utterance from the user into "
            "a single utterance that does not need the history to understand the user's intent. "
            "Output in proper JSON format indicating the 'class' (standalone or non-standalone) "
            "and the 'rewritten version' of the last utterance.\n\n"
            "Rules:\n"
            "- Do not do any unnecessary rephrasing or introduction of new terms or concepts.\n"
            "- Be minimal, staying as close as possible to the shape and meaning of the last user utterance.\n"
            "- If the last user utterance is already clear and standalone, the rewritten version should be THE SAME.\n"
            "- Use information from 'assistant' turns ONLY to resolve pronouns (e.g., 'it', 'he', 'that') or clarify ambiguous references.\n\n"
            "Format: {\"class\": \"...\", \"rewritten version\": \"...\"}"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return parsed.get("rewritten version", query)
            return query
        except:
            return query

# ----------------------------
# 6. MAIN PIPELINE
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 TEST SET RETRIEVAL (Task A)")
    print("="*70)
    print(f"Input: {INPUT_FILE}")
    print(f"Output: {OUTPUT_FILE}")
    print(f"Rewrites: {REWRITES_FILE}")
    print(f"Top-K: 100")
    print("="*70)

    # Initialize Elasticsearch
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    # Load test set
    print("\n📂 Loading test set...")
    data = load_test_set(INPUT_FILE)
    print(f"✅ Loaded {len(data)} tasks")

    # Initialize rewriter
    rewriter = ELSERStructuralRewriter()

    # Extract query and history from input
    tasks = []
    for item in data:
        input_turns = item['input']

        # Το τελευταίο turn είναι το query
        query = input_turns[-1]['text']

        # Τα προηγούμενα turns είναι το history
        history = input_turns[:-1]

        tasks.append({
            'task_id': item['task_id'],
            'Collection': item['Collection'],
            'query': query,
            'history': history
        })

    # Rewrite queries
    print("\n📝 Rewriting queries...")
    rewrites_data = []

    with ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(rewriter.rewrite, t['query'], t['history']): i for i, t in enumerate(tasks)}

        for f in tqdm(as_completed(futures), total=len(futures), desc="Rewriting"):
            idx = futures[f]
            rewritten = f.result()
            tasks[idx]['rewritten_query'] = rewritten

            rewrites_data.append({
                "task_id": tasks[idx]['task_id'],
                "original_query": tasks[idx]['query'],
                "history": tasks[idx]['history'],
                "rewritten_query": rewritten
            })

    # Save rewrites
    with open(REWRITES_FILE, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)
    print(f"💾 Rewrites saved to: {REWRITES_FILE}")

    # Retrieval
    print("\n🔍 Retrieving from Elasticsearch...")
    predictions = []

    for task in tqdm(tasks, desc="Retrieving"):
        collection_name = task['Collection']
        index_name = INDEX_MAP.get(collection_name)

        if not index_name:
            print(f"⚠️ Unknown collection: {collection_name}")
            contexts = []
        else:
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": task['rewritten_query']
                            }
                        }
                    },
                    "size": 100
                })

                contexts = [
                    {
                        "document_id": h["_source"].get("doc_id", h["_id"]),
                        "score": h["_score"]
                    }
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Error for {task['task_id']}: {e}")
                contexts = []

        prediction = {
            "task_id": task['task_id'],
            "Collection": collection_name,  # ✅ Κρατάμε το απλό όνομα
            "contexts": contexts
        }
        predictions.append(prediction)

    # Save predictions
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {OUTPUT_FILE}")

    # Print first 3 lines
    print("\n" + "="*70)
    print("📋 FIRST 3 PREDICTIONS:")
    print("="*70)
    for i, pred in enumerate(predictions[:3], 1):
        print(f"\n--- Prediction {i} ---")
        print(f"Task ID: {pred['task_id']}")
        print(f"Collection: {pred['Collection']}")
        print(f"Contexts: {len(pred['contexts'])} retrieved")
        if pred['contexts']:
            print(f"Top result: {pred['contexts'][0]['document_id']} (score: {pred['contexts'][0]['score']:.4f})")

    # Cost report
    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


# Corpus based rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — TEST SET RETRIEVAL (Task A) - CORPUS-SPECIFIC REWRITERS
# INPUT: rag_taskAC.jsonl
# OUTPUT: Top-100 predictions with corpus-optimized query rewriting
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from elasticsearch import Elasticsearch
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
INPUT_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"
OUTPUT_FILE = f"{BASE_PATH}/test_set/predictions_elser_corpus_specific_top100.jsonl"
REWRITES_FILE = f"{BASE_PATH}/test_set/minimal_corpus_rewrites.json"

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

# Index mapping
INDEX_MAP = {
    "clapnq": "clapnq_elser_v1",
    "fiqa": "fiqa_elser_v1",
    "govt": "govt_elser_v1",
    "ibmcloud": "cloud_elser_v1"
}

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. HISTORY FORMATTING
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    """XML-structured conversation history (standard)"""
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['speaker'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['speaker'] == 'agent' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

def _history_to_text_optimal(history, u_max=6, a_max=0):
    """Optimized history (FiQA style - only user turns)"""
    output = ["<conversation_history>"]
    u_count = 0

    for h in reversed(history):
        if h['speaker'] == 'user' and u_count < u_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

# ----------------------------
# 4. LOAD TEST SET
# ----------------------------
def load_test_set(path: str):
    """Load rag_taskAC.jsonl"""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# ----------------------------
# 5. CORPUS-SPECIFIC REWRITERS
# ----------------------------
class ClapNQCorpusRewriter:
    def __init__(self):
        self.name = "clapnq_wikipedia_optimized2_6U3A"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are rewriting queries for retrieval from Wikipedia using ELSER (sparse semantic search).\n\n"
            "CORE OBJECTIVE:\n"
            "Transform the user's query into a standalone version that MAXIMIZES OVERLAP with Wikipedia passage text "
            "while preserving the original query's semantic intent.\n\n"
            "CRITICAL WIKIPEDIA + ELSER RETRIEVAL RULES:\n\n"
            "1. ENTITY FORMS - Use MULTIPLE variants when beneficial:\n"
            "   - Include both formal AND common names: 'Apple Inc. Apple' (not just 'Apple Inc.')\n"
            "   - For people: 'Francis Ford Coppola Coppola director'\n"
            "   - For places: 'New York NY New York City'\n"
            "   - For events: 'World War II WW2 Second World War'\n"
            "   ⚠️ ONLY add variants if they ADD retrieval value (don't spam)\n\n"
            "2. PRONOUN RESOLUTION - Be SURGICAL:\n"
            "   - 'he/she/it' → Exact entity name from conversation\n"
            "   - 'they/them' → Plural entity or organization name\n"
            "   - 'this/that' → Specific concept/event from context\n"
            "   - If pronoun refers to TITLE (movie/book): Use exact title, don't expand\n\n"
            "3. TEMPORAL HANDLING - Match Wikipedia's natural language:\n"
            "   ❌ DON'T force 'current' everywhere\n"
            "   ✅ DO use natural temporal phrasing:\n"
            "      - 'Who is the CEO?' → 'Who is CEO of Apple' (Wikipedia says 'serves as CEO', not 'current')\n"
            "      - 'When did X happen?' → Keep as-is\n"
            "      - Only add 'current' if user EXPLICITLY implies present time\n\n"
            "4. FORMALIZATION - MINIMAL ONLY:\n"
            "   ❌ DON'T over-formalize: 'cool facts' → 'notable characteristics' (too academic)\n"
            "   ✅ DO preserve conversational terms that appear in Wikipedia:\n"
            "      - 'facts about X' → 'facts about X' (Wikipedia uses 'facts'!)\n"
            "      - 'why did X happen' → 'why did X happen' OR 'causes of X' (both valid)\n"
            "   Rule: If the term EXISTS in Wikipedia articles, KEEP IT\n\n"
            "5. DISAMBIGUATION - Add ONLY when ambiguous:\n"
            "   - 'Mercury' after planet discussion → 'Mercury planet'\n"
            "   - 'Mercury' with no context → 'Mercury' (let retrieval handle it)\n"
            "   - Format: 'Entity (qualifier)' OR 'Entity qualifier' (both work)\n\n"
            "6. KEYWORD PRESERVATION:\n"
            "   - Preserve question words: who/what/when/where/why/how\n"
            "   - Keep domain-specific terms: director/actor/CEO/president/planet/movie\n"
            "   - Don't replace with synonyms unless they ADD coverage\n\n"
            "7. STRUCTURE:\n"
            "   - Keep original question structure (question → question, keywords → keywords)\n"
            "   - Don't expand 'population of France' into 'What is the population of France?'\n"
            "   - Natural language > Rigid formality\n\n"
            "MINIMAL INTERVENTION:\n"
            "- If query is standalone and has good entity names → Return UNCHANGED\n"
            "- Only rewrite to: resolve pronouns, add entity variants, clarify ambiguity\n"
            "- Goal: Better RETRIEVAL, not prettier English\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            '{\"class\": \"standalone|non-standalone\", \"rewritten version\": \"query here\"}\n'
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return parsed.get("rewritten version", query)
            return query
        except:
            return query


class FiQAOptimalRewriter:
    def __init__(self):
        self.name = "fiqa_optimal_6U0A"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text_optimal(history, u_max=6, a_max=0)

        system = (
            "You rewrite user queries for a finance Q&A dataset.\n"
            "Rewrite the final user utterance into ONE standalone question.\n"
            "Return STRICT JSON:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\"}\n\n"
            "Rules:\n"
            "- Minimal changes; preserve user's wording.\n"
            "- Resolve references ONLY using the conversation.\n"
            "- Preserve exact amounts, currencies, tickers, percentages, time horizons, product names mentioned.\n"
            "- Do NOT add finance jargon unless it already appears.\n"
            "- Do NOT introduce new entities/assumptions.\n"
            "- If already standalone: output SAME query + class='standalone'."
        )

        prompt_user = f"HISTORY:\n{hist_text}\n\nFINAL QUERY: {query}\n\nJSON:"

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return parsed.get("rewritten version", query).strip()
            return query
        except:
            return query


class GovtMinimalRewriter:
    def __init__(self):
        self.name = "govt_corpus_baseline_6U3A"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You rewrite user queries for a government information / forms / benefits dataset.\n"
            "Rewrite the final user utterance into ONE standalone question.\n"
            "Return STRICT JSON:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\"}\n\n"
            "Rules:\n"
            "- Minimal changes; preserve user's wording.\n"
            "- Resolve references ONLY using the conversation.\n"
            "- Preserve exact program names, agency names/acronyms, form IDs, deadlines, eligibility constraints, locations.\n"
            "- Do NOT introduce new agencies, programs, legal terms, requirements.\n"
            "- If already standalone: output SAME query + class='standalone'."
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                rewritten = parsed.get("rewritten version", query).strip()
                return rewritten if rewritten else query
            return query
        except:
            return query


class CloudMinimalRewriter:
    def __init__(self):
        self.name = "cloud_corpus_baseline_6U3A"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You rewrite user queries for a cloud troubleshooting dataset.\n"
            "Rewrite the final user utterance into ONE standalone question.\n"
            "Return STRICT JSON:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\"}\n\n"
            "Rules:\n"
            "- Minimal changes; preserve user's wording.\n"
            "- Resolve references ONLY using the conversation.\n"
            "- Preserve exact error messages/codes, stack trace snippets, CLI commands, flags, config keys, service names.\n"
            "- Include environment details (OS/region/service/resource) ONLY if present.\n"
            "- Do NOT add new tools/providers/services/config keys.\n"
            "- If already standalone: output SAME query + class='standalone'."
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 400)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                rewritten = parsed.get("rewritten version", query).strip()
                return rewritten if rewritten else query
            return query
        except:
            return query


# ----------------------------
# 6. REWRITER FACTORY
# ----------------------------
def get_rewriter_for_collection(collection_name: str):
    """Returns the appropriate rewriter based on collection name"""
    rewriters = {
        "clapnq": ClapNQCorpusRewriter(),
        "fiqa": FiQAOptimalRewriter(),
        "govt": GovtMinimalRewriter(),
        "ibmcloud": CloudMinimalRewriter()
    }
    return rewriters.get(collection_name, ClapNQCorpusRewriter())  # fallback


# ----------------------------
# 7. MAIN PIPELINE
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 TEST SET - CORPUS-SPECIFIC REWRITERS")
    print("="*70)
    print(f"Input: {INPUT_FILE}")
    print(f"Output: {OUTPUT_FILE}")
    print(f"Rewrites: {REWRITES_FILE}")
    print(f"Top-K: 100")
    print("="*70)

    # Initialize Elasticsearch
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    # Load test set
    print("\n📂 Loading test set...")
    data = load_test_set(INPUT_FILE)
    print(f"✅ Loaded {len(data)} tasks")

    # Extract query and history from input
    tasks = []
    for item in data:
        input_turns = item['input']
        query = input_turns[-1]['text']
        history = input_turns[:-1]

        tasks.append({
            'task_id': item['task_id'],
            'Collection': item['Collection'],
            'query': query,
            'history': history
        })

    # Rewrite queries (corpus-specific)
    print("\n📝 Rewriting queries (corpus-specific)...")
    rewrites_data = []

    def rewrite_task(task):
        collection = task['Collection']
        rewriter = get_rewriter_for_collection(collection)
        rewritten = rewriter.rewrite(task['query'], task['history'])
        return rewritten, rewriter.name

    with ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(rewrite_task, t): i for i, t in enumerate(tasks)}

        for f in tqdm(as_completed(futures), total=len(futures), desc="Rewriting"):
            idx = futures[f]
            rewritten, rewriter_name = f.result()
            tasks[idx]['rewritten_query'] = rewritten
            tasks[idx]['rewriter_used'] = rewriter_name

            rewrites_data.append({
                "task_id": tasks[idx]['task_id'],
                "collection": tasks[idx]['Collection'],
                "rewriter": rewriter_name,
                "original_query": tasks[idx]['query'],
                "history": tasks[idx]['history'],
                "rewritten_query": rewritten
            })

    # Save rewrites
    with open(REWRITES_FILE, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)
    print(f"💾 Rewrites saved to: {REWRITES_FILE}")

    # Retrieval
    print("\n🔍 Retrieving from Elasticsearch...")
    predictions = []

    for task in tqdm(tasks, desc="Retrieving"):
        collection_name = task['Collection']
        index_name = INDEX_MAP.get(collection_name)

        if not index_name:
            print(f"⚠️ Unknown collection: {collection_name}")
            contexts = []
        else:
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": task['rewritten_query']
                            }
                        }
                    },
                    "size": 100
                })

                contexts = [
                    {
                        "document_id": h["_source"].get("doc_id", h["_id"]),
                        "score": h["_score"]
                    }
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Error for {task['task_id']}: {e}")
                contexts = []

        prediction = {
            "task_id": task['task_id'],
            "Collection": collection_name,
            "contexts": contexts
        }
        predictions.append(prediction)

    # Save predictions
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {OUTPUT_FILE}")

    # Print first 3 lines
    print("\n" + "="*70)
    print("📋 FIRST 3 PREDICTIONS:")
    print("="*70)
    for i, pred in enumerate(predictions[:3], 1):
        print(f"\n--- Prediction {i} ---")
        print(f"Task ID: {pred['task_id']}")
        print(f"Collection: {pred['Collection']}")
        print(f"Rewriter: {tasks[i-1]['rewriter_used']}")
        print(f"Contexts: {len(pred['contexts'])} retrieved")
        if pred['contexts']:
            print(f"Top result: {pred['contexts'][0]['document_id']} (score: {pred['contexts'][0]['score']:.4f})")

    # Rewriter usage stats
    print("\n" + "="*70)
    print("📊 REWRITER USAGE STATS:")
    print("="*70)
    from collections import Counter
    rewriter_counts = Counter([t['rewriter_used'] for t in tasks])
    for rewriter, count in rewriter_counts.items():
        print(f"  {rewriter}: {count} tasks")

    # Cost report
    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


# COT and Hyde rewriters

In [ ]:
# ==============================================================================
# ELSER v1 — TEST SET RETRIEVAL (Task A) - COT & HYDE REWRITERS
# INPUT: rag_taskAC.jsonl
# OUTPUT: Top-100 predictions with CoT & HyDE rewriting
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from elasticsearch import Elasticsearch
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
INPUT_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

# Index mapping
INDEX_MAP = {
    "clapnq": "clapnq_elser_v1",
    "fiqa": "fiqa_elser_v1",
    "govt": "govt_elser_v1",
    "ibmcloud": "cloud_elser_v1"
}

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. HISTORY FORMATTING
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=3):
    """XML-structured conversation history"""
    output = ["<conversation_history>"]
    u_count, a_count = 0, 0

    for h in reversed(history):
        if h['speaker'] == 'user' and u_count < user_max:
            output.append(f"  <turn role='user'>{h['text']}</turn> </s>")
            u_count += 1
        elif h['speaker'] == 'agent' and a_count < assistant_max:
            output.append(f"  <turn role='assistant'>{h['text']}</turn> </s>")
            a_count += 1

    output.append("</conversation_history>")
    return "\n".join(reversed(output))

# ----------------------------
# 4. LOAD TEST SET
# ----------------------------
def load_test_set(path: str):
    """Load rag_taskAC.jsonl"""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# ----------------------------
# 5. COT & HYDE REWRITERS
# ----------------------------
class ChainOfThoughtRewriter:
    def __init__(self):
        self.name = "elser_cot_6U3A"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are an expert query rewriter for information retrieval systems.\n\n"
            "TASK: Rewrite the user's final query into a standalone version for optimal retrieval.\n\n"
            "PROCESS (Chain-of-Thought):\n"
            "1. ANALYZE the conversation history to identify:\n"
            "   - What entities/topics are being discussed\n"
            "   - What pronouns or references need resolution\n"
            "   - What context is needed to understand the final query\n\n"
            "2. REASON about the user's intent:\n"
            "   - Is the query standalone or does it depend on history?\n"
            "   - What information from history is essential?\n"
            "   - What should be preserved vs added?\n\n"
            "3. REWRITE the query:\n"
            "   - Resolve all pronouns (he/she/it/they/this/that)\n"
            "   - Add missing entities/context from history\n"
            "   - Keep original wording when possible\n"
            "   - Make it self-contained and clear\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            "{\n"
            "  \"reasoning\": \"<your step-by-step analysis>\",\n"
            "  \"class\": \"standalone|non-standalone\",\n"
            "  \"rewritten version\": \"<rewritten query>\"\n"
            "}\n\n"
            "RULES:\n"
            "- Be minimal: don't add unnecessary words\n"
            "- Be precise: use exact entities from history\n"
            "- Be faithful: preserve user's intent and terminology\n"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE with reasoning):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 600)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                return parsed.get("rewritten version", query)
            return query
        except:
            return query


class HyDERewriter:
    def __init__(self):
        self.name = "elser_hyde_6U3A"

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=6, assistant_max=3)

        system = (
            "You are generating hypothetical document passages for retrieval optimization.\n\n"
            "TASK: Given a conversation and final user query, generate a HYPOTHETICAL PASSAGE "
            "that would perfectly answer the query. This passage will be used for retrieval.\n\n"
            "STRATEGY (HyDE - Hypothetical Document Embeddings):\n"
            "1. First, understand the query in context of the conversation\n"
            "2. Generate a realistic passage (2-4 sentences) that:\n"
            "   - Directly answers the query\n"
            "   - Uses terminology likely to appear in real documents\n"
            "   - Contains relevant keywords and entities\n"
            "   - Sounds natural and informative\n\n"
            "OUTPUT FORMAT (strict JSON):\n"
            "{\n"
            "  \"standalone_query\": \"<query rewritten to be standalone>\",\n"
            "  \"hypothetical_passage\": \"<2-4 sentence passage that would answer the query>\"\n"
            "}\n\n"
            "EXAMPLE:\n"
            "Query: 'What is its capital?'\n"
            "Context: Discussion about France\n"
            "Output:\n"
            "{\n"
            "  \"standalone_query\": \"What is the capital of France?\",\n"
            "  \"hypothetical_passage\": \"The capital of France is Paris. Paris is located in "
            "the north-central part of the country on the River Seine. It is the most populous "
            "city in France with over 2 million residents in the city proper.\"\n"
            "}\n\n"
            "RULES:\n"
            "- Write as if you're a document that answers the question\n"
            "- Include specific facts, names, dates when relevant\n"
            "- Use vocabulary that would appear in authoritative sources\n"
            "- Keep it concise but informative (2-4 sentences)\n"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 800)
            match = re.search(r"\{.*\}", txt, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                standalone = parsed.get("standalone_query", query)
                passage = parsed.get("hypothetical_passage", "")
                combined = f"{standalone} {passage}".strip()
                return combined
            return query
        except:
            return query

# ----------------------------
# 6. EXPERIMENT RUNNER
# ----------------------------
def run_experiment_with_rewriter(rewriter, data, es_client):
    """Run complete experiment with given rewriter"""

    print("\n" + "="*70)
    print(f"🚀 RUNNING REWRITER: {rewriter.name.upper()}")
    print("="*70)

    # Prepare output files
    output_file = f"{BASE_PATH}/test_set/predictions_{rewriter.name}_top100.jsonl"
    rewrites_file = f"{BASE_PATH}/test_set/minimal_rewrites_{rewriter.name}.json"

    # Extract tasks
    tasks = []
    for item in data:
        input_turns = item['input']
        query = input_turns[-1]['text']
        history = input_turns[:-1]

        tasks.append({
            'task_id': item['task_id'],
            'Collection': item['Collection'],
            'query': query,
            'history': history
        })

    # Rewrite queries
    print(f"\n📝 Rewriting {len(tasks)} queries with {rewriter.name}...")
    rewrites_data = []

    def rewrite_task(task):
        rewritten = rewriter.rewrite(task['query'], task['history'])
        return rewritten

    with ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(rewrite_task, t): i for i, t in enumerate(tasks)}

        for f in tqdm(as_completed(futures), total=len(futures), desc="Rewriting"):
            idx = futures[f]
            rewritten = f.result()
            tasks[idx]['rewritten_query'] = rewritten

            rewrites_data.append({
                "task_id": tasks[idx]['task_id'],
                "collection": tasks[idx]['Collection'],
                "rewriter": rewriter.name,
                "original_query": tasks[idx]['query'],
                "history": tasks[idx]['history'],
                "rewritten_query": rewritten
            })

    # Save rewrites
    with open(rewrites_file, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)
    print(f"💾 Rewrites saved to: {rewrites_file}")

    # Retrieval
    print(f"\n🔍 Retrieving from Elasticsearch...")
    predictions = []

    for task in tqdm(tasks, desc="Retrieving"):
        collection_name = task['Collection']
        index_name = INDEX_MAP.get(collection_name)

        if not index_name:
            print(f"⚠️ Unknown collection: {collection_name}")
            contexts = []
        else:
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": task['rewritten_query']
                            }
                        }
                    },
                    "size": 100
                })

                contexts = [
                    {
                        "document_id": h["_source"].get("doc_id", h["_id"]),
                        "score": h["_score"]
                    }
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Error for {task['task_id']}: {e}")
                contexts = []

        prediction = {
            "task_id": task['task_id'],
            "Collection": collection_name,
            "contexts": contexts
        }
        predictions.append(prediction)

    # Save predictions
    with open(output_file, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {output_file}")

    # Print first 3 lines
    print("\n" + "="*70)
    print(f"📋 FIRST 3 PREDICTIONS ({rewriter.name}):")
    print("="*70)
    for i, pred in enumerate(predictions[:3], 1):
        print(f"\n--- Prediction {i} ---")
        print(f"Task ID: {pred['task_id']}")
        print(f"Collection: {pred['Collection']}")
        print(f"Original: {tasks[i-1]['query'][:80]}...")
        print(f"Rewritten: {tasks[i-1]['rewritten_query'][:80]}...")
        print(f"Contexts: {len(pred['contexts'])} retrieved")
        if pred['contexts']:
            print(f"Top result: {pred['contexts'][0]['document_id']} (score: {pred['contexts'][0]['score']:.4f})")

    return output_file, rewrites_file

# ----------------------------
# 7. MAIN EXECUTION
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 TEST SET - COT & HYDE REWRITERS")
    print("="*70)
    print(f"Input: {INPUT_FILE}")
    print("Rewriters: Chain-of-Thought (CoT) + HyDE")
    print(f"Top-K: 100")
    print("="*70)

    # Initialize Elasticsearch
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    # Load test set
    print("\n📂 Loading test set...")
    data = load_test_set(INPUT_FILE)
    print(f"✅ Loaded {len(data)} tasks")

    # Initialize rewriters
    rewriters = [
        ChainOfThoughtRewriter(),
        HyDERewriter()
    ]

    # Run experiments
    results = []
    for rewriter in rewriters:
        try:
            output_file, rewrites_file = run_experiment_with_rewriter(rewriter, data, es_client)
            results.append({
                'rewriter': rewriter.name,
                'output': output_file,
                'rewrites': rewrites_file
            })
        except Exception as e:
            print(f"❌ Error with {rewriter.name}: {e}")
            import traceback
            traceback.print_exc()

    # Summary
    print("\n" + "="*70)
    print("📊 EXPERIMENT SUMMARY")
    print("="*70)
    for res in results:
        print(f"\n✅ {res['rewriter'].upper()}")
        print(f"   Predictions: {res['output']}")
        print(f"   Rewrites: {res['rewrites']}")

    # Cost report
    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


# Anchor-keyword rewriter

In [ ]:
# ==============================================================================
# ELSER v1 — TEST SET RETRIEVAL (Task A) - ANCHOR+KEYWORD REWRITER
# INPUT: rag_taskAC.jsonl
# OUTPUT: Top-100 predictions with Anchor+Keyword combo rewriting (ELSER-friendly)
# ==============================================================================

import os, json, re, time, threading, requests
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from elasticsearch import Elasticsearch
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# ----------------------------
# 1. SETTINGS & CONFIG
# ----------------------------
BASE_PATH = "/content/drive/MyDrive/Project"
INPUT_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"
OUTPUT_FILE = f"{BASE_PATH}/test_set/predictions_elser_anchor_keyword_top100.jsonl"
REWRITES_FILE = f"{BASE_PATH}/test_set/minimal_anchor_rewrites.json"

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

CLOUD_ID = os.environ["CLOUD_ID"]
PASSWORD = os.environ["PASSWORD"]

# Index mapping
INDEX_MAP = {
    "clapnq": "clapnq_elser_v1",
    "fiqa": "fiqa_elser_v1",
    "govt": "govt_elser_v1",
    "ibmcloud": "cloud_elser_v1"
}

# ----------------------------
# 2. COST TRACKER
# ----------------------------
class CostTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        return f"In: {self.total_input} | Out: {self.total_output} | Est. Cost: ${cost:.4f}"

tracker = CostTracker()

def llm_invoke_text(system: str, user: str, max_tokens: int) -> str:
    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-V3.2",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }
    for attempt in range(7):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                return res_json['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                time.sleep(5 * (2 ** attempt))
                continue
        except:
            time.sleep(5)
    return ""

# ----------------------------
# 3. HISTORY FORMATTING
# ----------------------------
def _history_to_text(history, user_max=6, assistant_max=1):
    """XML-structured conversation history (fixed chronological order)"""
    turns = []
    u_count = a_count = 0

    # Walk from most recent backwards to cap properly
    for h in reversed(history):
        if h['speaker'] == 'user' and u_count < user_max:
            turns.append(f"  <turn role='user'>{h['text']}</turn>")
            u_count += 1
        elif h['speaker'] == 'agent' and a_count < assistant_max:
            turns.append(f"  <turn role='assistant'>{h['text']}</turn>")
            a_count += 1

    turns.reverse()  # Now oldest -> newest
    return "<conversation_history>\n" + "\n".join(turns) + "\n</conversation_history>"

# ----------------------------
# 4. HELPER FUNCTIONS FOR ANCHOR+KEYWORD
# ----------------------------
def _norm_term(s: str) -> str:
    """Normalize term for deduplication"""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[\"'`]", "", s)
    return s

def _dedupe_terms(terms: List[str]) -> List[str]:
    """Remove duplicate terms (case-insensitive)"""
    seen = set()
    out = []
    for t in terms:
        if not t:
            continue
        key = _norm_term(t)
        if key and key not in seen:
            seen.add(key)
            out.append(t.strip())
    return out

def _cap_by_words(text: str, max_words: int) -> str:
    """Cap text to max_words"""
    w = text.split()
    return text.strip() if len(w) <= max_words else " ".join(w[:max_words]).strip()

# ----------------------------
# 5. LOAD TEST SET
# ----------------------------
def load_test_set(path: str):
    """Load rag_taskAC.jsonl"""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# ----------------------------
# 6. ANCHOR+KEYWORD REWRITER
# ----------------------------
class ELSERAnchorKeywordRewriter:
    def __init__(self):
        self.name = "elser_anchor_keyword_combo_6U1A"
        self.user_max = 6
        self.assistant_max = 1
        self.max_words = 28
        self.max_anchors = 8
        self.max_keywords = 12

    def rewrite(self, query, history):
        if not history:
            return query

        hist_text = _history_to_text(history, user_max=self.user_max, assistant_max=self.assistant_max)

        system = (
            "Given the following conversation, rewrite the final user utterance into ONE standalone query for RETRIEVAL.\n\n"
            "Additionally, extract compact RETRIEVAL TERMS to help ELSER sparse search:\n"
            "- anchors: exact entity names, titles, product/service names, acronyms, IDs, error codes, CLI flags\n"
            "- keywords: short bag-of-terms reflecting core intent\n\n"
            "Rules:\n"
            "- Do NOT invent new entities/facts.\n"
            "- Be minimal; only resolve pronouns/ambiguous references using the conversation.\n"
            "- Preserve numbers/codes/tickers exactly.\n"
            f"- anchors max {self.max_anchors}, keywords max {self.max_keywords}.\n"
            f"- Keep the rewritten query short (aim <= {self.max_words} words).\n\n"
            "Output STRICT JSON ONLY, in this exact schema:\n"
            "{\"class\":\"standalone|non-standalone\",\"rewritten version\":\"...\",\"anchors\":[\"...\"],\"keywords\":[\"...\"]}"
        )

        prompt_user = (
            f"CONTEXT_BLOCK_START\n{hist_text}\nCONTEXT_BLOCK_END\n\n"
            f"<target_query_to_rewrite>\nuser: {query}\n</target_query_to_rewrite>\n\n"
            f"ASSISTANT (JSON_RESPONSE):"
        )

        try:
            txt = llm_invoke_text(system, prompt_user, 450)
            m = re.search(r"\{.*\}", txt, re.DOTALL)
            if not m:
                return query

            obj = json.loads(m.group(0))

            base = (obj.get("rewritten version") or query).strip()

            anchors = obj.get("anchors") or []
            keywords = obj.get("keywords") or []

            if not isinstance(anchors, list):
                anchors = []
            if not isinstance(keywords, list):
                keywords = []

            anchors = _dedupe_terms([str(x) for x in anchors])[:self.max_anchors]
            keywords = _dedupe_terms([str(x) for x in keywords])[:self.max_keywords]

            # Combine: base query + anchors + keywords
            combined = " ".join([base] + anchors + keywords).strip()
            combined = _cap_by_words(combined, self.max_words)

            return combined if combined else query

        except Exception:
            return query

# ----------------------------
# 7. MAIN PIPELINE
# ----------------------------
def main():
    print("\n" + "="*70)
    print("🚀 ELSER v1 TEST SET - ANCHOR+KEYWORD REWRITER")
    print("="*70)
    print(f"Input: {INPUT_FILE}")
    print(f"Output: {OUTPUT_FILE}")
    print(f"Rewrites: {REWRITES_FILE}")
    print(f"Strategy: 6U/1A XML + Anchors/Keywords (ELSER-friendly)")
    print(f"Top-K: 100")
    print("="*70)

    # Initialize Elasticsearch
    es_client = Elasticsearch(
        cloud_id=CLOUD_ID,
        basic_auth=("elastic", PASSWORD),
        request_timeout=180
    )

    # Load test set
    print("\n📂 Loading test set...")
    data = load_test_set(INPUT_FILE)
    print(f"✅ Loaded {len(data)} tasks")

    # Initialize rewriter
    rewriter = ELSERAnchorKeywordRewriter()

    # Extract tasks
    tasks = []
    for item in data:
        input_turns = item['input']
        query = input_turns[-1]['text']
        history = input_turns[:-1]

        tasks.append({
            'task_id': item['task_id'],
            'Collection': item['Collection'],
            'query': query,
            'history': history
        })

    # Rewrite queries
    print(f"\n📝 Rewriting {len(tasks)} queries with {rewriter.name}...")
    rewrites_data = []

    def rewrite_task(task):
        rewritten = rewriter.rewrite(task['query'], task['history'])
        return rewritten

    with ThreadPoolExecutor(max_workers=5) as ex:
        futures = {ex.submit(rewrite_task, t): i for i, t in enumerate(tasks)}

        for f in tqdm(as_completed(futures), total=len(futures), desc="Rewriting"):
            idx = futures[f]
            rewritten = f.result()
            tasks[idx]['rewritten_query'] = rewritten

            rewrites_data.append({
                "task_id": tasks[idx]['task_id'],
                "collection": tasks[idx]['Collection'],
                "rewriter": rewriter.name,
                "original_query": tasks[idx]['query'],
                "history": tasks[idx]['history'],
                "rewritten_query": rewritten
            })

    # Save rewrites
    with open(REWRITES_FILE, "w", encoding="utf-8") as f:
        json.dump(rewrites_data, f, indent=2, ensure_ascii=False)
    print(f"💾 Rewrites saved to: {REWRITES_FILE}")

    # Retrieval
    print("\n🔍 Retrieving from Elasticsearch...")
    predictions = []

    for task in tqdm(tasks, desc="Retrieving"):
        collection_name = task['Collection']
        index_name = INDEX_MAP.get(collection_name)

        if not index_name:
            print(f"⚠️ Unknown collection: {collection_name}")
            contexts = []
        else:
            try:
                res = es_client.search(index=index_name, body={
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": ".elser_model_1",
                                "model_text": task['rewritten_query']
                            }
                        }
                    },
                    "size": 100
                })

                contexts = [
                    {
                        "document_id": h["_source"].get("doc_id", h["_id"]),
                        "score": h["_score"]
                    }
                    for h in res["hits"]["hits"]
                ]
            except Exception as e:
                print(f"⚠️ Error for {task['task_id']}: {e}")
                contexts = []

        prediction = {
            "task_id": task['task_id'],
            "Collection": collection_name,
            "contexts": contexts
        }
        predictions.append(prediction)

    # Save predictions
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + "\n")

    print(f"✅ Predictions saved to: {OUTPUT_FILE}")

    # Print first 3 lines
    print("\n" + "="*70)
    print("📋 FIRST 3 PREDICTIONS:")
    print("="*70)
    for i, pred in enumerate(predictions[:3], 1):
        print(f"\n--- Prediction {i} ---")
        print(f"Task ID: {pred['task_id']}")
        print(f"Collection: {pred['Collection']}")
        print(f"Original: {tasks[i-1]['query'][:60]}...")
        print(f"Rewritten: {tasks[i-1]['rewritten_query'][:80]}...")
        print(f"Contexts: {len(pred['contexts'])} retrieved")
        if pred['contexts']:
            print(f"Top result: {pred['contexts'][0]['document_id']} (score: {pred['contexts'][0]['score']:.4f})")

    # Show example rewrite breakdown
    print("\n" + "="*70)
    print("🔍 REWRITE EXAMPLE (detailed):")
    print("="*70)
    example = tasks[0]
    print(f"Original query: {example['query']}")
    print(f"Has history: {len(example['history'])} turns")
    print(f"Rewritten (with anchors+keywords): {example['rewritten_query']}")

    # Cost report
    print("\n" + "="*70)
    print("💰 COST SUMMARY")
    print("="*70)
    print(tracker.get_report())
    print("="*70)

if __name__ == "__main__":
    main()


# Cohere Reranking per rewriter

In [ ]:
# ==============================================================================
# COHERE V4 RERANK - BATCH PROCESSING FOR 5 REWRITERS (TEST SET)
# Saves: Cohere-Only + Hybrid 0.5/0.5
# No evaluation - just file generation
# ==============================================================================

import os, json, time, requests
import numpy as np
from tqdm.auto import tqdm
from typing import Dict, List

BASE_PATH = "/content/drive/MyDrive/Project"

# ==============================================================================
# 1. CONFIGURATION - 5 REWRITERS
# ==============================================================================

REWRITER_CONFIGS = {
    "minimal_elser": {
        "predictions": f"{BASE_PATH}/test_set/predictions_minimal_elser_top100.jsonl",
        "rewrites": f"{BASE_PATH}/test_set/minimal_rewrites.json"
    },
    "corpus_specific": {
        "predictions": f"{BASE_PATH}/test_set/predictions_elser_corpus_specific_top100.jsonl",
        "rewrites": f"{BASE_PATH}/test_set/minimal_corpus_rewrites.json"
    },
    "hyde": {
        "predictions": f"{BASE_PATH}/test_set/predictions_elser_hyde_6U3A_top100.jsonl",
        "rewrites": f"{BASE_PATH}/test_set/minimal_rewrites_elser_hyde_6U3A.json"
    },
    "cot": {
        "predictions": f"{BASE_PATH}/test_set/predictions_elser_cot_6U3A_top100.jsonl",
        "rewrites": f"{BASE_PATH}/test_set/minimal_rewrites_elser_cot_6U3A.json"
    },
    "anchor_keyword": {
        "predictions": f"{BASE_PATH}/test_set/predictions_elser_anchor_keyword_top100.jsonl",
        "rewrites": f"{BASE_PATH}/test_set/minimal_anchor_rewrites.json"
    }
}

# Corpus files
CORPUS_FILES = {
    "clapnq": f"{BASE_PATH}/corpora/clapnq.jsonl",
    "fiqa": f"{BASE_PATH}/corpora/fiqa.jsonl",
    "govt": f"{BASE_PATH}/corpora/govt.jsonl",
    "ibmcloud": f"{BASE_PATH}/corpora/cloud.jsonl"
}

# Output directory
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = os.path.join(BASE_PATH, "test_set_reranked", RUN_TAG)
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Azure Cohere Credentials
AZURE_COHERE_ENDPOINT = os.environ["ENDPOINT"]
AZURE_COHERE_KEY = os.environ["_KEY"]

TOP_K_RERANK = 100

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def load_json(path: str) -> Dict:
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path):
        return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text[:2000]
    return corpus

def load_all_corpora() -> Dict[str, Dict[str, str]]:
    """Load all 4 corpora into memory"""
    all_corpora = {}
    for collection, path in CORPUS_FILES.items():
        print(f"📥 Loading {collection} corpus...")
        all_corpora[collection] = load_corpus(path)
        print(f"   ✅ Loaded {len(all_corpora[collection])} documents")
    return all_corpora

def load_rewrites(path: str) -> Dict[str, str]:
    rewrites_map = {}
    if not os.path.exists(path):
        print(f"⚠️ Rewrites file not found: {path}")
        return rewrites_map

    try:
        rewrites_data = load_json(path)

        if isinstance(rewrites_data, list):
            for item in rewrites_data:
                qid = item.get("task_id") or item.get("qid")
                rewrite = item.get("rewritten_query") or item.get("rewrite")
                if qid and rewrite:
                    rewrites_map[str(qid)] = rewrite
        elif isinstance(rewrites_data, dict):
            rewrites_map = {str(k): str(v) for k, v in rewrites_data.items()}

        print(f"   ✅ Loaded {len(rewrites_map)} rewrites")
    except Exception as e:
        print(f"   ❌ Error loading rewrites: {e}")

    return rewrites_map

def azure_cohere_rerank(query: str, docs_text: List[str]) -> List[float]:
    if not query or not docs_text:
        return [0.0] * len(docs_text)

    headers = {"Content-Type": "application/json", "api-key": AZURE_COHERE_KEY}
    payload = {
        "model": "Cohere-rerank-v4.0-pro",
        "query": query,
        "documents": docs_text,
        "top_n": len(docs_text)
    }

    try:
        response = requests.post(AZURE_COHERE_ENDPOINT, headers=headers, json=payload, timeout=60)
        if response.status_code == 200:
            results = response.json()["results"]
            scores = [0.0] * len(docs_text)
            for r in results:
                scores[r["index"]] = float(r["relevance_score"])
            return scores
        elif response.status_code == 429:
            time.sleep(2)
            return azure_cohere_rerank(query, docs_text)
    except Exception as e:
        print(f"❌ Cohere API Error: {e}")

    return [0.0] * len(docs_text)

def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min():
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

# ==============================================================================
# 3. PROCESS SINGLE REWRITER
# ==============================================================================

def process_rewriter(rewriter_name: str, config: Dict, all_corpora: Dict) -> None:
    """Process a single rewriter and save Cohere-Only + Hybrid 0.5/0.5"""

    print("\n" + "="*80)
    print(f"🎯 PROCESSING: {rewriter_name.upper()}")
    print("="*80)

    # Create rewriter-specific output directory
    output_dir = os.path.join(OUTPUT_BASE, rewriter_name)
    os.makedirs(output_dir, exist_ok=True)

    # Load data
    print("📥 Loading predictions and rewrites...")
    preds = load_jsonl(config["predictions"])
    rewrites_map = load_rewrites(config["rewrites"])

    if not preds:
        print(f"❌ Error: No predictions found for {rewriter_name}")
        return

    print(f"✅ Loaded {len(preds)} predictions")

    # Statistics
    queries_with_rewrites = sum(1 for p in preds if str(p["task_id"]) in rewrites_map)
    print(f"📊 Queries with rewrites: {queries_with_rewrites}/{len(preds)} ({queries_with_rewrites/len(preds)*100:.1f}%)")

    # Cache for reranking
    cache = []

    # Run Cohere reranking
    print("\n🎯 Running Cohere reranking...")
    for p in tqdm(preds, desc=f"Reranking {rewriter_name}", ncols=100):
        qid = str(p["task_id"])
        collection = p["Collection"]

        # Get query text (rewritten or task_id as fallback)
        q_txt = rewrites_map.get(qid, qid)

        # Get corpus for this collection
        corpus_map = all_corpora.get(collection, {})

        # Top-K contexts for reranking
        top_ctxs = p["contexts"][:TOP_K_RERANK]
        rest_ctxs = p["contexts"][TOP_K_RERANK:]

        # Get document texts
        doc_texts = [corpus_map.get(str(c["document_id"]), "N/A") for c in top_ctxs]

        # Cohere rerank
        coh_scores = azure_cohere_rerank(q_txt, doc_texts)
        orig_scores = [float(c["score"]) for c in top_ctxs]

        cache.append({
            "task_id": qid,
            "Collection": collection,
            "top_ctxs": top_ctxs,
            "rest": rest_ctxs,
            "orig_scores": orig_scores,
            "coh_scores": coh_scores
        })

    # Save Cohere-Only
    print("\n💾 Saving Cohere-Only predictions...")
    cohere_file = os.path.join(output_dir, f"{rewriter_name}_COHERE_ONLY.jsonl")
    with open(cohere_file, "w", encoding="utf-8") as f:
        for item in cache:
            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(item["coh_scores"][i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)

            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }, ensure_ascii=False) + "\n")

    print(f"   ✅ Saved: {cohere_file}")

    # Save Hybrid 0.5/0.5
    print("\n💾 Saving Hybrid 0.5/0.5 predictions...")
    hybrid_file = os.path.join(output_dir, f"{rewriter_name}_HYBRID_0.5.jsonl")
    with open(hybrid_file, "w", encoding="utf-8") as f:
        for item in cache:
            n_orig = minmax_norm(item["orig_scores"])
            n_coh = minmax_norm(item["coh_scores"])
            combined = (0.5 * n_orig) + (0.5 * n_coh)

            new_ctxs = []
            for i, ctx in enumerate(item["top_ctxs"]):
                nc = ctx.copy()
                nc["score"] = float(combined[i])
                new_ctxs.append(nc)
            new_ctxs.sort(key=lambda x: x["score"], reverse=True)

            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": new_ctxs + item["rest"]
            }, ensure_ascii=False) + "\n")

    print(f"   ✅ Saved: {hybrid_file}")
    print(f"\n✅ {rewriter_name} complete!")

# ==============================================================================
# 4. MAIN - BATCH PROCESSING
# ==============================================================================

def main():
    print("="*80)
    print("🔥 COHERE V4 BATCH RERANKING - 5 REWRITERS (TEST SET)")
    print("="*80)
    print(f"\n📂 Processing {len(REWRITER_CONFIGS)} rewriters:")
    for name in REWRITER_CONFIGS.keys():
        print(f"   • {name}")
    print(f"\n💾 Output base: {OUTPUT_BASE}\n")

    # Load all corpora once
    print("="*80)
    print("📥 LOADING ALL CORPORA")
    print("="*80)
    all_corpora = load_all_corpora()

    # Process each rewriter
    for rewriter_name, config in REWRITER_CONFIGS.items():
        try:
            process_rewriter(rewriter_name, config, all_corpora)
        except Exception as e:
            print(f"❌ Error processing {rewriter_name}: {e}")
            import traceback
            traceback.print_exc()

    # Summary
    print("\n" + "="*80)
    print("🎉 BATCH PROCESSING COMPLETE")
    print("="*80)
    print(f"\n📁 All reranked files saved to: {OUTPUT_BASE}\n")

    # List generated files
    print("📋 GENERATED FILES:")
    print("="*80)
    for rewriter_name in REWRITER_CONFIGS.keys():
        output_dir = os.path.join(OUTPUT_BASE, rewriter_name)
        if os.path.exists(output_dir):
            print(f"\n{rewriter_name}:")
            for fname in sorted(os.listdir(output_dir)):
                fpath = os.path.join(output_dir, fname)
                size_mb = os.path.getsize(fpath) / (1024*1024)
                print(f"   ✅ {fname} ({size_mb:.2f} MB)")

if __name__ == "__main__":
    main()


# RRF

In [ ]:
# ==============================================================================
# NESTED RRF ENSEMBLE - TEST SET (NO EVALUATION)
# Uses best configurations from dev set experiments
# ==============================================================================

import os, json, time
from tqdm.auto import tqdm

BASE_PATH = "/content/drive/MyDrive/Project"
RERANKED_BASE = f"{BASE_PATH}/test_set_reranked/20260119_014449"
OUTPUT_DIR = f"{BASE_PATH}/test_set_final_ensemble"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 1. BEST CONFIGURATIONS FROM DEV SET (LOCKED)
# ==============================================================================

FINAL_CONFIGS = {
    "clapnq": {
        "k_final": 20,
        "weights": {"Structural_Minimal": 0.55, "Corpus_Specific": 0.40, "Weak_Group_Consensus": 0.05},
        "k_internal": 40
    },
    "fiqa": {
        "k_final": 60,
        "weights": {"Structural_Minimal": 0.45, "Corpus_Specific": 0.40, "Weak_Group_Consensus": 0.15},
        "k_internal": 40
    },
    "govt": {
        "k_final": 40,
        "weights": {"Structural_Minimal": 0.65, "Corpus_Specific": 0.25, "Weak_Group_Consensus": 0.10},
        "k_internal": 40
    },
    "ibmcloud": {  # ✅ ΔΙΟΡΘΩΣΗ: "cloud" -> "ibmcloud"
        "k_final": 20,
        "weights": {"Structural_Minimal": 0.65, "Corpus_Specific": 0.30, "Weak_Group_Consensus": 0.05},
        "k_internal": 40
    }
}

WEAK_INTERNAL_WEIGHTS = {"Anchor_Keyword": 0.33, "COT": 0.33, "Hyde": 0.34}

# ==============================================================================
# 2. FILE MAPPING (RERANKED FILES)
# ==============================================================================

FILE_MAP = {
    "Structural_Minimal": f"{RERANKED_BASE}/minimal_elser/minimal_elser_HYBRID_0.5.jsonl",
    "Corpus_Specific": f"{RERANKED_BASE}/corpus_specific/corpus_specific_HYBRID_0.5.jsonl",
    "Anchor_Keyword": f"{RERANKED_BASE}/anchor_keyword/anchor_keyword_HYBRID_0.5.jsonl",
    "COT": f"{RERANKED_BASE}/cot/cot_HYBRID_0.5.jsonl",
    "Hyde": f"{RERANKED_BASE}/hyde/hyde_HYBRID_0.5.jsonl"
}

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def load_predictions(path: str):
    """Load predictions and organize by collection"""
    data = {}
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            task_id = str(item["task_id"])
            collection = item["Collection"]
            docs = [str(c["document_id"]) for c in item.get("contexts", [])]

            if collection not in data:
                data[collection] = {}
            data[collection][task_id] = docs

    return data

def load_all_strategies():
    """Load all strategies and organize by collection"""
    all_data = {}

    for strat_name, path in FILE_MAP.items():
        print(f"📥 Loading {strat_name}...")
        strat_data = load_predictions(path)

        for collection, tasks in strat_data.items():
            if collection not in all_data:
                all_data[collection] = {}
            if strat_name not in all_data[collection]:
                all_data[collection][strat_name] = {}
            all_data[collection][strat_name] = tasks

        print(f"   ✅ Loaded {sum(len(t) for t in strat_data.values())} tasks")

    return all_data

def rrf_fusion(rank_cache, weights, k, strats):
    """RRF fusion for a set of strategies"""
    sample_strat = strats[0]
    qids = list(rank_cache[sample_strat].keys())
    combined = {}

    for qid in qids:
        scores = {}
        for s in strats:
            w = weights.get(s, 1.0)
            for rank, did in enumerate(rank_cache[s].get(qid, []), 1):
                scores[did] = scores.get(did, 0.0) + w * (1.0 / (k + rank))

        combined[qid] = [d for d, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

    return combined

# ==============================================================================
# 4. MAIN EXECUTION
# ==============================================================================

def main():
    print("\n" + "="*80)
    print("🚀 NESTED RRF ENSEMBLE - TEST SET")
    print("="*80)
    print("Using best configurations from dev set")
    print("No evaluation (test set has no ground truth)")
    print("="*80)

    # Load all strategies
    print("\n📥 LOADING ALL STRATEGIES")
    print("="*80)
    all_data = load_all_strategies()

    print("\n📊 Collections found:")
    for collection in all_data.keys():
        num_tasks = len(next(iter(all_data[collection].values())))
        print(f"   • {collection}: {num_tasks} tasks")

    # Process each collection
    all_results = []

    for collection, cache in all_data.items():
        print(f"\n{'='*80}")
        print(f"🎯 PROCESSING: {collection.upper()}")
        print("="*80)

        if collection not in FINAL_CONFIGS:
            print(f"⚠️ No config found for {collection}, skipping...")
            continue

        cfg = FINAL_CONFIGS[collection]

        print(f"Configuration:")
        print(f"   • k_final: {cfg['k_final']}")
        print(f"   • k_internal: {cfg['k_internal']}")
        print(f"   • Final weights: {cfg['weights']}")

        # Stage 1: Weak Consensus
        print("\n🔄 Stage 1: Creating Weak Group Consensus...")
        weak_strats = ["Anchor_Keyword", "COT", "Hyde"]
        weak_res = rrf_fusion(cache, WEAK_INTERNAL_WEIGHTS, cfg["k_internal"], weak_strats)
        cache["Weak_Group_Consensus"] = weak_res
        print(f"   ✅ Created consensus for {len(weak_res)} tasks")

        # Stage 2: Final Nested Fusion
        print("\n🔄 Stage 2: Final Nested RRF Fusion...")
        final_strats = ["Structural_Minimal", "Corpus_Specific", "Weak_Group_Consensus"]
        final_rankings = rrf_fusion(cache, cfg["weights"], cfg["k_final"], final_strats)
        print(f"   ✅ Final ensemble for {len(final_rankings)} tasks")

        # Save results
        output_path = f"{OUTPUT_DIR}/{collection}_nested_rrf_final.jsonl"
        with open(output_path, 'w', encoding='utf-8') as f:
            for qid, docs in final_rankings.items():
                contexts = [
                    {"document_id": d, "score": 1.0/(i+1)}
                    for i, d in enumerate(docs[:100])
                ]
                f.write(json.dumps({
                    "task_id": qid,
                    "Collection": collection,
                    "contexts": contexts
                }, ensure_ascii=False) + "\n")

        print(f"   💾 Saved: {output_path}")

        all_results.append({
            "collection": collection,
            "num_tasks": len(final_rankings),
            "output_file": output_path
        })

    # Merge all collections into single file
    print(f"\n{'='*80}")
    print("🔗 MERGING ALL COLLECTIONS")
    print("="*80)

    merged_file = f"{OUTPUT_DIR}/test_set_nested_rrf_ALL.jsonl"
    with open(merged_file, 'w', encoding='utf-8') as out_f:
        for result in all_results:
            with open(result["output_file"], 'r', encoding='utf-8') as in_f:
                for line in in_f:
                    out_f.write(line)

    print(f"✅ Merged file: {merged_file}")

    # Summary
    print(f"\n{'='*80}")
    print("📊 SUMMARY")
    print("="*80)

    total_tasks = sum(r["num_tasks"] for r in all_results)
    print(f"\nTotal tasks processed: {total_tasks}")
    print(f"\nPer-collection breakdown:")
    for result in all_results:
        print(f"   • {result['collection']}: {result['num_tasks']} tasks")

    print(f"\n📁 Output files:")
    print(f"   • Per-collection: {OUTPUT_DIR}/<collection>_nested_rrf_final.jsonl")
    print(f"   • Merged: {merged_file}")

    print(f"\n{'='*80}")
    print("✅ NESTED RRF ENSEMBLE COMPLETE!")
    print("="*80)
    print("\n⚠️ Note: No evaluation performed (test set has no ground truth)")
    print("   Submit the merged file for official evaluation.")

if __name__ == "__main__":
    main()


# Final LLM Reranking

In [ ]:
# ==============================================================================
# DEEPSEEK-R1 RERANKING - TEST SET (NESTED ENSEMBLE)
# Input: test_set_nested_rrf_ALL.jsonl
# Rewrites: minimal_rewrites.json
# No evaluation (test set has no ground truth)
# ==============================================================================

import os, json, time, requests, re
import numpy as np
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

BASE_PATH = "/content/drive/MyDrive/Project"

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

INPUT_FILE = f"{BASE_PATH}/test_set_final_ensemble/test_set_nested_rrf_ALL.jsonl"
REWRITES_FILE = f"{BASE_PATH}/test_set/minimal_rewrites.json"

# Corpus files
CORPUS_FILES = {
    "clapnq": f"{BASE_PATH}/corpora/clapnq.jsonl",
    "fiqa": f"{BASE_PATH}/corpora/fiqa.jsonl",
    "govt": f"{BASE_PATH}/corpora/govt.jsonl",
    "ibmcloud": f"{BASE_PATH}/corpora/cloud.jsonl"
}

RUN_TAG = time.strftime("%Y%m%d_%H%M%S") + "_R1_TEST_SET"
OUTPUT_BASE = f"{BASE_PATH}/test_set_r1_reranked/{RUN_TAG}"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Azure DeepSeek-R1
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

TOP_K_RERANK = 20
DOC_TRUNCATE = 2500
LLM_WEIGHTS = [0.3, 0.5, 0.7]

total_input_tokens = 0
total_output_tokens = 0

# ==============================================================================
# 2. PROMPT
# ==============================================================================

LISTWISE_SYSTEM = """You are an expert document ranker.
Your task is to rank documents based on their relevance to a user query.

OUTPUT FORMAT:
Provide a JSON array with rankings: [{"rank": 1, "doc_id": index, "score": 95}, ...]
- Use integer scores from 0-100
- Higher scores indicate greater relevance
- Include all documents provided
"""

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def add_usage(prompt_tokens: int, completion_tokens: int):
    global total_input_tokens, total_output_tokens
    total_input_tokens += prompt_tokens
    total_output_tokens += completion_tokens

def get_cost():
    return (total_input_tokens * 0.20 / 1e6) + (total_output_tokens * 0.50 / 1e6)

def load_jsonl(path: str) -> List[dict]:
    data = []
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def load_json(path: str) -> Dict:
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    if not os.path.exists(path):
        return corpus
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            did = str(j.get("_id") or j.get("id"))
            text = (j.get("title", "") + " " + (j.get("text") or j.get("contents") or "")).strip()
            corpus[did] = text
    return corpus

def load_all_corpora() -> Dict[str, Dict[str, str]]:
    """Load all 4 corpora into memory"""
    all_corpora = {}
    print("📥 Loading all corpora...")
    for collection, path in CORPUS_FILES.items():
        all_corpora[collection] = load_corpus(path)
        print(f"   ✅ {collection}: {len(all_corpora[collection])} documents")
    return all_corpora

def load_rewrites(path: str) -> Dict[str, str]:
    rewrites_map = {}
    rewrites_data = load_json(path)

    if isinstance(rewrites_data, list):
        for item in rewrites_data:
            qid = str(item.get("task_id") or item.get("qid"))
            rewrite = item.get("rewritten_query") or item.get("rewrite")
            if qid and rewrite:
                rewrites_map[qid] = rewrite
    elif isinstance(rewrites_data, dict):
        rewrites_map = {str(k): str(v) for k, v in rewrites_data.items()}

    print(f"✅ Loaded {len(rewrites_map)} rewrites")
    return rewrites_map

def minmax_norm(scores: List[float]) -> np.ndarray:
    arr = np.array(scores, dtype=np.float32)
    if arr.max() == arr.min():
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

def call_deepseek_r1_listwise(query: str, docs: List[Dict]) -> Tuple[str, str]:
    if not AZURE_DS_KEY:
        return "", ""

    docs_block = ""
    for i, doc in enumerate(docs):
        text = doc.get("text", "")[:DOC_TRUNCATE]
        docs_block += f"--- DOCUMENT [{i}] ---\n{text}\n\n"

    user_msg = f"QUERY: {query}\n\n{docs_block}\nRank these {len(docs)} documents by relevance."

    headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
    payload = {
        "model": "DeepSeek-R1",
        "messages": [
            {"role": "system", "content": LISTWISE_SYSTEM},
            {"role": "user", "content": user_msg}
        ],
        "max_tokens": 2500,
        "temperature": 0.0
    }

    for attempt in range(3):
        try:
            response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=120)
            if response.status_code == 200:
                res_json = response.json()
                usage = res_json.get('usage', {})
                add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))

                choice = res_json['choices'][0]['message']
                reasoning = choice.get('reasoning_content', '')
                content = choice.get('content', '').strip()

                return reasoning, content
            elif response.status_code == 429:
                time.sleep(5)
        except Exception as e:
            time.sleep(2)
    return "", ""

def parse_listwise_response(response: str) -> List[Dict]:
    try:
        return json.loads(response)
    except:
        pass

    try:
        match = re.search(r'\[\s*\{.*\}\s*\]', response, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except:
        pass
    return []

# ==============================================================================
# 4. MAIN PROCESSING
# ==============================================================================

def main():
    print("=" * 80)
    print("🚀 DEEPSEEK-R1 RERANKING - TEST SET (NESTED ENSEMBLE)")
    print("=" * 80)
    print(f"Input: {INPUT_FILE}")
    print(f"Rewrites: {REWRITES_FILE}")
    print(f"Output: {OUTPUT_BASE}")
    print("=" * 80)

    # Load data
    print("\n📥 Loading datasets...")
    predictions = load_jsonl(INPUT_FILE)
    all_corpora = load_all_corpora()
    rewrites_map = load_rewrites(REWRITES_FILE)

    print(f"\n✅ Loaded {len(predictions)} predictions")

    # Statistics
    queries_with_rewrites = sum(1 for p in predictions if str(p["task_id"]) in rewrites_map)
    print(f"📊 Queries with rewrites: {queries_with_rewrites}/{len(predictions)} ({queries_with_rewrites/len(predictions)*100:.1f}%)")

    # Reranking
    print("\n🎯 Processing with DeepSeek-R1...")
    cache = []
    reasoning_log = []

    for pred in tqdm(predictions, desc="Reranking", ncols=100):
        qid = str(pred["task_id"])
        collection = pred["Collection"]

        # Get query (rewritten or task_id as fallback)
        query = rewrites_map.get(qid, qid)

        # Get corpus
        corpus_map = all_corpora.get(collection, {})

        # Top-20 contexts
        contexts = pred.get("contexts", [])
        top_20 = contexts[:TOP_K_RERANK]
        rest = contexts[TOP_K_RERANK:]

        # Add document text
        for ctx in top_20:
            ctx["text"] = corpus_map.get(str(ctx["document_id"]), "")

        # API Call
        reasoning, response = call_deepseek_r1_listwise(query, top_20)

        reasoning_log.append({
            "qid": qid,
            "collection": collection,
            "query": query,
            "reasoning": reasoning[:500]
        })

        ranked_json = parse_listwise_response(response)

        # Map scores
        llm_scores_map = {int(item['doc_id']): item['score'] for item in ranked_json if 'doc_id' in item}

        final_top = []
        for i, doc in enumerate(top_20):
            raw_s = llm_scores_map.get(i, 0.0)
            doc["llm_score"] = raw_s / 100.0 if raw_s > 1.0 else raw_s
            final_top.append(doc)

        cache.append({
            "task_id": qid,
            "Collection": collection,
            "top_ctxs": final_top,
            "rest": rest,
            "orig_scores": [float(c.get("score", 0.0)) for c in final_top],
            "llm_scores": [float(c.get("llm_score", 0.0)) for c in final_top]
        })

    # Save reasoning log
    reasoning_file = os.path.join(OUTPUT_BASE, "reasoning_log.json")
    with open(reasoning_file, "w", encoding="utf-8") as f:
        json.dump(reasoning_log, f, indent=2)
    print(f"\n💾 Reasoning log saved: {reasoning_file}")

    # Generate outputs
    print("\n💾 Generating output files...")

    # LLM ONLY
    llm_file = os.path.join(OUTPUT_BASE, "test_set_R1_ONLY.jsonl")
    with open(llm_file, "w", encoding="utf-8") as f:
        for item in cache:
            ctxs = [{"document_id": c["document_id"], "score": c["llm_score"]} for c in item["top_ctxs"]]
            ctxs.sort(key=lambda x: x["score"], reverse=True)
            f.write(json.dumps({
                "task_id": item["task_id"],
                "Collection": item["Collection"],
                "contexts": ctxs + item["rest"]
            }, ensure_ascii=False) + "\n")
    print(f"   ✅ R1-Only: {llm_file}")

    # HYBRID VARIANTS
    for w in LLM_WEIGHTS:
        hybrid_file = os.path.join(OUTPUT_BASE, f"test_set_HYBRID_{w}.jsonl")
        with open(hybrid_file, "w", encoding="utf-8") as f:
            for item in cache:
                n_orig = minmax_norm(item["orig_scores"])
                n_llm = minmax_norm(item["llm_scores"])
                combined = ((1-w) * n_orig) + (w * n_llm)

                ctxs = [{"document_id": c["document_id"], "score": float(combined[i])} for i, c in enumerate(item["top_ctxs"])]
                ctxs.sort(key=lambda x: x["score"], reverse=True)

                f.write(json.dumps({
                    "task_id": item["task_id"],
                    "Collection": item["Collection"],
                    "contexts": ctxs + item["rest"]
                }, ensure_ascii=False) + "\n")
        print(f"   ✅ Hybrid {w}: {hybrid_file}")

    # Summary
    print("\n" + "=" * 80)
    print("🎉 RERANKING COMPLETE")
    print("=" * 80)
    print(f"\n💵 Total Cost: ${get_cost():.3f}")
    print(f"💰 Total Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")
    print(f"\n📁 Output files:")
    print(f"   • R1-Only: test_set_R1_ONLY.jsonl")
    print(f"   • Hybrid 0.3: test_set_HYBRID_0.3.jsonl")
    print(f"   • Hybrid 0.5: test_set_HYBRID_0.5.jsonl")
    print(f"   • Hybrid 0.7: test_set_HYBRID_0.7.jsonl")
    print(f"   • Reasoning log: reasoning_log.json")
    print(f"\n⚠️ Note: No evaluation performed (test set has no ground truth)")
    print("   Choose best configuration based on dev set results for submission.")

if __name__ == "__main__":
    main()


# Final Submission File

In [ ]:
# ==============================================================================
# EXTRACT TOP-10 FOR SUBMISSION
# Input: test_set_nested_rrf_ALL.jsonl
# Output: test_set_TOP10_submission.jsonl (format checker compliant)
# ==============================================================================

import json
import os

BASE_PATH = "/content/drive/MyDrive/Project"
INPUT_FILE = f"{BASE_PATH}/test_set_final_ensemble/test_set_nested_rrf_ALL.jsonl"
OUTPUT_FILE = f"{BASE_PATH}/test_set_final_ensemble/test_set_TOP10_submission.jsonl"

print("=" * 80)
print("🚀 EXTRACTING TOP-10 CONTEXTS FOR SUBMISSION")
print("=" * 80)
print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print("=" * 80)

# Load, trim to top-10, and save
count = 0
with open(INPUT_FILE, "r", encoding="utf-8") as infile:
    with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
        for line in infile:
            if not line.strip():
                continue

            data = json.loads(line)

            # Keep only top-10 contexts
            top_10_contexts = data.get("contexts", [])[:10]

            # Create output with required format
            output = {
                "task_id": data["task_id"],
                "Collection": data["Collection"],
                "contexts": top_10_contexts
            }

            outfile.write(json.dumps(output, ensure_ascii=False) + "\n")
            count += 1

print(f"\n✅ Processed {count} predictions")
print(f"✅ Each prediction now contains exactly {10 if count > 0 else 0} contexts (max)")

# Verification: Check first 3 entries
print("\n" + "=" * 80)
print("📋 FIRST 3 PREDICTIONS (verification):")
print("=" * 80)

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i > 3:
            break
        data = json.loads(line)
        print(f"\n--- Prediction {i} ---")
        print(f"Task ID:    {data['task_id']}")
        print(f"Collection: {data['Collection']}")
        print(f"Contexts:   {len(data['contexts'])} documents")
        if data['contexts']:
            print(f"Top doc:    {data['contexts'][0]['document_id']} (score: {data['contexts'][0]['score']:.6f})")

print("\n" + "=" * 80)
print("✅ File ready for submission!")
print(f"📁 Location: {OUTPUT_FILE}")
print("=" * 80)


# Format checker

In [ ]:
# ==============================================================================
# RUN OFFICIAL FORMAT CHECKER
# Uses the MT-RAG evaluation script
# ==============================================================================

import os
import subprocess

BASE_PATH = "/content/drive/MyDrive/Project"

# Paths
FORMAT_CHECKER_SCRIPT = f"{BASE_PATH}/test_set/format_checker.py"
INPUT_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"
PREDICTION_FILE = f"{BASE_PATH}/test_set_final_ensemble/test_set_TOP10_submission.jsonl"

print("=" * 80)
print("🔍 RUNNING OFFICIAL MT-RAG FORMAT CHECKER")
print("=" * 80)
print(f"Format checker: {FORMAT_CHECKER_SCRIPT}")
print(f"Input file:     {INPUT_FILE}")
print(f"Prediction:     {PREDICTION_FILE}")
print(f"Mode:           retrieval_taska")
print("=" * 80)

# Check if format checker exists
if not os.path.exists(FORMAT_CHECKER_SCRIPT):
    print(f"\n❌ ERROR: Format checker script not found!")
    print(f"Expected location: {FORMAT_CHECKER_SCRIPT}")
    print("\n💡 Make sure you have downloaded the MT-RAG evaluation scripts:")
    print("   https://github.com/primeqa/mtrag")
    exit(1)

# Check if input file exists
if not os.path.exists(INPUT_FILE):
    print(f"\n❌ ERROR: Input file not found!")
    print(f"Expected location: {INPUT_FILE}")
    exit(1)

# Check if prediction file exists
if not os.path.exists(PREDICTION_FILE):
    print(f"\n❌ ERROR: Prediction file not found!")
    print(f"Expected location: {PREDICTION_FILE}")
    exit(1)

# Run format checker
print("\n🔍 Running format checker...\n")

cmd = [
    "python",
    FORMAT_CHECKER_SCRIPT,
    "--input_file", INPUT_FILE,
    "--prediction_file", PREDICTION_FILE,
    "--mode", "retrieval_taska"
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=60
    )

    # Print output
    print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    # Check exit code
    if result.returncode == 0:
        print("\n" + "=" * 80)
        print("✅ FORMAT CHECKER COMPLETED SUCCESSFULLY")
        print("=" * 80)
    else:
        print("\n" + "=" * 80)
        print(f"❌ FORMAT CHECKER FAILED (exit code: {result.returncode})")
        print("=" * 80)

except subprocess.TimeoutExpired:
    print("\n❌ ERROR: Format checker timed out after 60 seconds")
except Exception as e:
    print(f"\n❌ ERROR: {e}")

print("\n📁 Files checked:")
print(f"   • Input:      {INPUT_FILE}")
print(f"   • Prediction: {PREDICTION_FILE}")


# Generation input format

In [ ]:
# ==============================================================================
# PREPARE FILE FOR RAG TASK C
# Adds: input field (conversation), text in contexts
# ==============================================================================

import json
import os
from tqdm.auto import tqdm

BASE_PATH = "/content/drive/MyDrive/Project"

# Input/Output
RETRIEVAL_FILE = f"{BASE_PATH}/test_set_final_ensemble/test_set_TOP10_submission.jsonl"
INPUT_TASKS_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"  # Has conversation history
OUTPUT_FILE = f"{BASE_PATH}/test_set_final_ensemble/test_set_TOP10_RAG_taskC.jsonl"

# Corpus files
CORPUS_FILES = {
    "clapnq": f"{BASE_PATH}/corpora/clapnq.jsonl",
    "fiqa": f"{BASE_PATH}/corpora/fiqa.jsonl",
    "govt": f"{BASE_PATH}/corpora/govt.jsonl",
    "ibmcloud": f"{BASE_PATH}/corpora/cloud.jsonl"
}

print("=" * 80)
print("📝 PREPARING FILE FOR RAG TASK C")
print("=" * 80)
print(f"Retrieval results: {RETRIEVAL_FILE}")
print(f"Input tasks:       {INPUT_TASKS_FILE}")
print(f"Output:            {OUTPUT_FILE}")
print("=" * 80)

# Load corpus
def load_corpus(path: str) -> dict:
    corpus = {}
    if not os.path.exists(path):
        return corpus

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            doc_id = str(data.get("_id") or data.get("id"))
            title = data.get("title", "")
            text = data.get("text") or data.get("contents", "")
            full_text = f"{title}\n{text}".strip() if title else text
            corpus[doc_id] = full_text

    return corpus

print("\n📥 Loading corpora...")
all_corpora = {}
for collection, path in CORPUS_FILES.items():
    all_corpora[collection] = load_corpus(path)
    print(f"   ✅ {collection}: {len(all_corpora[collection])} documents")

# Load input tasks (for conversation history)
print(f"\n📥 Loading input tasks (conversation history)...")
input_tasks = {}
with open(INPUT_TASKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        data = json.loads(line)
        task_id = data["task_id"]
        input_tasks[task_id] = data

print(f"   ✅ Loaded {len(input_tasks)} input tasks")

# Load retrieval results
print(f"\n📥 Loading retrieval results...")
retrieval_results = {}
with open(RETRIEVAL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        data = json.loads(line)
        task_id = data["task_id"]
        retrieval_results[task_id] = data

print(f"   ✅ Loaded {len(retrieval_results)} retrieval results")

# Merge and prepare output
print(f"\n🔄 Merging data for Task C format...")
count_success = 0
count_missing_input = 0
count_missing_text = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
    for task_id, retrieval_data in tqdm(retrieval_results.items(), desc="Processing"):

        # Get input task (conversation history)
        input_task = input_tasks.get(task_id)
        if not input_task:
            count_missing_input += 1
            print(f"⚠️ Warning: No input found for task_id {task_id}")
            continue

        collection = retrieval_data["Collection"]
        corpus = all_corpora.get(collection, {})

        # Add text to contexts
        contexts_with_text = []
        for ctx in retrieval_data.get("contexts", []):
            doc_id = str(ctx["document_id"])
            doc_text = corpus.get(doc_id, "")

            if not doc_text:
                count_missing_text += 1

            context = {
                "document_id": ctx["document_id"],
                "text": doc_text,
                "score": ctx["score"],
                "source": "",  # Optional field
                "title": ""    # Optional field
            }
            contexts_with_text.append(context)

        # Build Task C format
        output = {
            "task_id": task_id,
            "Collection": collection,
            "input": input_task.get("input", []),  # ✅ Conversation history
            "contexts": contexts_with_text
        }

        # Optional fields from input task
        if "conversation_id" in input_task:
            output["conversation_id"] = input_task["conversation_id"]
        if "task_type" in input_task:
            output["task_type"] = input_task["task_type"]
        if "turn" in input_task:
            output["turn"] = input_task["turn"]

        outfile.write(json.dumps(output, ensure_ascii=False) + "\n")
        count_success += 1

print(f"\n✅ Processing complete!")
print(f"   • Successful tasks: {count_success}")
print(f"   • Missing input:    {count_missing_input}")
print(f"   • Missing texts:    {count_missing_text}")

# Verification
print("\n" + "=" * 80)
print("📋 VERIFICATION - First prediction:")
print("=" * 80)

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    first = json.loads(f.readline())
    print(f"Task ID:         {first['task_id']}")
    print(f"Collection:      {first['Collection']}")
    print(f"Conversation ID: {first.get('conversation_id', 'N/A')}")
    print(f"Turn:            {first.get('turn', 'N/A')}")
    print(f"\nInput (conversation):")
    for i, turn in enumerate(first.get('input', [])):
        speaker = turn.get('speaker', 'unknown')
        text = turn.get('text', '')[:80]
        print(f"   {i+1}. [{speaker}]: {text}...")

    print(f"\nContexts: {len(first['contexts'])} documents")
    if first['contexts']:
        ctx = first['contexts'][0]
        print(f"   First context:")
        print(f"     • document_id: {ctx['document_id']}")
        print(f"     • score:       {ctx['score']:.6f}")
        print(f"     • text:        {ctx['text'][:100]}..." if ctx['text'] else "     • text:        [MISSING]")

print("\n" + "=" * 80)
print("✅ FILE READY FOR RAG TASK C")
print(f"📁 Output: {OUTPUT_FILE}")
print("=" * 80)


# Generation Task C

In [ ]:
# ================================================================
# HYBRID v5.7 — "v5.6 + User-Satisfaction Judge"
# ================================================================
# 🎯 v5.7 ADDITIONS:
#    • User-Satisfaction Judge (Stage 6b)
#    • Judges from USER perspective: "Does this satisfy what I asked?"
#    • Triggers on 30% of non-IDK answers
#    • All v5.6 features maintained
# ================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q -c /content/drive/MyDrive/Project/constraints.txt -r /content/drive/MyDrive/Project/requirements.txt
!pip install -q pandas numpy huggingface_hub tqdm requests

import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from huggingface_hub import login
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# HF LOGIN
# =========================
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if HF_TOKEN:
    login(token=HF_TOKEN)

# =========================
# AZURE DEEPSEEK CONFIGURATION
# =========================
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# =========================
# 🎯 v5.7 CONFIGURATION
# =========================
N_PASSAGES = 5

# Pipeline stages
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_MULTI_SAMPLING = True
ENABLE_SELECTIVE_JUDGE = True          # Stage 6a: Technical Judge
ENABLE_USER_SATISFACTION_JUDGE = True  # 🔥 NEW Stage 6b: User Judge
ENABLE_EXTRACTIVE_POLISH = True
ENABLE_COMPRESSION = False

# Answerability
ANSWERABILITY_BIAS = "ANSWERABLE"

# Retry
RETRY_IF_IDK_ON_ANSWERABLE = True
NUM_RETRIES = 2

# Span extraction
MIN_SPANS = 1
MAX_SPANS = 8
SPAN_RETRY_ATTEMPTS = 2

# Generation
NUM_CANDIDATES = 2
GENERATION_TEMPERATURE_1 = 0.0
GENERATION_TEMPERATURE_2 = 0.1

# Word count
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105

SUGGESTED_PARTIAL_MIN = 60
SUGGESTED_PARTIAL_TARGET = 70
SUGGESTED_PARTIAL_MAX = 85

# Extractiveness
TARGET_EXTRACTIVENESS = 0.32

# Soft trim
SOFT_TRIM_THRESHOLD = 300

# Technical Judge (Stage 6a)
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
JUDGE_EXTR_MIN = 0.10

# 🔥 NEW: User-Satisfaction Judge (Stage 6b)
USER_SAT_JUDGE_SAMPLE_RATE = 0.30           # Run on 30% of non-IDK answers
USER_SAT_CONFIDENCE_THRESHOLD = ["HIGH", "MEDIUM"]  # Regen on HIGH/MEDIUM confidence

# IDK & Conversational
IDK_PHRASE = "I do not have specific information"
MAX_WORDS_CONV = 50

# Debug
DEBUG_MODE = True
MAX_WORKERS = 5

# Set random seed for reproducibility
random.seed(42)

# =========================
# STATS TRACKER
# =========================
class StatsTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.answerability_counts = {"ANSWERABLE": 0, "PARTIAL": 0, "UNANSWERABLE": 0, "CONVERSATIONAL": 0}
        self.idk_count = 0
        self.full_answer_count = 0
        self.answerable_to_idk_count = 0
        self.span_retry_count = 0
        self.judge_trigger_count = 0
        self.judge_regenerate_count = 0
        # 🔥 NEW stats
        self.user_sat_judge_count = 0
        self.user_sat_unsatisfied_count = 0
        self.user_sat_regenerate_count = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def add_answerability(self, verdict: str):
        with self.lock:
            self.answerability_counts[verdict] = self.answerability_counts.get(verdict, 0) + 1

    def add_prediction(self, is_idk: bool, answerability: str = None):
        with self.lock:
            if is_idk:
                self.idk_count += 1
                if answerability in ["ANSWERABLE", "PARTIAL"]:
                    self.answerable_to_idk_count += 1
            else:
                self.full_answer_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_judge_trigger(self, regenerated: bool = False):
        with self.lock:
            self.judge_trigger_count += 1
            if regenerated:
                self.judge_regenerate_count += 1

    def add_user_sat_judge(self, unsatisfied: bool = False, regenerated: bool = False):
        with self.lock:
            self.user_sat_judge_count += 1
            if unsatisfied:
                self.user_sat_unsatisfied_count += 1
            if regenerated:
                self.user_sat_regenerate_count += 1

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        total = sum(self.answerability_counts.values())

        report = f"\n{'='*70}\n"
        report += "📊 GENERATION STATISTICS (v5.7 - User-Satisfaction Judge)\n"
        report += f"{'='*70}\n"

        report += f"\n💰 Token Usage:\n"
        report += f"   Input:  {self.total_input:,}\n"
        report += f"   Output: {self.total_output:,}\n"
        report += f"   Cost:   ${cost:.4f}\n"

        report += f"\n🎯 Answerability Predictions (Total: {total}):\n"
        for verdict, count in sorted(self.answerability_counts.items()):
            pct = (count / total * 100) if total > 0 else 0
            report += f"   {verdict:15s}: {count:4d} ({pct:5.1f}%)\n"

        total_preds = self.idk_count + self.full_answer_count
        report += f"\n📝 Final Predictions (Total: {total_preds}):\n"
        idk_pct = (self.idk_count / total_preds * 100) if total_preds > 0 else 0
        ans_pct = (self.full_answer_count / total_preds * 100) if total_preds > 0 else 0
        report += f"   IDK Responses:   {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
        report += f"   Full Answers:    {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"

        if self.span_retry_count > 0:
            report += f"\n🔄 Span Extraction Retries: {self.span_retry_count}\n"

        if self.judge_trigger_count > 0:
            judge_rate = (self.judge_trigger_count / total_preds * 100) if total_preds > 0 else 0
            regen_rate = (self.judge_regenerate_count / self.judge_trigger_count * 100) if self.judge_trigger_count > 0 else 0
            report += f"\n👨‍⚖️ Technical Judge (Stage 6a):\n"
            report += f"   Triggered:    {self.judge_trigger_count:4d} ({judge_rate:5.1f}% of tasks)\n"
            report += f"   Regenerated:  {self.judge_regenerate_count:4d} ({regen_rate:5.1f}% of judged)\n"

        # 🔥 NEW stats
        if self.user_sat_judge_count > 0:
            usr_judge_rate = (self.user_sat_judge_count / total_preds * 100) if total_preds > 0 else 0
            unsat_rate = (self.user_sat_unsatisfied_count / self.user_sat_judge_count * 100) if self.user_sat_judge_count > 0 else 0
            regen_rate = (self.user_sat_regenerate_count / self.user_sat_unsatisfied_count * 100) if self.user_sat_unsatisfied_count > 0 else 0
            report += f"\n👤 User-Satisfaction Judge (Stage 6b):\n"
            report += f"   Evaluated:    {self.user_sat_judge_count:4d} ({usr_judge_rate:5.1f}% of tasks)\n"
            report += f"   Unsatisfied:  {self.user_sat_unsatisfied_count:4d} ({unsat_rate:5.1f}% of evaluated)\n"
            report += f"   Regenerated:  {self.user_sat_regenerate_count:4d} ({regen_rate:5.1f}% of unsatisfied)\n"

        if self.answerable_to_idk_count > 0:
            answerable_total = self.answerability_counts.get("ANSWERABLE", 0) + self.answerability_counts.get("PARTIAL", 0)
            false_refusal_rate = (self.answerable_to_idk_count / answerable_total * 100) if answerable_total > 0 else 0
            report += f"\n⚠️  ANSWERABLE → IDK Cases: {self.answerable_to_idk_count} ({false_refusal_rate:.1f}%)\n"

        report += f"\n📖 v5.7 Features:\n"
        report += f"   MAX_SPANS: 8\n"
        report += f"   Target Extractiveness: 32%\n"
        report += f"   Technical Judge: ENABLED\n"
        report += f"   🔥 User-Satisfaction Judge: ENABLED ({int(USER_SAT_JUDGE_SAMPLE_RATE*100)}% sample)\n"

        report += f"{'='*70}\n"
        return report

tracker = StatsTracker()

# =========================
# HELPERS (SAME as v5.6)
# =========================
def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])

def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s

def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r"^(Agent:|Assistant:)\s*", "", t.strip(), flags=re.IGNORECASE)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

def soft_trim_to_word_limit_by_sentences(text: str, max_words: int) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    kept = []
    for sentence in sentences:
        candidate = " ".join(kept + [sentence])
        if word_count(candidate) <= max_words:
            kept.append(sentence)
        else:
            break

    result = " ".join(kept).strip()

    if not result and sentences:
        result = " ".join(sentences[:3])

    return result

def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    canonical = IDK_PHRASE.lower()
    if re.sub(r"[.\s]+", " ", low).strip() == re.sub(r"[.\s]+", " ", canonical).strip():
        return True
    idk_indicators = [
        "i don't know", "i do not know",
        "do not have enough information", "don't have enough information",
        "do not have specific information", "don't have specific information",
        "insufficient information", "not enough information",
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 24:
        return True
    return False

def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return IDK_PHRASE
    return t

def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words)-n+1))

def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0

    passage_text = ' '.join([c.get('text', '') for c in contexts])
    passage_grams = get_ngrams(passage_text, 4)

    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0

# =========================
# MAIN CLASS v5.7
# =========================
class MTRAGGenerationV57:
    def __init__(self):
        print("🔧 v5.7 — User-Satisfaction Judge + v5.6 Features")
        print("✅ DeepSeek via Azure initialized")
        print(f"⚡ Workers: {MAX_WORKERS}")
        print(f"✅ Technical Judge (Stage 6a): ENABLED")
        print(f"🔥 User-Satisfaction Judge (Stage 6b): ENABLED ({int(USER_SAT_JUDGE_SAMPLE_RATE*100)}% sample)")
        print(f"✅ All v5.6 features maintained")

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        print(f"\n📂 Loading: {input_file}")
        tasks = []
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))
        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"✅ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks: List[Dict], output_file: str):
        print(f"\n💾 Saving: {output_file}")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as f:
            for task in tasks:
                f.write(json.dumps(task, ensure_ascii=False) + "\n")
        print(f"✅ Saved {len(tasks)} predictions")

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'^hi\b', r'^hello\b', r'^hey\b', r'^thanks?\b', r'^thank you\b',
            r'^ok\b', r'^okay\b', r'^i see\b', r'^got it\b', r'^alright\b',
        ]
        return any(re.match(p, q_low) for p in conversational_patterns)

    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        """Build history (USER turns only)"""
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_question:
                break
            if speaker == "user":
                history_lines.append(f"User: {text}")
        return history_lines

    # =========================
    # ANSWERABILITY (SAME as v5.6)
    # =========================
    def predict_answerability(self, question: str, contexts: List[Dict], history_lines: List[str]) -> str:
        if not contexts:
            return "UNANSWERABLE"

        q_keywords = set(re.findall(r'\b[A-Z][a-z]+|\b\d+|\b\w{4,}\b', question))
        passage_text = " ".join([c.get("text", "").lower() for c in contexts])
        q_keywords_lower = {k.lower() for k in q_keywords if len(k) > 2}

        if q_keywords_lower:
            overlap_ratio = sum(1 for kw in q_keywords_lower if kw in passage_text) / len(q_keywords_lower)
            if overlap_ratio > 0.40:
                return "ANSWERABLE"

        passages_block = "DOCUMENTS:\n"
        for i, ctx in enumerate(contexts, 1):
            text = (ctx.get("text", "") or "").strip()
            passages_block += f"[PASSAGE {i}]\n{text}\n\n"

        history_block = ""
        if history_lines:
            history_block = "CONVERSATION HISTORY:\n" + "\n".join(history_lines[-2:]).strip() + "\n\n"

        system_msg = "You judge answerability. BIAS towards ANSWERABLE."
        user_msg = f"""ANSWERABILITY JUDGMENT:

ANSWERABLE: Docs contain relevant facts (even partial)
PARTIAL: Docs contain some info but missing key details
UNANSWERABLE: Temporal mismatch OR completely missing info

Bias: When uncertain → ANSWERABLE

{passages_block}
{history_block}QUESTION: {question}

Return JSON: {{"answerability": "ANSWERABLE"|"PARTIAL"|"UNANSWERABLE", "confidence": "HIGH"|"MEDIUM"|"LOW"}}"""

        try:
            response = self.generate_response_with_system(system_msg, user_msg, temperature=0.0, max_tokens=150)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                answerability = result.get("answerability", "").strip().upper()
                confidence = result.get("confidence", "MEDIUM").strip().upper()

                if answerability == "UNANSWERABLE" and confidence == "LOW":
                    return "ANSWERABLE"

                if answerability in {"ANSWERABLE", "PARTIAL", "UNANSWERABLE"}:
                    return answerability
        except:
            pass

        return "ANSWERABLE" if contexts else "UNANSWERABLE"

    # =========================
    # SPAN EXTRACTION (SAME as v5.6)
    # =========================
    def extract_relevant_spans_with_retry(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        answerability: str
    ) -> List[Dict]:

        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"   🔄 Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()

            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)

            if spans and len(spans) > 0:
                return spans

            if answerability == "ANSWERABLE" and attempt < SPAN_RETRY_ATTEMPTS:
                continue

            break

        if DEBUG_MODE:
            print(f"   ⚠️  Smart fallback (keyword-based)")
        return self.smart_fallback(question, contexts)

    def extract_relevant_spans_cot(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        attempt: int = 1
    ) -> List[Dict]:

        passages_block = ""
        for i, ctx in enumerate(contexts[:5], 1):
            text = (ctx.get("text", "") or "").strip()

            relevance_note = ""
            if i == 1:
                relevance_note = " [MOST RELEVANT]"
            elif i == 2:
                relevance_note = " [HIGHLY RELEVANT]"

            passages_block += f"[PASSAGE {i}]{relevance_note}\n{text}\n\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONVERSATION HISTORY:\n{recent}\n\n"

        urgency = "The answer IS in the passages - look carefully!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question.

{urgency}

{history_block}QUESTION: {question}

{passages_block}

RULES:
1. PRIORITIZE earlier passages ([MOST RELEVANT])
2. Extract VERBATIM sentences (exact quotes)
3. COMPLETENESS PRIORITY:
   - Include ALL sentences with proper nouns (names, places, organizations)
   - Include ALL sentences with numbers (dates, quantities, measurements)
   - Include ALL sentences needed for complete answer
4. Better to include 1 extra sentence than miss key entity
5. Max {MAX_SPANS} sentences (use all {MAX_SPANS} if needed for completeness)
6. If unanswerable, return empty extracted_spans: []

Return JSON:
{{
  "reasoning": "Brief explanation of what facts are being extracted",
  "extracted_spans": [
    {{"passage_id": 1, "sentence": "exact sentence"}},
    {{"passage_id": 2, "sentence": "another sentence"}}
  ]
}}
"""

        try:
            response = self.generate_response(span_prompt, temperature=0.0, max_tokens=800)

            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extracted_spans", [])
                reasoning = result.get("reasoning", "")

                if DEBUG_MODE and attempt == 1:
                    print(f"\n🔍 Extracted {len(spans)} spans")
                    if reasoning:
                        print(f"   Reasoning: {reasoning[:80]}...")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "document_id": f"span_from_passage_{span['passage_id']}",
                            "source": "cot_extracted",
                            "reference": True
                        })

                    if filtered:
                        return filtered

        except Exception as e:
            if DEBUG_MODE:
                print(f"⚠️  Span extraction failed: {e}")

        return []

    def smart_fallback(self, question: str, contexts: List[Dict]) -> List[Dict]:
        q_keywords = set(re.findall(r'\b[A-Z][a-z]+|\b\d+|\b\w{5,}\b', question))
        q_keywords_lower = {k.lower() for k in q_keywords if len(k) > 3}

        scored = []
        for idx, ctx in enumerate(contexts[:5]):
            text = ctx.get("text", "").lower()
            overlap = sum(1 for kw in q_keywords_lower if kw in text)
            position_bonus = (5 - idx) * 0.5
            score = overlap + position_bonus
            scored.append((score, ctx))

        scored.sort(reverse=True, key=lambda x: x[0])

        return [ctx for _, ctx in scored[:3]]

    # =========================
    # GENERATION (SAME as v5.6)
    # =========================
    def create_generation_prompt_v57(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str],
        answerability: str
    ) -> str:

        if answerability == "PARTIAL":
            word_suggestion = f"typically {SUGGESTED_PARTIAL_TARGET}w (simple questions may be shorter)"
        else:
            word_suggestion = f"typically {SUGGESTED_TARGET}w (adapt to question complexity)"

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"[{i}] {text}\n"
        spans_block += "\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONTEXT:\n{recent}\n\n"

        instruction = f"""Generate a natural answer using the EXTRACTED FACTS.

REQUIREMENTS:
• Length: {word_suggestion}
• Include ALL facts from EXTRACTED FACTS:
  ✓ EVERY name (person, place, organization)
  ✓ EVERY number (quantity, date, year, percentage, step number)
  ✓ EVERY key term from question
• Use EXACT wording for:
  - Proper nouns: copy verbatim
  - Numbers: copy exactly as written (including "1.", "2." for lists)
  - Technical terms: copy verbatim
• Paraphrase only: connecting words ("and", "also", "while")
• Target ~{int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap
• If EXTRACTED FACTS incomplete: "{IDK_PHRASE}"

🚫 CRITICAL: Missing ANY entity/number from EXTRACTED FACTS = FAILURE

{history_block}{spans_block}QUESTION:
{question}

ANSWER:"""

        return instruction

    # =========================
    # MULTI-SAMPLING (SAME as v5.6)
    # =========================
    def generate_candidates(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str],
        answerability: str
    ) -> List[str]:

        prompt = self.create_generation_prompt_v57(spans, question, history_lines, answerability)

        candidates = []

        cand1 = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_1, max_tokens=450)
        cand1 = normalize_idk_partial_safe(clean_model_output(cand1))
        candidates.append(cand1)

        if ENABLE_MULTI_SAMPLING and NUM_CANDIDATES >= 2:
            cand2 = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_2, max_tokens=450)
            cand2 = normalize_idk_partial_safe(clean_model_output(cand2))
            candidates.append(cand2)

        return candidates

    # =========================
    # CANDIDATE SELECTION (SAME as v5.6)
    # =========================
    def score_answer_quality(self, answer: str, answerability: str) -> float:

        if is_pure_idk(answer):
            return 0.0

        score = 0.0
        wc = word_count(answer)

        if answerability == "PARTIAL":
            target = SUGGESTED_PARTIAL_TARGET
            min_acceptable = 40
            max_acceptable = 100
        else:
            target = SUGGESTED_TARGET
            min_acceptable = 60
            max_acceptable = 130

        if min_acceptable <= wc <= max_acceptable:
            distance = abs(wc - target)
            score += max(0, 1.0 - (distance / 40.0)) * 0.3
        elif wc < min_acceptable:
            score += (wc / min_acceptable) * 0.15
        else:
            score += max(0, 1.0 - ((wc - max_acceptable) / 60.0)) * 0.15

        entities = len(re.findall(r'\b[A-Z][a-z]+', answer))
        score += min(entities / 10.0, 0.3)

        numbers = len(re.findall(r'\b\d+', answer))
        score += min(numbers / 5.0, 0.2)

        sentences = re.split(r'[.!?]+', answer)
        complete = [s for s in sentences if len(s.strip()) > 10]
        score += min(len(complete) / 5.0, 0.2)

        return score

    def select_best_candidate(self, candidates: List[str], answerability: str) -> str:

        if len(candidates) == 1:
            return candidates[0]

        scored = []
        for cand in candidates:
            score = self.score_answer_quality(cand, answerability)
            scored.append((score, cand))

        scored.sort(reverse=True, key=lambda x: x[0])
        return scored[0][1]

    # =========================
    # STAGE 6a: TECHNICAL JUDGE (SAME as v5.6)
    # =========================
    def should_use_judge(
        self,
        answer: str,
        answerability: str,
        spans: List[Dict],
        retry_count: int = 0
    ) -> Tuple[bool, str]:

        wc = word_count(answer)

        if answerability in ["ANSWERABLE", "PARTIAL"] and is_pure_idk(answer):
            return True, "false_refusal"

        if answerability == "ANSWERABLE":
            if wc < JUDGE_WC_MIN:
                return True, "too_short"
            if wc > JUDGE_WC_MAX:
                return True, "too_long"

        if not is_pure_idk(answer) and wc > 30:
            extr = calculate_extractiveness(answer, spans)
            if extr < JUDGE_EXTR_MIN:
                return True, "low_extractiveness"

        if retry_count >= NUM_RETRIES and is_pure_idk(answer):
            return True, "retry_exhausted"

        return False, ""

    def judge_answer_lightweight(
        self,
        answer: str,
        question: str,
        spans: List[Dict],
        trigger_reason: str
    ) -> Dict:

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"[{i}] {span.get('text', '')}\n"

        if "refusal" in trigger_reason:
            focus = "Can this question be answered using EXTRACTED FACTS?"
        elif "short" in trigger_reason:
            focus = "Are all key facts included? Is it complete?"
        elif "long" in trigger_reason:
            focus = "Is this concise? Any redundancy?"
        elif "extractiveness" in trigger_reason:
            focus = "Is this grounded in EXTRACTED FACTS (not hallucinating)?"
        else:
            focus = "Is this a good quality answer?"

        judge_prompt = f"""Quick quality check.

QUESTION: {question}

{spans_block}

ANSWER:
{answer}

FOCUS: {focus}

Return JSON:
{{
  "verdict": "ACCEPT" | "REGENERATE",
  "issue": "What's wrong (if REGENERATE)",
  "suggestion": "How to fix (if REGENERATE)"
}}

Rules:
• ACCEPT if answer is reasonable
• REGENERATE only if clear problem
"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "verdict": result.get("verdict", "ACCEPT").upper(),
                    "issue": result.get("issue", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except:
            pass

        return {"verdict": "ACCEPT", "issue": "", "suggestion": ""}

    # =========================
    # 🔥 STAGE 6b: USER-SATISFACTION JUDGE (NEW!)
    # =========================
    def user_satisfaction_judge(
        self,
        question: str,
        answer: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> Dict:
        """Judge from USER perspective: Does this satisfy what I asked?"""

        # Build context
        passages_preview = ""
        for i, ctx in enumerate(contexts[:5]):
            text = ctx.get('text', '')[:200]
            passages_preview += f"[{i+1}] {text}...\n"

        history_block = ""
        if history_lines:
            history_block = "CONVERSATION:\n" + "\n".join(history_lines[-3:]) + "\n\n"

        judge_prompt = f"""You are a USER who asked a question. Judge if the answer satisfies YOUR question.

{history_block}YOUR QUESTION:
{question}

AVAILABLE INFORMATION (what the system had access to):
{passages_preview}

SYSTEM'S ANSWER:
{answer}

TASK: As a user, would you be SATISFIED with this answer?

Consider:
1. **Intent Match**: Does the answer address what I was REALLY asking?
   - Not just "does it mention the same entities?"
   - But "does it answer the SPIRIT of my question?"

2. **Completeness for MY Needs**:
   - If I asked "how", does it explain the process?
   - If I asked "why", does it give reasons?
   - If I asked "what", does it give the definition/description?
   - If I asked for comparison, does it compare?

3. **Clarity**:
   - Can I understand the answer without re-reading?
   - Is it organized logically?

4. **Relevance**:
   - Does it focus on what I care about?
   - Or does it give me irrelevant details?

5. **Context Awareness**:
   - Given the conversation history, is this appropriate?
   - Does it answer MY follow-up properly?

IMPORTANT:
- Ignore minor wording differences
- Ignore if answer is slightly shorter/longer than ideal
- Focus on: "Did I get what I was looking for?"

Return JSON:
{{
  "satisfaction": "SATISFIED" | "UNSATISFIED",
  "confidence": "HIGH" | "MEDIUM" | "LOW",
  "reason": "Brief explanation of why satisfied/unsatisfied",
  "missing_aspect": "What the user wanted but didn't get (if unsatisfied)",
  "suggestion": "How to better answer this question (if unsatisfied)"
}}

Examples:

Q: "How do I install the SDK?"
A: "The SDK is a software development kit for iOS and Android."
→ UNSATISFIED (didn't explain HOW to install, just defined what SDK is)

Q: "How do I install the SDK?"
A: "Install via CocoaPods: add 'pod OpenWhisk' to Podfile, run 'pod install'."
→ SATISFIED (explains the process step-by-step)

Q: "What is the capital of France?"
A: "The capital of France is Paris, which is also its largest city."
→ SATISFIED (answers question, extra info is fine)

Q: "Why was the request rejected?"
A: "The request was rejected in 2023 by the committee."
→ UNSATISFIED (says WHEN and WHO, but not WHY)

Now judge the actual answer above.
"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=300)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "satisfaction": result.get("satisfaction", "SATISFIED").upper(),
                    "confidence": result.get("confidence", "MEDIUM").upper(),
                    "reason": result.get("reason", ""),
                    "missing_aspect": result.get("missing_aspect", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except Exception as e:
            if DEBUG_MODE:
                print(f"⚠️  User-satisfaction judge failed: {e}")

        return {"satisfaction": "SATISFIED", "confidence": "LOW", "reason": "Judge failed"}

    # =========================
    # POLISH (SAME as v5.6)
    # =========================
    def build_extractive_polish_prompt(self, spans: List[Dict], raw_answer: str) -> str:

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"[{i}] {span.get('text', '')}\n"

        polish_prompt = f"""{spans_block}

CURRENT ANSWER:
{raw_answer}

TASK: Revise to achieve ~{int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap with EXTRACTED FACTS.

RULES:
• Keep EXACT: names, numbers, dates, locations (4-6 word spans)
• Paraphrase: connecting words, transitions
• Target: ~{int(TARGET_EXTRACTIVENESS*100)}% of answer = exact phrases from FACTS
• Maintain natural flow
• Keep same length ({word_count(raw_answer)}w)

REVISED ANSWER:"""

        return polish_prompt

    # =========================
    # NORMALIZATION (SAME as v5.6)
    # =========================
    def soft_normalize(self, text: str, answerability: str) -> str:

        text = re.sub(r'\s+', ' ', text.strip())
        text = re.sub(r'\n{3,}', '\n\n', text)

        if is_pure_idk(text):
            return IDK_PHRASE

        wc = word_count(text)

        if wc > SOFT_TRIM_THRESHOLD:
            if DEBUG_MODE:
                print(f"   ⚠️  Very long ({wc}w) - soft trimming to 250w")
            text = soft_trim_to_word_limit_by_sentences(text, 250)

        return text

    # =========================
    # RETRY (SAME as v5.6)
    # =========================
    def create_answerable_retry_prompt(
        self,
        spans: List[Dict],
        history_lines: List[str],
        question: str,
        answerability: str
    ) -> str:

        word_suggestion = f"{SUGGESTED_TARGET}w" if answerability == "ANSWERABLE" else f"{SUGGESTED_PARTIAL_TARGET}w"

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"[{i}] {span.get('text', '')}\n"

        history_block = ""
        if history_lines:
            history_block = f"CONTEXT:\n{' '.join(history_lines[-2:])}\n\n"

        return f"""The answer IS in the EXTRACTED FACTS - find it!

{history_block}{spans_block}

QUESTION: {question}

Write typically {word_suggestion} using the facts above.
🚫 DO NOT output '{IDK_PHRASE}' - the answer is present!

ANSWER:"""

    # =========================
    # CONVERSATIONAL (SAME as v5.6)
    # =========================
    def create_conversational_prompt(self, history_lines: List[str], question: str) -> str:
        instruction = f"Reply naturally and briefly. Keep it under {MAX_WORDS_CONV} words."
        history_block = ""
        if history_lines:
            history_block = "\n".join(history_lines[-4:]).strip() + "\n"
        return f"{instruction}\n\n{history_block}User: {question}\nAssistant:"

    # =========================
    # LLM CALLS (SAME as v5.6)
    # =========================
    def generate_response_with_system(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }
        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get('usage', {})
                    tracker.add_usage(usage.get('prompt_tokens', 0), usage.get('completion_tokens', 0))
                    return res_json['choices'][0]['message']['content'].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                time.sleep(5)
        return ""

    def generate_response(self, prompt: str, temperature: float = 0.0, max_tokens: int = 360) -> str:
        return self.generate_response_with_system("You are a helpful assistant.", prompt, temperature, max_tokens)

    # =========================
    # 🔥 ENHANCED PROCESS TASK (with Stage 6b)
    # =========================
    def process_task(self, task: Dict) -> Dict:
        inputs = task.get("input", []) or []
        q = self.get_current_question(task)

        history_lines = self.build_history_lines(inputs, q)

        contexts_original = task.get("contexts", [])[:N_PASSAGES]

        # Conversational
        if self.is_conversational(q):
            prompt = self.create_conversational_prompt(history_lines, q)
            raw = self.generate_response(prompt, temperature=0.3, max_tokens=160)
            raw = clean_model_output(raw)
            if word_count(raw) > MAX_WORDS_CONV:
                raw = soft_trim_to_word_limit_by_sentences(raw, MAX_WORDS_CONV)

            task["contexts"] = contexts_original
            task["predictions"] = [{"text": raw}]
            task["debug_info"] = {"type": "CONVERSATIONAL"}
            tracker.add_answerability("CONVERSATIONAL")
            tracker.add_prediction(False)
            return task

        # Answerability
        ans = self.predict_answerability(q, contexts_original, history_lines)
        tracker.add_answerability(ans)

        debug_steps = {"answerability": ans} if DEBUG_MODE else {}

        if ans == "UNANSWERABLE":
            task["contexts"] = contexts_original
            task["predictions"] = [{"text": IDK_PHRASE}]
            task["debug_info"] = {"type": "UNANSWERABLE", "steps": debug_steps}
            tracker.add_prediction(True, ans)
            return task

        # Span extraction
        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(q, contexts_original, history_lines, ans)
        else:
            spans = contexts_original[:3]

        if DEBUG_MODE:
            debug_steps["spans_extracted"] = len(spans)

        if not spans:
            task["contexts"] = contexts_original
            task["predictions"] = [{"text": IDK_PHRASE}]
            task["debug_info"] = {"type": "NO_SPANS", "steps": debug_steps}
            tracker.add_prediction(True, ans)
            return task

        # Generate candidates
        candidates = self.generate_candidates(spans, q, history_lines, ans)
        final_answer = self.select_best_candidate(candidates, ans)

        # Retry if IDK on ANSWERABLE
        retry_count = 0
        if RETRY_IF_IDK_ON_ANSWERABLE and ans in {"ANSWERABLE", "PARTIAL"} and final_answer == IDK_PHRASE:
            for retry_num in range(1, NUM_RETRIES + 1):
                retry_prompt = self.create_answerable_retry_prompt(spans, history_lines, q, ans)
                raw2 = self.generate_response(retry_prompt, temperature=0.0 if retry_num == 1 else 0.15, max_tokens=450)
                raw2 = normalize_idk_partial_safe(clean_model_output(raw2))

                if raw2 and raw2 != IDK_PHRASE and word_count(raw2) >= 40:
                    final_answer = raw2
                    retry_count = retry_num
                    if DEBUG_MODE:
                        debug_steps[f"retry_{retry_num}"] = "✅"
                    break

        # STAGE 6a: Technical Judge
        if ENABLE_SELECTIVE_JUDGE and final_answer != IDK_PHRASE:
            should_judge, trigger_reason = self.should_use_judge(final_answer, ans, spans, retry_count)

            if should_judge:
                if DEBUG_MODE:
                    print(f"   🔍 Technical judge triggered: {trigger_reason}")

                tracker.add_judge_trigger(regenerated=False)
                judgment = self.judge_answer_lightweight(final_answer, q, spans, trigger_reason)

                if judgment["verdict"] == "REGENERATE":
                    if DEBUG_MODE:
                        print(f"   🔄 Regenerating: {judgment['issue'][:60]}...")

                    regen_prompt = f"""Previous answer had issues: {judgment['issue']}

Suggestion: {judgment['suggestion']}

{self.create_generation_prompt_v57(spans, q, history_lines, ans)}"""

                    regenerated = self.generate_response(regen_prompt, temperature=0.15, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))

                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_judge_trigger(regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["tech_judge_regenerated"] = "✅"

        # 🔥 STAGE 6b: User-Satisfaction Judge
        if ENABLE_USER_SATISFACTION_JUDGE and final_answer != IDK_PHRASE:
            # Sample based on configured rate
            if random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                sat_judgment = self.user_satisfaction_judge(q, final_answer, contexts_original, history_lines)

                unsatisfied = (sat_judgment['satisfaction'] == 'UNSATISFIED')
                tracker.add_user_sat_judge(unsatisfied=unsatisfied, regenerated=False)

                if DEBUG_MODE:
                    print(f"   👤 User satisfaction: {sat_judgment['satisfaction']} ({sat_judgment['confidence']})")
                    if unsatisfied:
                        print(f"      Reason: {sat_judgment['reason'][:80]}...")

                # Regenerate if unsatisfied with high/medium confidence
                if unsatisfied and sat_judgment['confidence'] in USER_SAT_CONFIDENCE_THRESHOLD:
                    if DEBUG_MODE:
                        print(f"   🔄 Regenerating for user satisfaction...")

                    regen_prompt = f"""Previous answer didn't satisfy the user's INTENT.

USER'S QUESTION: {q}

PREVIOUS ANSWER: {final_answer}

ISSUE: {sat_judgment['reason']}
MISSING: {sat_judgment['missing_aspect']}
HOW TO FIX: {sat_judgment['suggestion']}

{self.create_generation_prompt_v57(spans, q, history_lines, ans)}

Focus on answering what the user REALLY wants to know."""

                    regenerated = self.generate_response(regen_prompt, temperature=0.2, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))

                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_user_sat_judge(unsatisfied=False, regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["user_sat_regenerated"] = "✅"

        # Polish
        if ENABLE_EXTRACTIVE_POLISH and final_answer != IDK_PHRASE:
            polish_prompt = self.build_extractive_polish_prompt(spans, final_answer)
            polished = self.generate_response(polish_prompt, temperature=0.0, max_tokens=400)
            polished = normalize_idk_partial_safe(clean_model_output(polished))

            if polished and polished != IDK_PHRASE:
                final_answer = polished
                if DEBUG_MODE:
                    debug_steps["polished"] = "✅"

        # Soft normalization
        final_answer = self.soft_normalize(final_answer, ans)

        if DEBUG_MODE:
            debug_steps["final_wc"] = word_count(final_answer)

        task["contexts"] = contexts_original
        task["contexts_used"] = [s.get("text", "")[:100] for s in spans]
        task["predictions"] = [{"text": final_answer}]
        task["debug_info"] = {
            "type": ans,
            "answer_preview": final_answer[:80],
            "steps": debug_steps
        }

        tracker.add_prediction(final_answer == IDK_PHRASE, ans)
        return task

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None):
        print("\n" + "=" * 70)
        print("📝 v5.7 — User-Satisfaction Judge + v5.6 Features")
        print("=" * 70)
        tasks = self.load_tasks(input_file, max_tasks)

        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}

            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="v5.7"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task

                    wc = word_count(out_task["predictions"][0]["text"])
                    is_idk = (out_task["predictions"][0]["text"] == IDK_PHRASE)
                    debug = out_task.get("debug_info", {})
                    ans_type = debug.get("type", "?")

                    tqdm.write(f"[{idx+1}] {wc}w | {ans_type}")
                except Exception as e:
                    tqdm.write(f"❌ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)
        print("\n✅ v5.7 COMPLETE!")
        return output_file

# ================================================================
# RUN
# ================================================================
INPUT_FILE = "/content/drive/MyDrive/Project/test_set_final_ensemble/test_set_TOP10_RAG_taskC.jsonl"
PREDICTIONS_OUTPUT = "/content/drive/MyDrive/Project/outputs/AILS-NTUA_taskC.jsonl"

generator = MTRAGGenerationV57()
generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=None)

print("\n🏆 v5.7 COMPLETE!")
print("\n📊 New Features:")
print("  • User-Satisfaction Judge (Stage 6b)")
print("  • Judges from USER perspective")
print("  • 30% sample rate")
print("  • Regenerates on HIGH/MEDIUM confidence unsatisfied")
print("\n🎯 Expected: 0.61-0.615 harmonic mean (+2-3% from v5.6)")


In [ ]:
# ================================================================
# v5.7-implicit-calibrated — "Less Conservative IDK"
# ================================================================
# Changes from v5.7-implicit:
#   • Calibrated generation prompt (prefer answering)
#   • IDK calibration with keyword coverage retry
#   • All v5.7 pipeline features maintained
# ================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q -c /content/drive/MyDrive/Project/constraints.txt -r /content/drive/MyDrive/Project/requirements.txt
!pip install -q pandas numpy huggingface_hub tqdm requests

import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from huggingface_hub import login
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# CONFIGURATION
# =========================
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if HF_TOKEN:
    login(token=HF_TOKEN)

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

N_PASSAGES = 5

# Pipeline stages
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_MULTISAMPLING = True
ENABLE_SELECTIVE_JUDGE = True
ENABLE_USER_SATISFACTION_JUDGE = True
ENABLE_EXTRACTIVE_POLISH = True
ENABLE_IDK_CALIBRATION = True  # NEW: Coverage-based retry

# Span extraction
MIN_SPANS = 1
MAX_SPANS = 8
SPAN_RETRY_ATTEMPTS = 2

# Generation
NUM_CANDIDATES = 2
GENERATION_TEMPERATURE_1 = 0.0
GENERATION_TEMPERATURE_2 = 0.1

# Word count
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105

# Extractiveness
TARGET_EXTRACTIVENESS = 0.32

# Soft trim
SOFT_TRIM_THRESHOLD = 300

# Technical Judge
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
JUDGE_EXTR_MIN = 0.10

# User-Satisfaction Judge
USER_SAT_JUDGE_SAMPLE_RATE = 0.30
USER_SAT_CONFIDENCE_THRESHOLD = ["HIGH", "MEDIUM"]

# IDK Calibration (NEW)
IDK_COVERAGE_THRESHOLD = 0.40  # If coverage > 40%, retry aggressive
IDK_MIN_SPANS_FOR_RETRY = 3     # Need at least 3 spans

IDK_PHRASE = "I do not have specific information"
MAX_WORDS_CONV = 50

DEBUG_MODE = True
MAX_WORKERS = 5

random.seed(42)


class StatsTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.span_retry_count = 0
        self.judge_trigger_count = 0
        self.judge_regenerate_count = 0
        self.user_sat_judge_count = 0
        self.user_sat_unsatisfied_count = 0
        self.user_sat_regenerate_count = 0
        self.idk_calibration_count = 0  # NEW
        self.idk_calibration_success = 0  # NEW
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def add_prediction(self, is_idk: bool):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_judge_trigger(self, regenerated: bool = False):
        with self.lock:
            self.judge_trigger_count += 1
            if regenerated:
                self.judge_regenerate_count += 1

    def add_user_sat_judge(self, unsatisfied: bool = False, regenerated: bool = False):
        with self.lock:
            self.user_sat_judge_count += 1
            if unsatisfied:
                self.user_sat_unsatisfied_count += 1
            if regenerated:
                self.user_sat_regenerate_count += 1

    def add_idk_calibration(self, success: bool):
        with self.lock:
            self.idk_calibration_count += 1
            if success:
                self.idk_calibration_success += 1

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)
        report = f"\n{'='*70}\n"
        report += "📊 GENERATION STATISTICS (v5.7-implicit-calibrated)\n"
        report += f"{'='*70}\n"

        report += f"\n💰 Token Usage:\n"
        report += f"   Input:  {self.total_input:,}\n"
        report += f"   Output: {self.total_output:,}\n"
        report += f"   Cost:   ${cost:.4f}\n"

        total_preds = self.idk_count + self.full_answer_count
        report += f"\n📝 Final Predictions (Total: {total_preds}):\n"
        idk_pct = (self.idk_count / total_preds * 100) if total_preds > 0 else 0
        ans_pct = (self.full_answer_count / total_preds * 100) if total_preds > 0 else 0
        report += f"   IDK Responses:   {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
        report += f"   Full Answers:    {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"

        if self.span_retry_count > 0:
            report += f"\n🔄 Span Extraction Retries: {self.span_retry_count}\n"

        # NEW: IDK Calibration stats
        if self.idk_calibration_count > 0:
            success_rate = (self.idk_calibration_success / self.idk_calibration_count * 100)
            report += f"\n🎯 IDK Calibration (Coverage-based retry):\n"
            report += f"   Triggered:   {self.idk_calibration_count:4d}\n"
            report += f"   Successful:  {self.idk_calibration_success:4d} ({success_rate:5.1f}%)\n"

        if self.judge_trigger_count > 0:
            judge_rate = (self.judge_trigger_count / total_preds * 100) if total_preds > 0 else 0
            regen_rate = (self.judge_regenerate_count / self.judge_trigger_count * 100) if self.judge_trigger_count > 0 else 0
            report += f"\n👨‍⚖️ Technical Judge (Stage 6a):\n"
            report += f"   Triggered:   {self.judge_trigger_count:4d} ({judge_rate:5.1f}% of tasks)\n"
            report += f"   Regenerated: {self.judge_regenerate_count:4d} ({regen_rate:5.1f}% of judged)\n"

        if self.user_sat_judge_count > 0:
            usr_judge_rate = (self.user_sat_judge_count / total_preds * 100) if total_preds > 0 else 0
            unsat_rate = (self.user_sat_unsatisfied_count / self.user_sat_judge_count * 100) if self.user_sat_judge_count > 0 else 0
            regen_rate = (self.user_sat_regenerate_count / self.user_sat_unsatisfied_count * 100) if self.user_sat_unsatisfied_count > 0 else 0
            report += f"\n👤 User-Satisfaction Judge (Stage 6b):\n"
            report += f"   Evaluated:   {self.user_sat_judge_count:4d} ({usr_judge_rate:5.1f}% of tasks)\n"
            report += f"   Unsatisfied: {self.user_sat_unsatisfied_count:4d} ({unsat_rate:5.1f}% of evaluated)\n"
            report += f"   Regenerated: {self.user_sat_regenerate_count:4d} ({regen_rate:5.1f}% of unsatisfied)\n"

        report += f"\n📖 v5.7-implicit-calibrated Features:\n"
        report += f"   • NO Answerability Prediction (implicit)\n"
        report += f"   • Calibrated generation prompt (less conservative)\n"
        report += f"   • IDK calibration with keyword coverage\n"
        report += f"   • MAX_SPANS: 8\n"
        report += f"   • Target Extractiveness: 32%\n"
        report += f"   • Technical Judge: ENABLED\n"
        report += f"   • User-Satisfaction Judge: ENABLED (30% sample)\n"

        report += f"{'='*70}\n"
        return report


tracker = StatsTracker()


# =========================
# HELPERS
# =========================
def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])


def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s


def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r"(Agent|Assistant):\s*", "", t.strip(), flags=re.IGNORECASE)
    t = re.sub(r"\s{3,}", " ", t)
    t = re.sub(r"\n\s*\n", "\n", t)
    return t.strip()


def soft_trim_to_word_limit_by_sentences(text: str, max_words: int) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    kept = []
    for sentence in sentences:
        candidate = " ".join(kept + [sentence])
        if word_count(candidate) <= max_words:
            kept.append(sentence)
        else:
            break
    result = " ".join(kept).strip()
    if not result and sentences:
        result = " ".join(sentences[:3])
    return result


def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    canonical = IDK_PHRASE.lower()
    if re.sub(r'\W', '', low.strip()) == re.sub(r'\W', '', canonical.strip()):
        return True
    idk_indicators = [
        "i don't know", "i do not know",
        "do not have enough information", "don't have enough information",
        "do not have specific information", "don't have specific information",
        "insufficient information", "not enough information"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) < 24:
        return True
    return False


def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return IDK_PHRASE
    return t


def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))


def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0


def calculate_keyword_coverage(question: str, spans: List[Dict]) -> float:
    """Calculate what % of question keywords appear in spans"""
    q_keywords = set(re.findall(r'\b\w{4,}\b', question.lower()))
    if not q_keywords:
        return 0.0

    span_text = " ".join(s.get("text", "").lower() for s in spans)
    overlap = sum(1 for kw in q_keywords if kw in span_text)

    return overlap / len(q_keywords)


class MTRAGGenerationV57ImplicitCalibrated:
    def __init__(self):
        print("="*70)
        print("v5.7-implicit-calibrated: Less Conservative IDK")
        print("="*70)
        print("DeepSeek via Azure initialized")
        print(f"→ Workers: {MAX_WORKERS}")
        print(f"→ IMPLICIT Answerability (no prediction step)")
        print(f"→ Calibrated prompt (prefer answering)")
        print(f"→ IDK calibration with coverage threshold: {IDK_COVERAGE_THRESHOLD}")
        print(f"→ Technical Judge (Stage 6a): ENABLED")
        print(f"→ User-Satisfaction Judge (Stage 6b): ENABLED (30% sample)")
        print("="*70)

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        print(f"→ Loading {input_file}")
        tasks = []
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))
        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"→ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks: List[Dict], output_file: str):
        print(f"→ Saving {output_file}")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as f:
            for task in tasks:
                f.write(json.dumps(task, ensure_ascii=False) + "\n")
        print(f"→ Saved {len(tasks)} predictions")

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return inp.get("text", "") or "".strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'\bhi\b', r'\bhello\b', r'\bhey\b', r'\bthanks?\b', r'\bthank you\b',
            r'\bok\b', r'\bokay\b', r'\bi see\b', r'\bgot it\b', r'\balright\b'
        ]
        return any(re.match(p, q_low) for p in conversational_patterns)

    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_question:
                break
            if speaker == "user":
                history_lines.append(f"User: {text}")
        return history_lines

    def extract_relevant_spans_with_retry(
        self, question: str, contexts: List[Dict], history_lines: List[str]
    ) -> List[Dict]:
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"  → Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()
            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)
            if spans and len(spans) > 0:
                return spans
        if DEBUG_MODE:
            print("  → Smart fallback (keyword-based)")
        return self.smart_fallback(question, contexts)

    def extract_relevant_spans_cot(
        self, question: str, contexts: List[Dict], history_lines: List[str], attempt: int = 1
    ) -> List[Dict]:
        passages_block = ""
        for i, ctx in enumerate(contexts[:5], 1):
            text = ctx.get("text", "") or "".strip()
            relevance_note = ""
            if i == 1:
                relevance_note = " (MOST RELEVANT)"
            elif i == 2:
                relevance_note = " (HIGHLY RELEVANT)"
            passages_block += f"PASSAGE {i}{relevance_note}:\n{text}\n\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONVERSATION HISTORY:\n{recent}\n\n"

        urgency = "The answer IS in the passages - look carefully!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question. {urgency}

{history_block}QUESTION: {question}

{passages_block}RULES:
1. PRIORITIZE earlier passages (MOST RELEVANT)
2. Extract VERBATIM sentences (exact quotes)
3. COMPLETENESS PRIORITY:
   - Include ALL sentences with proper nouns (names, places, organizations)
   - Include ALL sentences with numbers (dates, quantities, measurements)
   - Include ALL sentences needed for complete answer
4. Better to include 1 extra sentence than miss key entity
5. Max {MAX_SPANS} sentences (use all {MAX_SPANS} if needed for completeness)
6. If unanswerable, return empty extractedSpans

Return JSON: {{"reasoning": "Brief explanation of what facts are being extracted", "extractedSpans": [{{"passageId": 1, "sentence": "exact sentence"}}, {{"passageId": 2, "sentence": "another sentence"}}]}}"""

        try:
            response = self.generate_response(span_prompt, temperature=0.0, max_tokens=800)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extractedSpans", [])
                reasoning = result.get("reasoning", "")

                if DEBUG_MODE and attempt == 1:
                    print(f"  → Extracted {len(spans)} spans")
                    if reasoning:
                        print(f"     Reasoning: {reasoning[:80]}...")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "documentId": f"span:frompassage{span['passageId']}",
                            "source": "cot_extracted",
                            "reference": True
                        })
                    if filtered:
                        return filtered
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ✗ Span extraction failed: {e}")
        return []

    def smart_fallback(self, question: str, contexts: List[Dict]) -> List[Dict]:
        q_keywords = set(re.findall(r'[A-Za-z]{5,}', question))
        q_keywords_lower = {k.lower() for k in q_keywords if len(k) >= 3}
        scored = []
        for idx, ctx in enumerate(contexts[:5]):
            text = ctx.get("text", "").lower()
            overlap = sum(1 for kw in q_keywords_lower if kw in text)
            position_bonus = (5 - idx) * 0.5
            score = overlap + position_bonus
            scored.append((score, ctx))
        scored.sort(reverse=True, key=lambda x: x[0])
        return [ctx for _, ctx in scored[:3]]

    # =========================
    # CALIBRATED GENERATION PROMPT (CHANGED)
    # =========================
    def create_generation_prompt_v57_calibrated(
        self, spans: List[Dict], question: str, history_lines: List[str]
    ) -> str:
        word_suggestion = f"typically {SUGGESTED_TARGET}w (adapt to question complexity)"

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"{i}. {text}\n"
        spans_block += "\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONTEXT:\n{recent}\n\n"

        # CALIBRATED PROMPT - Less conservative
        instruction = f"""Generate a natural answer using the EXTRACTED FACTS.

🎯 DECISION RULES:
1. If ≥80% of question facts are in EXTRACTED FACTS → Write FULL answer
2. If 40-80% of facts present → Write PARTIAL answer + mention what's missing
3. If <40% present → "{IDK_PHRASE}"

⚖️ BIAS: **Prefer answering over refusing** (be helpful, not paranoid)

When in doubt:
• If you can say SOMETHING useful → say it (partial answer is OK)
• ONLY refuse if answering would be pure speculation/hallucination

REQUIREMENTS:
• Length: {word_suggestion}
• Include ALL available facts from EXTRACTED FACTS:
  - EVERY name (person, place, organization)
  - EVERY number (quantity, date, year, percentage)
  - EVERY key term from question
• Use EXACT wording for:
  - Proper nouns (copy verbatim)
  - Numbers (copy exactly as written)
  - Technical terms (copy verbatim)
• Paraphrase only connecting words ("and", "also", "while")
• Target ~{int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap

{history_block}{spans_block}QUESTION: {question}

ANSWER:"""
        return instruction

    def generate_candidates(
        self, spans: List[Dict], question: str, history_lines: List[str]
    ) -> List[str]:
        prompt = self.create_generation_prompt_v57_calibrated(spans, question, history_lines)
        candidates = []

        cand1 = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_1, max_tokens=450)
        cand1 = normalize_idk_partial_safe(clean_model_output(cand1))
        candidates.append(cand1)

        if ENABLE_MULTISAMPLING and NUM_CANDIDATES >= 2:
            cand2 = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_2, max_tokens=450)
            cand2 = normalize_idk_partial_safe(clean_model_output(cand2))
            candidates.append(cand2)

        return candidates

    def score_answer_quality(self, answer: str) -> float:
        if is_pure_idk(answer):
            return 0.0

        score = 0.0
        wc = word_count(answer)
        target = SUGGESTED_TARGET
        min_acceptable = 60
        max_acceptable = 130

        if min_acceptable <= wc <= max_acceptable:
            distance = abs(wc - target)
            score += max(0, 1.0 - (distance / 40.0)) * 0.3
        elif wc < min_acceptable:
            score += (wc / min_acceptable) * 0.15
        else:
            score += max(0, 1.0 - ((wc - max_acceptable) / 60.0)) * 0.15

        entities = len(re.findall(r'[A-Z][a-z]+', answer))
        score += min(entities / 10.0, 0.3)

        numbers = len(re.findall(r'\d+', answer))
        score += min(numbers / 5.0, 0.2)

        sentences = re.split(r'[.!?]', answer)
        complete = [s for s in sentences if len(s.strip()) > 10]
        score += min(len(complete) / 5.0, 0.2)

        return score

    def select_best_candidate(self, candidates: List[str]) -> str:
        if len(candidates) == 1:
            return candidates[0]

        scored = []
        for cand in candidates:
            score = self.score_answer_quality(cand)
            scored.append((score, cand))

        scored.sort(reverse=True, key=lambda x: x[0])
        return scored[0][1]

    # =========================
    # IDK CALIBRATION (NEW)
    # =========================
    def calibrate_idk_response(
        self, answer: str, question: str, spans: List[Dict], history_lines: List[str]
    ) -> str:
        """If model says IDK but we have good coverage, retry aggressively"""

        if not ENABLE_IDK_CALIBRATION:
            return answer

        if answer != IDK_PHRASE:
            return answer

        if len(spans) < IDK_MIN_SPANS_FOR_RETRY:
            return answer

        # Calculate keyword coverage
        coverage = calculate_keyword_coverage(question, spans)

        if coverage < IDK_COVERAGE_THRESHOLD:
            return answer  # Coverage too low, IDK is probably correct

        # We have good coverage - retry aggressively
        if DEBUG_MODE:
            print(f"  🎯 IDK calibration: coverage={coverage:.2f} > threshold={IDK_COVERAGE_THRESHOLD}")

        retry_prompt = f"""The EXTRACTED FACTS contain useful information relevant to the question.

⚠️  DO NOT refuse to answer - provide what you CAN say based on the facts.

Even PARTIAL answers are valuable to the user. ONLY output "{IDK_PHRASE}" if
answering would be complete fabrication with NO factual basis.

{self.create_generation_prompt_v57_calibrated(spans, question, history_lines)}"""

        retried = self.generate_response(retry_prompt, temperature=0.15, max_tokens=450)
        retried = normalize_idk_partial_safe(clean_model_output(retried))

        if retried and retried != IDK_PHRASE and word_count(retried) >= 30:
            if DEBUG_MODE:
                print(f"     ✓ Calibration success: {word_count(retried)}w")
            tracker.add_idk_calibration(success=True)
            return retried
        else:
            tracker.add_idk_calibration(success=False)
            return answer

    def should_use_judge(self, answer: str, spans: List[Dict]) -> Tuple[bool, str]:
        wc = word_count(answer)

        if wc < JUDGE_WC_MIN:
            return True, "too_short"
        if wc > JUDGE_WC_MAX:
            return True, "too_long"

        if not is_pure_idk(answer) and wc > 30:
            extr = calculate_extractiveness(answer, spans)
            if extr < JUDGE_EXTR_MIN:
                return True, "low_extractiveness"

        return False, ""

    def judge_answer_lightweight(
        self, answer: str, question: str, spans: List[Dict], trigger_reason: str
    ) -> Dict:
        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"{i}. {span.get('text', '')}\n"

        if "short" in trigger_reason:
            focus = "Are all key facts included? Is it complete?"
        elif "long" in trigger_reason:
            focus = "Is this concise? Any redundancy?"
        elif "extractiveness" in trigger_reason:
            focus = "Is this grounded in EXTRACTED FACTS (not hallucinating)?"
        else:
            focus = "Is this a good quality answer?"

        judge_prompt = f"""Quick quality check.

QUESTION: {question}

{spans_block}
ANSWER: {answer}

FOCUS: {focus}

Return JSON: {{"verdict": "ACCEPT|REGENERATE", "issue": "What's wrong (if REGENERATE)", "suggestion": "How to fix (if REGENERATE)"}}

Rules:
- ACCEPT if answer is reasonable
- REGENERATE only if clear problem"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "verdict": result.get("verdict", "ACCEPT").upper(),
                    "issue": result.get("issue", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except:
            pass

        return {"verdict": "ACCEPT", "issue": "", "suggestion": ""}

    def user_satisfaction_judge(
        self, question: str, answer: str, contexts: List[Dict], history_lines: List[str]
    ) -> Dict:
        passages_preview = ""
        for i, ctx in enumerate(contexts[:5]):
            text = ctx.get("text", "")[:200]
            passages_preview += f"{i+1}. {text}...\n"

        history_block = ""
        if history_lines:
            history_block = "CONVERSATION:\n" + "\n".join(history_lines[-3:]).strip() + "\n\n"

        judge_prompt = f"""You are a USER who asked a question. Judge if the answer satisfies YOUR question.

{history_block}YOUR QUESTION: {question}

AVAILABLE INFORMATION (what the system had access to):
{passages_preview}

SYSTEM'S ANSWER: {answer}

TASK: As a user, would you be SATISFIED with this answer?

Consider:
1. Intent Match: Does the answer address what I was REALLY asking?
2. Completeness for MY Needs
3. Clarity
4. Relevance
5. Context Awareness

IMPORTANT:
- Ignore minor wording differences
- Ignore if answer is slightly shorter/longer than ideal
- Focus on: "Did I get what I was looking for?"

Return JSON: {{"satisfaction": "SATISFIED|UNSATISFIED", "confidence": "HIGH|MEDIUM|LOW", "reason": "Brief explanation", "missing_aspect": "What's missing if unsatisfied", "suggestion": "How to improve if unsatisfied"}}"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=300)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "satisfaction": result.get("satisfaction", "SATISFIED").upper(),
                    "confidence": result.get("confidence", "MEDIUM").upper(),
                    "reason": result.get("reason", ""),
                    "missing_aspect": result.get("missing_aspect", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ✗ User-satisfaction judge failed: {e}")

        return {"satisfaction": "SATISFIED", "confidence": "LOW", "reason": "Judge failed"}

    def build_extractive_polish_prompt(self, spans: List[Dict], raw_answer: str) -> str:
        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"{i}. {span.get('text', '')}\n"

        polish_prompt = f"""{spans_block}
CURRENT ANSWER: {raw_answer}

TASK: Revise to achieve {int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap with EXTRACTED FACTS.

RULES:
• Keep EXACT: names, numbers, dates, locations (4-6 word spans)
• Paraphrase: connecting words, transitions
• Target: {int(TARGET_EXTRACTIVENESS*100)}% of answer = exact phrases from FACTS
• Maintain natural flow
• Keep same length (~{word_count(raw_answer)}w)

REVISED ANSWER:"""
        return polish_prompt

    def soft_normalize(self, text: str) -> str:
        text = re.sub(r'\n', ' ', text.strip())
        text = re.sub(r'\s{3,}', ' ', text)

        if is_pure_idk(text):
            return IDK_PHRASE

        wc = word_count(text)
        if wc > SOFT_TRIM_THRESHOLD:
            if DEBUG_MODE:
                print(f"  → Very long ({wc}w) - soft trimming to 250w")
            text = soft_trim_to_word_limit_by_sentences(text, 250)

        return text

    def create_conversational_prompt(self, history_lines: List[str], question: str) -> str:
        instruction = f"Reply naturally and briefly. Keep it under {MAX_WORDS_CONV} words."
        history_block = ""
        if history_lines:
            history_block = "\n".join(history_lines[-4:]).strip() + "\n\n"
        return f"{instruction}\n\n{history_block}User: {question}\n\nAssistant:"

    def process_task(self, task: Dict) -> Dict:
        inputs = task.get("input", []) or []
        q = self.get_current_question(task)
        history_lines = self.build_history_lines(inputs, q)
        contexts_original = task.get("contexts", [])[:N_PASSAGES]

        debug_steps = {} if DEBUG_MODE else {}

        if self.is_conversational(q):
            prompt = self.create_conversational_prompt(history_lines, q)
            raw = self.generate_response(prompt, temperature=0.3, max_tokens=160)
            raw = clean_model_output(raw)
            if word_count(raw) > MAX_WORDS_CONV:
                raw = soft_trim_to_word_limit_by_sentences(raw, MAX_WORDS_CONV)

            task["contexts"] = contexts_original
            task["predictions"] = [{"text": raw}]
            task["debug_info"] = {"type": "CONVERSATIONAL"}
            tracker.add_prediction(False)
            return task

        # Span extraction
        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(q, contexts_original, history_lines)
        else:
            spans = contexts_original[:3]

        if DEBUG_MODE:
            debug_steps["spans_extracted"] = len(spans)

        if not spans:
            if DEBUG_MODE:
                print("  ⚠ No spans extracted, using fallback")
            spans = contexts_original[:3]

        # Generate candidates
        candidates = self.generate_candidates(spans, q, history_lines)
        final_answer = self.select_best_candidate(candidates)

        # IDK CALIBRATION (NEW)
        final_answer = self.calibrate_idk_response(final_answer, q, spans, history_lines)

        # Technical Judge
        if ENABLE_SELECTIVE_JUDGE and final_answer != IDK_PHRASE:
            should_judge, trigger_reason = self.should_use_judge(final_answer, spans)

            if should_judge:
                if DEBUG_MODE:
                    print(f"  → Technical judge triggered: {trigger_reason}")
                tracker.add_judge_trigger(regenerated=False)

                judgment = self.judge_answer_lightweight(final_answer, q, spans, trigger_reason)

                if judgment["verdict"] == "REGENERATE":
                    if DEBUG_MODE:
                        print(f"  → Regenerating: {judgment['issue'][:60]}...")

                    regen_prompt = f"""Previous answer had issues: {judgment['issue']}
Suggestion: {judgment['suggestion']}

{self.create_generation_prompt_v57_calibrated(spans, q, history_lines)}"""

                    regenerated = self.generate_response(regen_prompt, temperature=0.15, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))

                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_judge_trigger(regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["tech_judge_regenerated"] = "✓"

        # Extractive Polish
        if ENABLE_EXTRACTIVE_POLISH and final_answer != IDK_PHRASE:
            polish_prompt = self.build_extractive_polish_prompt(spans, final_answer)
            polished = self.generate_response(polish_prompt, temperature=0.0, max_tokens=400)
            polished = normalize_idk_partial_safe(clean_model_output(polished))

            if polished and polished != IDK_PHRASE:
                final_answer = polished
                if DEBUG_MODE:
                    debug_steps["polished"] = "✓"

        # User-Satisfaction Judge
        if ENABLE_USER_SATISFACTION_JUDGE and final_answer != IDK_PHRASE:
            if random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                sat_judgment = self.user_satisfaction_judge(q, final_answer, contexts_original, history_lines)
                unsatisfied = (sat_judgment["satisfaction"] == "UNSATISFIED")
                tracker.add_user_sat_judge(unsatisfied=unsatisfied, regenerated=False)

                if DEBUG_MODE:
                    print(f"  → User satisfaction: {sat_judgment['satisfaction']} ({sat_judgment['confidence']})")
                    if unsatisfied:
                        print(f"     Reason: {sat_judgment['reason'][:80]}...")

                if unsatisfied and sat_judgment["confidence"] in USER_SAT_CONFIDENCE_THRESHOLD:
                    if DEBUG_MODE:
                        print(f"  → Regenerating for user satisfaction...")

                    regen_prompt = f"""Previous answer didn't satisfy the user's INTENT.

USER'S QUESTION: {q}
PREVIOUS ANSWER: {final_answer}
ISSUE: {sat_judgment['reason']}
MISSING: {sat_judgment['missing_aspect']}
HOW TO FIX: {sat_judgment['suggestion']}

{self.create_generation_prompt_v57_calibrated(spans, q, history_lines)}

Focus on answering what the user REALLY wants to know.

ANSWER:"""

                    regenerated = self.generate_response(regen_prompt, temperature=0.2, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))

                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_user_sat_judge(unsatisfied=False, regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["user_sat_regenerated"] = "✓"

        # Soft normalization
        final_answer = self.soft_normalize(final_answer)

        if DEBUG_MODE:
            debug_steps["final_wc"] = word_count(final_answer)

        task["contexts"] = contexts_original
        task["contexts_used"] = [s.get("text", "")[:100] for s in spans]
        task["predictions"] = [{"text": final_answer}]

        is_idk = (final_answer == IDK_PHRASE)
        task["debug_info"] = {
            "type": "IDK" if is_idk else "FULL_ANSWER",
            "answer_preview": final_answer[:80],
            "steps": debug_steps
        }
        tracker.add_prediction(is_idk)
        return task

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None):
        print("="*70)
        print("v5.7-implicit-calibrated: Less Conservative IDK")
        print("="*70)
        tasks = self.load_tasks(input_file, max_tasks)
        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}
            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="v5.7-calibrated"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task
                    wc = word_count(out_task["predictions"][0]["text"])
                    is_idk = out_task["predictions"][0]["text"] == IDK_PHRASE
                    debug = out_task.get("debug_info", {})
                    ans_type = debug.get("type", "?")
                    tqdm.write(f"#{idx+1}: {wc}w [{ans_type}]")
                except Exception as e:
                    tqdm.write(f"✗ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)
        print("\nv5.7-implicit-calibrated COMPLETE!")
        return output_file

    def generate_response_with_system(
        self, system: str, user: str, temperature: float = 0.0, max_tokens: int = 360
    ) -> str:
        headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                time.sleep(5)
        return ""

    def generate_response(self, prompt: str, temperature: float = 0.0, max_tokens: int = 360) -> str:
        return self.generate_response_with_system("You are a helpful assistant.", prompt, temperature, max_tokens)


# =========================
# MAIN EXECUTION
# =========================
INPUT_FILE = "/content/drive/MyDrive/Project/test_set_final_ensemble/test_set_TOP10_RAG_taskC.jsonl"
PREDICTIONS_OUTPUT = "/content/drive/MyDrive/Project/outputs/deepseek_RAG_v57_implicit_calibrated_test.jsonl"

generator = MTRAGGenerationV57ImplicitCalibrated()
generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=None)

print("\n" + "="*70)
print("✅ v5.7-implicit-calibrated COMPLETE!")
print("="*70)
print("\n🔑 Key Changes:")
print("• Calibrated generation prompt: 'Prefer answering over refusing'")
print("• IDK calibration: Coverage-based retry (threshold: 40%)")
print("• Removed answerability prediction (implicit)")
print("• All v5.7 pipeline features maintained")
print("\n📈 Expected Results:")
print("• RL_F: ~0.70-0.73 (down from 0.90 - less over-conservative)")
print("• RB_llm: ~0.66-0.68 (up from 0.57 - better completeness)")
print("• RB_alg: ~0.42-0.44 (up from 0.38)")
print("• Harmonic Mean: ~0.58-0.60 (target: paper-level)")
print("="*70)


In [ ]:
# ================================================================
# v5.8-smart-comparison (Natural Language - No Meta-Phrases)
# ================================================================
# ONLY CHANGE: Avoid "based on the extracted facts" and similar
# ================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q -c /content/drive/MyDrive/Project/constraints.txt -r /content/drive/MyDrive/Project/requirements.txt
!pip install -q pandas numpy huggingface_hub tqdm requests

import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from huggingface_hub import login
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# HF LOGIN
# =========================
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if HF_TOKEN:
    login(token=HF_TOKEN)

# =========================
# AZURE DEEPSEEK CONFIGURATION
# =========================
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# =========================
# CONFIGURATION
# =========================
N_PASSAGES = 5

# Pipeline stages
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_MULTISAMPLING = False  # CHANGED: Disabled (using pre-post polish comparison)
ENABLE_SELECTIVE_JUDGE = True
ENABLE_USER_SATISFACTION_JUDGE = True
ENABLE_EXTRACTIVE_POLISH = True
ENABLE_IDK_CALIBRATION = True
ENABLE_PRE_POST_COMPARISON = True  # NEW: Compare pre-polish vs post-polish

# Span extraction
MIN_SPANS = 1
MAX_SPANS = 8
SPAN_RETRY_ATTEMPTS = 2

# Generation
NUM_CANDIDATES = 2
GENERATION_TEMPERATURE_1 = 0.0
GENERATION_TEMPERATURE_2 = 0.1

# Word count
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105

# Extractiveness
TARGET_EXTRACTIVENESS = 0.35  # Sweet spot for balance (CHANGED: 32% → 35%)

# Soft trim
SOFT_TRIM_THRESHOLD = 300

# Technical Judge (CHANGED: Stricter extractiveness)
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
JUDGE_EXTR_MIN = 0.15  # Was 0.10, now stricter

# User-Satisfaction Judge (CHANGED: 30% → 50%)
USER_SAT_JUDGE_SAMPLE_RATE = 0.50  # Balanced (not 100%, not too low)
USER_SAT_CONFIDENCE_THRESHOLD = ["HIGH", "MEDIUM"]

# IDK Calibration (CHANGED: Adaptive thresholds)
IDK_COVERAGE_THRESHOLD_FACTOID = 0.65    # Strict (need high coverage)
IDK_COVERAGE_THRESHOLD_EXPLANATION = 0.35  # Lenient (partial OK)
IDK_COVERAGE_THRESHOLD_GENERAL = 0.45      # Default
IDK_MIN_SPANS_FOR_RETRY = 3

IDK_PHRASE = "I do not have specific information"
MAX_WORDS_CONV = 50

DEBUG_MODE = True
MAX_WORKERS = 5

# Set random seed for reproducibility
random.seed(42)

# =========================
# STATS TRACKER
# =========================
class StatsTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.span_retry_count = 0
        self.judge_trigger_count = 0
        self.judge_regenerate_count = 0
        self.user_sat_judge_count = 0
        self.user_sat_unsatisfied_count = 0
        self.user_sat_regenerate_count = 0
        self.idk_calibration_count = 0
        self.idk_calibration_success = 0
        self.pre_post_comparison_count = 0
        self.pre_post_chose_pre_polish = 0
        self.pre_post_chose_post_polish = 0
        self.lock = threading.Lock()

    def add_usage(self, prompt, completion):
        with self.lock:
            self.total_input += prompt
            self.total_output += completion

    def add_prediction(self, is_idk: bool):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_judge_trigger(self, regenerated: bool = False):
        with self.lock:
            self.judge_trigger_count += 1
            if regenerated:
                self.judge_regenerate_count += 1

    def add_user_sat_judge(self, unsatisfied: bool = False, regenerated: bool = False):
        with self.lock:
            self.user_sat_judge_count += 1
            if unsatisfied:
                self.user_sat_unsatisfied_count += 1
            if regenerated:
                self.user_sat_regenerate_count += 1

    def add_idk_calibration(self, success: bool):
        with self.lock:
            self.idk_calibration_count += 1
            if success:
                self.idk_calibration_success += 1

    def add_pre_post_comparison(self, chose_pre_polish: bool):
        with self.lock:
            self.pre_post_comparison_count += 1
            if chose_pre_polish:
                self.pre_post_chose_pre_polish += 1
            else:
                self.pre_post_chose_post_polish += 1

    def get_report(self):
        cost = (self.total_input * 0.20 / 1e6) + (self.total_output * 0.50 / 1e6)

        report = f"\n{'='*70}\n"
        report += "📊 GENERATION STATISTICS (v5.8-natural-language)\n"
        report += f"{'='*70}\n"

        report += f"\n💰 Token Usage:\n"
        report += f"   Input:  {self.total_input:,}\n"
        report += f"   Output: {self.total_output:,}\n"
        report += f"   Cost:   ${cost:.4f}\n"

        total_preds = self.idk_count + self.full_answer_count
        report += f"\n📝 Final Predictions (Total: {total_preds}):\n"
        idk_pct = (self.idk_count / total_preds * 100) if total_preds > 0 else 0
        ans_pct = (self.full_answer_count / total_preds * 100) if total_preds > 0 else 0
        report += f"   IDK Responses:   {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
        report += f"   Full Answers:    {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"

        if self.span_retry_count > 0:
            report += f"\n🔄 Span Extraction Retries: {self.span_retry_count}\n"

        if self.idk_calibration_count > 0:
            success_rate = (self.idk_calibration_success / self.idk_calibration_count * 100)
            report += f"\n🎯 IDK Calibration (Adaptive threshold):\n"
            report += f"   Triggered:   {self.idk_calibration_count:4d}\n"
            report += f"   Successful:  {self.idk_calibration_success:4d} ({success_rate:5.1f}%)\n"

        if self.pre_post_comparison_count > 0:
            pre_pct = (self.pre_post_chose_pre_polish / self.pre_post_comparison_count * 100)
            post_pct = (self.pre_post_chose_post_polish / self.pre_post_comparison_count * 100)
            report += f"\n🔄 Pre-Post Polish Comparison:\n"
            report += f"   Total Comparisons: {self.pre_post_comparison_count:4d}\n"
            report += f"   Chose Pre-Polish:  {self.pre_post_chose_pre_polish:4d} ({pre_pct:5.1f}%)\n"
            report += f"   Chose Post-Polish: {self.pre_post_chose_post_polish:4d} ({post_pct:5.1f}%)\n"

        if self.judge_trigger_count > 0:
            judge_rate = (self.judge_trigger_count / total_preds * 100) if total_preds > 0 else 0
            regen_rate = (self.judge_regenerate_count / self.judge_trigger_count * 100) if self.judge_trigger_count > 0 else 0
            report += f"\n👨‍⚖️ Technical Judge:\n"
            report += f"   Triggered:   {self.judge_trigger_count:4d} ({judge_rate:5.1f}% of tasks)\n"
            report += f"   Regenerated: {self.judge_regenerate_count:4d} ({regen_rate:5.1f}% of judged)\n"

        if self.user_sat_judge_count > 0:
            usr_judge_rate = (self.user_sat_judge_count / total_preds * 100) if total_preds > 0 else 0
            unsat_rate = (self.user_sat_unsatisfied_count / self.user_sat_judge_count * 100) if self.user_sat_judge_count > 0 else 0
            regen_rate = (self.user_sat_regenerate_count / self.user_sat_unsatisfied_count * 100) if self.user_sat_unsatisfied_count > 0 else 0
            report += f"\n👤 User-Satisfaction Judge (50% sample):\n"
            report += f"   Evaluated:   {self.user_sat_judge_count:4d} ({usr_judge_rate:5.1f}% of tasks)\n"
            report += f"   Unsatisfied: {self.user_sat_unsatisfied_count:4d} ({unsat_rate:5.1f}% of evaluated)\n"
            report += f"   Regenerated: {self.user_sat_regenerate_count:4d} ({regen_rate:5.1f}% of unsatisfied)\n"

        report += f"\n📖 v5.8-natural-language Features:\n"
        report += f"   🔥 NEW: Avoid meta-phrases (no 'based on the extracted facts')\n"
        report += f"   • Pre-Post Polish Comparison (choose better naturalness)\n"
        report += f"   • Adaptive IDK thresholds (35-65% by question type)\n"
        report += f"   • Target Extractiveness: 35%\n"
        report += f"   • Stricter Tech Judge: 15% extractiveness\n"
        report += f"   • User-Satisfaction Judge: 50%\n"

        report += f"{'='*70}\n"
        return report


tracker = StatsTracker()

# =========================
# HELPERS (unchanged from v5.8)
# =========================
def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])

def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s

def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r"^(Agent|Assistant):\s*", "", t.strip(), flags=re.IGNORECASE)
    t = re.sub(r"\s{3,}", " ", t)
    t = re.sub(r"\n\s*\n", "\n", t)
    return t.strip()

def soft_trim_to_word_limit_by_sentences(text: str, max_words: int) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    kept = []
    for sentence in sentences:
        candidate = " ".join(kept + [sentence])
        if word_count(candidate) <= max_words:
            kept.append(sentence)
        else:
            break
    result = " ".join(kept).strip()
    if not result and sentences:
        result = " ".join(sentences[:3])
    return result

def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    canonical = IDK_PHRASE.lower()
    if re.sub(r'\W', '', low.strip()) == re.sub(r'\W', '', canonical.strip()):
        return True
    idk_indicators = [
        "i don't know", "i do not know",
        "do not have enough information", "don't have enough information",
        "do not have specific information", "don't have specific information",
        "insufficient information", "not enough information"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) < 24:
        return True
    return False

def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return IDK_PHRASE
    return t

def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))

def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0

def calculate_keyword_coverage(question: str, spans: List[Dict]) -> float:
    q_keywords = set(re.findall(r'\b\w{4,}\b', question.lower()))
    if not q_keywords:
        return 0.0
    span_text = " ".join(s.get("text", "").lower() for s in spans)
    overlap = sum(1 for kw in q_keywords if kw in span_text)
    return overlap / len(q_keywords)

def detect_question_type(question: str) -> str:
    q_lower = question.lower().strip()
    factoid_patterns = [
        r'\bwho\b', r'\bwhat\b', r'\bwhen\b', r'\bwhere\b',
        r'\bhow many\b', r'\bhow much\b', r'\byear\b', r'\bdate\b', r'\btime\b'
    ]
    if any(re.search(p, q_lower) for p in factoid_patterns):
        return "factoid"
    explanation_patterns = [
        r'\bwhy\b', r'\bhow does\b', r'\bhow did\b', r'\bexplain\b',
        r'\bcauses\b', r'\bis the reason\b'
    ]
    if any(re.search(p, q_lower) for p in explanation_patterns):
        return "explanation"
    return "general"

def get_adaptive_idk_threshold(question: str) -> float:
    qtype = detect_question_type(question)
    thresholds = {
        "factoid": IDK_COVERAGE_THRESHOLD_FACTOID,
        "explanation": IDK_COVERAGE_THRESHOLD_EXPLANATION,
        "general": IDK_COVERAGE_THRESHOLD_GENERAL
    }
    return thresholds.get(qtype, IDK_COVERAGE_THRESHOLD_GENERAL)

def get_adaptive_generation_threshold(question: str) -> Tuple[float, float]:
    qtype = detect_question_type(question)
    thresholds = {
        "factoid": (0.85, 0.60),
        "explanation": (0.70, 0.40),
        "general": (0.80, 0.50)
    }
    return thresholds.get(qtype, (0.80, 0.50))

# =========================
# MAIN CLASS
# =========================
class MTRAGGenerationV58NaturalLanguage:
    def __init__(self):
        print("="*70)
        print("v5.8-natural-language (No Meta-Phrases)")
        print("="*70)
        print("✅ DeepSeek via Azure initialized")
        print(f"→ Workers: {MAX_WORKERS}")
        print(f"🔥 NEW: Avoid 'based on the extracted facts' and similar phrases")
        print(f"→ Pre-Post Polish Comparison: ENABLED")
        print(f"→ Adaptive IDK thresholds: 35-65%")
        print(f"→ Target Extractiveness: 35%")
        print(f"→ User-Satisfaction Judge: 50%")
        print("="*70)

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        print(f"→ Loading {input_file}")
        tasks = []
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))
        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"→ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks: List[Dict], output_file: str):
        print(f"→ Saving {output_file}")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as f:
            for task in tasks:
                f.write(json.dumps(task, ensure_ascii=False) + "\n")
        print(f"→ Saved {len(tasks)} predictions")

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return inp.get("text", "") or "".strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'\bhi\b', r'\bhello\b', r'\bhey\b', r'\bthanks?\b', r'\bthank you\b',
            r'\bok\b', r'\bokay\b', r'\bi see\b', r'\bgot it\b', r'\balright\b'
        ]
        return any(re.match(p, q_low) for p in conversational_patterns)

    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_question:
                break
            if speaker == "user":
                history_lines.append(f"User: {text}")
        return history_lines

    def extract_relevant_spans_with_retry(
        self, question: str, contexts: List[Dict], history_lines: List[str]
    ) -> List[Dict]:
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"  → Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()
            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)
            if spans and len(spans) > 0:
                return spans
        if DEBUG_MODE:
            print("  → Smart fallback (keyword-based)")
        return self.smart_fallback(question, contexts)

    def extract_relevant_spans_cot(
        self, question: str, contexts: List[Dict], history_lines: List[str], attempt: int = 1
    ) -> List[Dict]:
        passages_block = ""
        for i, ctx in enumerate(contexts[:5], 1):
            text = (ctx.get("text", "") or "").strip()
            relevance_note = ""
            if i == 1:
                relevance_note = " (MOST RELEVANT)"
            elif i == 2:
                relevance_note = " (HIGHLY RELEVANT)"
            passages_block += f"PASSAGE {i}{relevance_note}:\n{text}\n\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONVERSATION HISTORY:\n{recent}\n\n"

        urgency = "The answer IS in the passages - look carefully!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question. {urgency}

{history_block}QUESTION: {question}

{passages_block}
RULES:
1. PRIORITIZE earlier passages (MOST RELEVANT)
2. Extract VERBATIM sentences (exact quotes)
3. COMPLETENESS PRIORITY:
   - Include ALL sentences with proper nouns (names, places, organizations)
   - Include ALL sentences with numbers (dates, quantities, measurements)
   - Include ALL sentences needed for complete answer
4. Better to include 1 extra sentence than miss key entity
5. Max {MAX_SPANS} sentences (use all {MAX_SPANS} if needed for completeness)
6. If unanswerable, return empty extractedSpans

Return JSON: {{"reasoning": "Brief explanation of what facts are being extracted", "extractedSpans": [{{"passageId": 1, "sentence": "exact sentence"}}, {{"passageId": 2, "sentence": "another sentence"}}]}}
"""

        try:
            response = self.generate_response(span_prompt, temperature=0.0, max_tokens=800)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extractedSpans", [])
                reasoning = result.get("reasoning", "")

                if DEBUG_MODE and attempt == 1:
                    print(f"  → Extracted {len(spans)} spans")
                    if reasoning:
                        print(f"     Reasoning: {reasoning[:80]}...")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "documentId": f"span:frompassage{span['passageId']}",
                            "source": "cot_extracted",
                            "reference": True
                        })
                    if filtered:
                        return filtered
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ✗ Span extraction failed: {e}")
        return []

    def smart_fallback(self, question: str, contexts: List[Dict]) -> List[Dict]:
        q_keywords = set(re.findall(r'[A-Za-z]{5,}', question))
        q_keywords_lower = {k.lower() for k in q_keywords if len(k) >= 3}
        scored = []
        for idx, ctx in enumerate(contexts[:5]):
            text = ctx.get("text", "").lower()
            overlap = sum(1 for kw in q_keywords_lower if kw in text)
            position_bonus = (5 - idx) * 0.5
            score = overlap + position_bonus
            scored.append((score, ctx))
        scored.sort(reverse=True, key=lambda x: x[0])
        return [ctx for _, ctx in scored[:3]]

    # 🔥 ONLY CHANGE: Modified generation prompt
    def create_generation_prompt_adaptive(
        self, spans: List[Dict], question: str, history_lines: List[str]
    ) -> str:
        word_suggestion = f"typically {SUGGESTED_TARGET}w (adapt to question complexity)"

        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"{i}. {text}\n"
        spans_block += "\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-3:])
            history_block = f"CONTEXT:\n{recent}\n\n"

        qtype = detect_question_type(question)
        full_thresh, partial_thresh = get_adaptive_generation_threshold(question)

        qtype_guidance = {
            "factoid": "This is a FACTOID question - be precise with facts",
            "explanation": "This is an EXPLANATION question - partial answers are helpful",
            "general": "Standard question - be complete but concise"
        }.get(qtype, "")

        # 🔥 NEW: Explicit instruction to avoid meta-phrases
        instruction = f"""Generate a natural answer using the EXTRACTED FACTS.

{qtype_guidance}

🎯 DECISION RULES (Adaptive for {qtype} question):
1. If ≥{int(full_thresh*100)}% of question facts in EXTRACTED FACTS → Write FULL answer
2. If {int(partial_thresh*100)}-{int(full_thresh*100)}% present → Write PARTIAL answer + mention what's missing
3. If <{int(partial_thresh*100)}% present → "{IDK_PHRASE}"

⚖️ BIAS: **Prefer answering over refusing** (be helpful, not paranoid)

When in doubt:
• If you can say SOMETHING useful → say it (partial answer is OK)
• ONLY refuse if answering would be pure speculation/hallucination

🚫 CRITICAL: DO NOT USE META-PHRASES
• ❌ FORBIDDEN: "based on the extracted facts", "according to the information",
  "the documents state", "the passages mention", "from the provided context",
  "the information shows", "as stated in", "the text indicates"
• ✅ CORRECT: Write DIRECTLY as if you know the facts naturally
• Examples:
  - ❌ BAD: "Based on the extracted facts, the capital is Paris."
  - ✅ GOOD: "The capital is Paris."
  - ❌ BAD: "According to the information, he was born in 1990."
  - ✅ GOOD: "He was born in 1990."

REQUIREMENTS:
• Length: {word_suggestion}
• Include ALL available facts from EXTRACTED FACTS:
  - EVERY name (person, place, organization)
  - EVERY number (quantity, date, year, percentage)
  - EVERY key term from question
• Use EXACT wording for:
  - Proper nouns (copy verbatim)
  - Numbers (copy exactly as written)
  - Technical terms (copy verbatim)
• Paraphrase only: connecting words ("and", "also", "while")
• Target ~{int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap
• Write naturally as if stating facts directly (NO meta-commentary about sources)

{history_block}{spans_block}QUESTION: {question}

ANSWER:"""

        return instruction

    # Rest of methods unchanged from v5.8
    def generate_single_candidate(
        self, spans: List[Dict], question: str, history_lines: List[str]
    ) -> str:
        prompt = self.create_generation_prompt_adaptive(spans, question, history_lines)
        cand = self.generate_response(prompt, temperature=GENERATION_TEMPERATURE_1, max_tokens=450)
        return normalize_idk_partial_safe(clean_model_output(cand))

    def calibrate_idk_response(
        self, answer: str, question: str, spans: List[Dict], history_lines: List[str]
    ) -> str:
        if not ENABLE_IDK_CALIBRATION:
            return answer
        if answer != IDK_PHRASE:
            return answer
        if len(spans) < IDK_MIN_SPANS_FOR_RETRY:
            return answer
        coverage = calculate_keyword_coverage(question, spans)
        threshold = get_adaptive_idk_threshold(question)
        if coverage < threshold:
            return answer
        if DEBUG_MODE:
            qtype = detect_question_type(question)
            print(f"  🎯 IDK calibration ({qtype}): coverage={coverage:.2f} > threshold={threshold:.2f}")
        retry_prompt = f"""The EXTRACTED FACTS contain useful information relevant to the question.

⚠️  DO NOT refuse to answer - provide what you CAN say based on the facts.

Even PARTIAL answers are valuable to the user. ONLY output "{IDK_PHRASE}" if
answering would be complete fabrication with NO factual basis.

{self.create_generation_prompt_adaptive(spans, question, history_lines)}"""

        retried = self.generate_response(retry_prompt, temperature=0.15, max_tokens=450)
        retried = normalize_idk_partial_safe(clean_model_output(retried))
        if retried and retried != IDK_PHRASE and word_count(retried) >= 30:
            if DEBUG_MODE:
                print(f"     ✓ Calibration success: {word_count(retried)}w")
            tracker.add_idk_calibration(success=True)
            return retried
        else:
            tracker.add_idk_calibration(success=False)
            return answer

    def should_use_judge(self, answer: str, spans: List[Dict]) -> Tuple[bool, str]:
        wc = word_count(answer)
        if wc < JUDGE_WC_MIN:
            return True, "too_short"
        if wc > JUDGE_WC_MAX:
            return True, "too_long"
        if not is_pure_idk(answer) and wc > 30:
            extr = calculate_extractiveness(answer, spans)
            if extr < JUDGE_EXTR_MIN:
                return True, "low_extractiveness"
        return False, ""

    def judge_answer_lightweight(
        self, answer: str, question: str, spans: List[Dict], trigger_reason: str
    ) -> Dict:
        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"{i}. {span.get('text', '')}\n"

        if "short" in trigger_reason:
            focus = "Are all key facts included? Is it complete?"
        elif "long" in trigger_reason:
            focus = "Is this concise? Any redundancy?"
        elif "extractiveness" in trigger_reason:
            focus = "Is this grounded in EXTRACTED FACTS (not hallucinating)?"
        else:
            focus = "Is this a good quality answer?"

        judge_prompt = f"""Quick quality check.

QUESTION: {question}

{spans_block}
ANSWER: {answer}

FOCUS: {focus}

Return JSON: {{"verdict": "ACCEPT|REGENERATE", "issue": "What's wrong (if REGENERATE)", "suggestion": "How to fix (if REGENERATE)"}}

Rules:
- ACCEPT if answer is reasonable
- REGENERATE only if clear problem"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "verdict": result.get("verdict", "ACCEPT").upper(),
                    "issue": result.get("issue", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except:
            pass
        return {"verdict": "ACCEPT", "issue": "", "suggestion": ""}

    def user_satisfaction_judge(
        self, question: str, answer: str, contexts: List[Dict], history_lines: List[str]
    ) -> Dict:
        passages_preview = ""
        for i, ctx in enumerate(contexts[:5]):
            text = ctx.get("text", "")[:200]
            passages_preview += f"{i+1}. {text}...\n"

        history_block = ""
        if history_lines:
            history_block = "CONVERSATION:\n" + "\n".join(history_lines[-3:]).strip() + "\n\n"

        judge_prompt = f"""You are a USER who asked a question. Judge if the answer satisfies YOUR question.

{history_block}YOUR QUESTION: {question}

AVAILABLE INFORMATION (what the system had access to):
{passages_preview}

SYSTEM'S ANSWER: {answer}

TASK: As a user, would you be SATISFIED with this answer?

Consider:
1. **Intent Match**: Does the answer address what I was REALLY asking?
2. **Completeness for MY Needs**
3. **Clarity**
4. **Relevance**
5. **Context Awareness**

IMPORTANT:
- Ignore minor wording differences
- Ignore if answer is slightly shorter/longer than ideal
- Focus on: "Did I get what I was looking for?"

Return JSON: {{"satisfaction": "SATISFIED|UNSATISFIED", "confidence": "HIGH|MEDIUM|LOW", "reason": "Brief explanation", "missing_aspect": "What's missing if unsatisfied", "suggestion": "How to improve if unsatisfied"}}"""

        try:
            response = self.generate_response(judge_prompt, temperature=0.0, max_tokens=300)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                return {
                    "satisfaction": result.get("satisfaction", "SATISFIED").upper(),
                    "confidence": result.get("confidence", "MEDIUM").upper(),
                    "reason": result.get("reason", ""),
                    "missing_aspect": result.get("missing_aspect", ""),
                    "suggestion": result.get("suggestion", "")
                }
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ✗ User-satisfaction judge failed: {e}")
        return {"satisfaction": "SATISFIED", "confidence": "LOW", "reason": "Judge failed"}

    def build_extractive_polish_prompt(self, spans: List[Dict], raw_answer: str) -> str:
        spans_block = "EXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            spans_block += f"{i}. {span.get('text', '')}\n"

        polish_prompt = f"""{spans_block}
CURRENT ANSWER: {raw_answer}

TASK: Revise to achieve {int(TARGET_EXTRACTIVENESS*100)}% verbatim overlap with EXTRACTED FACTS.

RULES:
• Keep EXACT: names, numbers, dates, locations (4-6 word spans)
• Paraphrase: connecting words, transitions
• Target: {int(TARGET_EXTRACTIVENESS*100)}% of answer = exact phrases from FACTS
• Maintain natural flow
• Keep same length (~{word_count(raw_answer)}w)
• 🔥 DO NOT add meta-phrases like "based on the extracted facts"

REVISED ANSWER:"""
        return polish_prompt

    def compare_pre_post_polish(
        self, pre_polish: str, post_polish: str, question: str, spans: List[Dict]
    ) -> str:
        pre_extr = calculate_extractiveness(pre_polish, spans)
        post_extr = calculate_extractiveness(post_polish, spans)

        comparison_prompt = f"""You are judging which answer is BETTER for the user.

QUESTION: {question}

VERSION A (Natural):
{pre_polish}
Extractiveness: {pre_extr:.2f}

VERSION B (Polished):
{post_polish}
Extractiveness: {post_extr:.2f}

TASK: Which version is better overall?

Consider:
1. Naturalness (A typically more natural, B more verbatim)
2. Completeness (both should have same facts)
3. Clarity
4. Extractiveness (need ~30% but not too robotic)

Return JSON: {{"choice": "A|B", "reason": "Brief explanation"}}

IMPORTANT: Choose B ONLY if A is too vague/paraphrased. If A is clear and natural, choose A."""

        try:
            response = self.generate_response(comparison_prompt, temperature=0.0, max_tokens=150)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                choice = result.get("choice", "B").upper()
                reason = result.get("reason", "")
                if DEBUG_MODE:
                    print(f"  → Pre-Post comparison: chose {choice}")
                    print(f"     Reason: {reason[:60]}...")
                if choice == "A":
                    tracker.add_pre_post_comparison(chose_pre_polish=True)
                    return pre_polish
                else:
                    tracker.add_pre_post_comparison(chose_pre_polish=False)
                    return post_polish
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ✗ Comparison failed: {e}, defaulting to post-polish")
        tracker.add_pre_post_comparison(chose_pre_polish=False)
        return post_polish

    def soft_normalize(self, text: str) -> str:
        text = re.sub(r'\n', ' ', text.strip())
        text = re.sub(r'\s{3,}', ' ', text)
        if is_pure_idk(text):
            return IDK_PHRASE
        wc = word_count(text)
        if wc > SOFT_TRIM_THRESHOLD:
            if DEBUG_MODE:
                print(f"  → Very long ({wc}w) - soft trimming to 250w")
            text = soft_trim_to_word_limit_by_sentences(text, 250)
        return text

    def create_conversational_prompt(self, history_lines: List[str], question: str) -> str:
        instruction = f"Reply naturally and briefly. Keep it under {MAX_WORDS_CONV} words."
        history_block = ""
        if history_lines:
            history_block = "\n".join(history_lines[-4:]).strip() + "\n\n"
        return f"{instruction}\n\n{history_block}User: {question}\n\nAssistant:"

    def process_task(self, task: Dict) -> Dict:
        inputs = task.get("input", []) or []
        q = self.get_current_question(task)
        history_lines = self.build_history_lines(inputs, q)
        contexts_original = task.get("contexts", [])[:N_PASSAGES]
        debug_steps = {} if DEBUG_MODE else {}

        if self.is_conversational(q):
            prompt = self.create_conversational_prompt(history_lines, q)
            raw = self.generate_response(prompt, temperature=0.3, max_tokens=160)
            raw = clean_model_output(raw)
            if word_count(raw) > MAX_WORDS_CONV:
                raw = soft_trim_to_word_limit_by_sentences(raw, MAX_WORDS_CONV)
            task["contexts"] = contexts_original
            task["predictions"] = [{"text": raw}]
            task["debug_info"] = {"type": "CONVERSATIONAL"}
            tracker.add_prediction(False)
            return task

        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(q, contexts_original, history_lines)
        else:
            spans = contexts_original[:3]

        if DEBUG_MODE:
            debug_steps["spans_extracted"] = len(spans)

        if not spans:
            if DEBUG_MODE:
                print("  ⚠ No spans extracted, using fallback")
            spans = contexts_original[:3]

        pre_polish_answer = self.generate_single_candidate(spans, q, history_lines)
        pre_polish_answer = self.calibrate_idk_response(pre_polish_answer, q, spans, history_lines)

        if ENABLE_SELECTIVE_JUDGE and pre_polish_answer != IDK_PHRASE:
            should_judge, trigger_reason = self.should_use_judge(pre_polish_answer, spans)
            if should_judge:
                if DEBUG_MODE:
                    print(f"  → Technical judge triggered: {trigger_reason}")
                tracker.add_judge_trigger(regenerated=False)
                judgment = self.judge_answer_lightweight(pre_polish_answer, q, spans, trigger_reason)
                if judgment["verdict"] == "REGENERATE":
                    if DEBUG_MODE:
                        print(f"  → Regenerating: {judgment['issue'][:60]}...")
                    regen_prompt = f"""Previous answer had issues: {judgment['issue']}
Suggestion: {judgment['suggestion']}

{self.create_generation_prompt_adaptive(spans, q, history_lines)}"""
                    regenerated = self.generate_response(regen_prompt, temperature=0.15, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))
                    if regenerated and regenerated != IDK_PHRASE:
                        pre_polish_answer = regenerated
                        tracker.add_judge_trigger(regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["tech_judge_regenerated"] = "✓"

        final_answer = pre_polish_answer

        if ENABLE_EXTRACTIVE_POLISH and ENABLE_PRE_POST_COMPARISON and pre_polish_answer != IDK_PHRASE:
            polish_prompt = self.build_extractive_polish_prompt(spans, pre_polish_answer)
            post_polish_answer = self.generate_response(polish_prompt, temperature=0.0, max_tokens=400)
            post_polish_answer = normalize_idk_partial_safe(clean_model_output(post_polish_answer))
            if post_polish_answer and post_polish_answer != IDK_PHRASE:
                final_answer = self.compare_pre_post_polish(
                    pre_polish_answer, post_polish_answer, q, spans
                )
                if DEBUG_MODE:
                    debug_steps["pre_post_comparison"] = "✓"

        if ENABLE_USER_SATISFACTION_JUDGE and final_answer != IDK_PHRASE:
            if random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                sat_judgment = self.user_satisfaction_judge(q, final_answer, contexts_original, history_lines)
                unsatisfied = (sat_judgment["satisfaction"] == "UNSATISFIED")
                tracker.add_user_sat_judge(unsatisfied=unsatisfied, regenerated=False)
                if DEBUG_MODE:
                    print(f"  → User satisfaction: {sat_judgment['satisfaction']} ({sat_judgment['confidence']})")
                    if unsatisfied:
                        print(f"     Reason: {sat_judgment['reason'][:80]}...")
                if unsatisfied and sat_judgment["confidence"] in USER_SAT_CONFIDENCE_THRESHOLD:
                    if DEBUG_MODE:
                        print(f"  → Regenerating for user satisfaction...")
                    regen_prompt = f"""Previous answer didn't satisfy the user's INTENT.

USER'S QUESTION: {q}
PREVIOUS ANSWER: {final_answer}
ISSUE: {sat_judgment['reason']}
MISSING: {sat_judgment['missing_aspect']}
HOW TO FIX: {sat_judgment['suggestion']}

{self.create_generation_prompt_adaptive(spans, q, history_lines)}

Focus on answering what the user REALLY wants to know.

ANSWER:"""
                    regenerated = self.generate_response(regen_prompt, temperature=0.2, max_tokens=450)
                    regenerated = normalize_idk_partial_safe(clean_model_output(regenerated))
                    if regenerated and regenerated != IDK_PHRASE:
                        final_answer = regenerated
                        tracker.add_user_sat_judge(unsatisfied=False, regenerated=True)
                        if DEBUG_MODE:
                            debug_steps["user_sat_regenerated"] = "✓"

        final_answer = self.soft_normalize(final_answer)
        if DEBUG_MODE:
            debug_steps["final_wc"] = word_count(final_answer)

        task["contexts"] = contexts_original
        task["contexts_used"] = [s.get("text", "")[:100] for s in spans]
        task["predictions"] = [{"text": final_answer}]
        is_idk = (final_answer == IDK_PHRASE)
        task["debug_info"] = {
            "type": "IDK" if is_idk else "FULL_ANSWER",
            "answer_preview": final_answer[:80],
            "steps": debug_steps
        }
        tracker.add_prediction(is_idk)
        return task

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None):
        print("="*70)
        print("v5.8-natural-language")
        print("="*70)
        tasks = self.load_tasks(input_file, max_tasks)
        processed = [None] * len(tasks)
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}
            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="v5.8-nat"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task
                    wc = word_count(out_task["predictions"][0]["text"])
                    is_idk = out_task["predictions"][0]["text"] == IDK_PHRASE
                    debug = out_task.get("debug_info", {})
                    ans_type = debug.get("type", "?")
                    tqdm.write(f"#{idx+1}: {wc}w [{ans_type}]")
                except Exception as e:
                    tqdm.write(f"✗ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]
        print(tracker.get_report())
        self.save_predictions(processed, output_file)
        print("\n✅ v5.8-natural-language COMPLETE!")
        return output_file

    def generate_response_with_system(
        self, system: str, user: str, temperature: float = 0.0, max_tokens: int = 360
    ) -> str:
        headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }
        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                time.sleep(5)
        return ""

    def generate_response(self, prompt: str, temperature: float = 0.0, max_tokens: int = 360) -> str:
        return self.generate_response_with_system("You are a helpful assistant.", prompt, temperature, max_tokens)


# =========================
# MAIN EXECUTION
# =========================
INPUT_FILE = "/content/drive/MyDrive/Project/test_set_final_ensemble/test_set_TOP10_RAG_taskC.jsonl"
PREDICTIONS_OUTPUT = "/content/drive/MyDrive/Project/outputs/deepseek_RAG_v58_natural_test.jsonl"

generator = MTRAGGenerationV58NaturalLanguage()
generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=None)

print("\n" + "="*70)
print("✅ v5.8-natural-language COMPLETE!")
print("="*70)
print("\n🔑 ONLY CHANGE from v5.8:")
print("• 🔥 Avoid meta-phrases: 'based on the extracted facts', 'according to', etc.")
print("• All other v5.8 features unchanged")
print("\n📈 Expected Results:")
print("• RB_llm: 0.66-0.68 → 0.68-0.70 (more natural language)")
print("• Other metrics: same as v5.8")
print("="*70)


In [ ]:
# ==============================================================================
# CLEAN SUBMISSION FILE - REMOVE DEBUG FIELDS
# Removes: contexts_used, debug_info, and other optional metadata
# Keeps only: task_id, Collection, input, contexts, predictions
# ==============================================================================

import json
import os
from tqdm.auto import tqdm


BASE_PATH = "/content/drive/MyDrive/Project"


# Input/Output
INPUT_FILE = f"{BASE_PATH}/outputs/AILS-NTUA_taskC.jsonl"
OUTPUT_FILE = f"{BASE_PATH}/outputs/AILS-NTUA_taskC_CLEAN.jsonl"
BACKUP_FILE = f"{BASE_PATH}/outputs/AILS-NTUA_taskC_BACKUP.jsonl"  # Safety backup


print("=" * 80)
print("🧹 CLEANING SUBMISSION FILE")
print("=" * 80)
print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print(f"Backup: {BACKUP_FILE}")
print("=" * 80)


# Fields to KEEP (Task C required + optional standard fields)
REQUIRED_FIELDS = [
    "task_id",
    "Collection",
    "input",
    "contexts",
    "predictions"
]

# Optional standard fields (keep if you want)
OPTIONAL_KEEP_FIELDS = [
    "conversation_id",  # Optional but standard
    "task_type",        # Optional but standard
    "turn"              # Optional but standard
]

KEEP_FIELDS = REQUIRED_FIELDS + OPTIONAL_KEEP_FIELDS


# Fields to REMOVE (debug/internal fields)
REMOVE_FIELDS = [
    "contexts_used",    # v5.7 debug
    "debug_info",       # v5.7 debug
    "spans_extracted",  # Internal
    "answerability",    # Internal
    "retry_count",      # Internal
    "judge_triggered",  # Internal
    "polished"          # Internal
]


print("\n📋 Configuration:")
print(f"   Keep fields:   {', '.join(KEEP_FIELDS)}")
print(f"   Remove fields: {', '.join(REMOVE_FIELDS)}")


# Create backup first
print("\n💾 Creating backup...")
import shutil
shutil.copy2(INPUT_FILE, BACKUP_FILE)
print(f"   ✅ Backup saved: {BACKUP_FILE}")


# Clean file
print("\n🧹 Cleaning file...")

cleaned_count = 0
removed_fields_stats = {}
total_tasks = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as outfile:
        for line in tqdm(infile, desc="Processing"):
            if not line.strip():
                continue

            data = json.loads(line)
            total_tasks += 1

            # Track removed fields
            for field in list(data.keys()):
                if field not in KEEP_FIELDS:
                    removed_fields_stats[field] = removed_fields_stats.get(field, 0) + 1

            # Create clean record with only allowed fields
            clean_data = {}
            for field in KEEP_FIELDS:
                if field in data:
                    clean_data[field] = data[field]

            # Write clean record
            outfile.write(json.dumps(clean_data, ensure_ascii=False) + "\n")
            cleaned_count += 1


print(f"\n✅ Cleaned {cleaned_count} tasks")


# Report removed fields
if removed_fields_stats:
    print("\n📊 Removed fields (occurrences):")
    for field, count in sorted(removed_fields_stats.items()):
        pct = (count / total_tasks * 100) if total_tasks > 0 else 0
        print(f"   • {field:20s}: {count:4d} ({pct:5.1f}%)")
else:
    print("\n✅ No debug fields found (file was already clean)")


# Verification
print("\n" + "=" * 80)
print("📋 VERIFICATION")
print("=" * 80)

# Compare file sizes
original_size = os.path.getsize(INPUT_FILE) / (1024 * 1024)
cleaned_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
size_reduction = original_size - cleaned_size
size_reduction_pct = (size_reduction / original_size * 100) if original_size > 0 else 0

print(f"\nFile sizes:")
print(f"   Original: {original_size:.2f} MB")
print(f"   Cleaned:  {cleaned_size:.2f} MB")
print(f"   Reduced:  {size_reduction:.2f} MB ({size_reduction_pct:.1f}%)")


# Verify first record
print("\n📄 First record (original):")
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    original = json.loads(f.readline())
    print(f"   Fields: {list(original.keys())}")

print("\n📄 First record (cleaned):")
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    cleaned = json.loads(f.readline())
    print(f"   Fields: {list(cleaned.keys())}")


# Verify prediction is intact
if "predictions" in cleaned and cleaned["predictions"]:
    pred_text = cleaned["predictions"][0]["text"]
    print(f"\n✅ Prediction intact:")
    print(f"   {pred_text[:100]}...")


# Count lines
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    clean_lines = sum(1 for line in f if line.strip())

print(f"\n✅ Task count preserved: {clean_lines}")


print("\n" + "=" * 80)
print("✅ CLEANING COMPLETE")
print("=" * 80)
print(f"\n📁 Files:")
print(f"   Original (preserved): {INPUT_FILE}")
print(f"   Backup:               {BACKUP_FILE}")
print(f"   Clean (use this):     {OUTPUT_FILE}")
print("\n💡 Next step: Run format checker on clean file")


# Format checker

In [ ]:
# ==============================================================================
# RUN OFFICIAL FORMAT CHECKER - TASK C (RAG)
# Validates final predictions file from v5.7 generation
# ==============================================================================

import os
import subprocess


BASE_PATH = "/content/drive/MyDrive/Project"


# Paths
FORMAT_CHECKER_SCRIPT = f"{BASE_PATH}/test_set/format_checker.py"
INPUT_FILE = f"{BASE_PATH}/test_set/rag_taskAC.jsonl"
PREDICTION_FILE = f"{BASE_PATH}/outputs/AILS-NTUA_taskC_CLEAN.jsonl"  # ✅ Your v5.7 output


print("=" * 80)
print("🔍 RUNNING OFFICIAL MT-RAG FORMAT CHECKER - TASK C")
print("=" * 80)
print(f"Format checker: {FORMAT_CHECKER_SCRIPT}")
print(f"Input file:     {INPUT_FILE}")
print(f"Prediction:     {PREDICTION_FILE}")
print(f"Mode:           rag_taskc")
print(f"Generator:      v5.7 (User-Satisfaction Judge)")
print("=" * 80)


# Verify files exist
files_to_check = {
    "Format checker": FORMAT_CHECKER_SCRIPT,
    "Input file": INPUT_FILE,
    "Prediction file": PREDICTION_FILE
}

missing_files = []
for name, path in files_to_check.items():
    if not os.path.exists(path):
        missing_files.append(f"❌ {name} not found: {path}")

if missing_files:
    print("\n⚠️ MISSING FILES:")
    for msg in missing_files:
        print(f"   {msg}")

    if not os.path.exists(FORMAT_CHECKER_SCRIPT):
        print("\n💡 Make sure you have downloaded the MT-RAG evaluation scripts:")
        print("   https://github.com/primeqa/mtrag")

    if not os.path.exists(PREDICTION_FILE):
        print("\n💡 Make sure you've run the v5.7 generation script first!")
        print(f"   Expected output: {PREDICTION_FILE}")

    exit(1)

print("\n✅ All required files found")


# Quick pre-check: Verify predictions field exists
print("\n🔍 Pre-checking file structure...")
try:
    with open(PREDICTION_FILE, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        if first_line.strip():
            import json
            sample = json.loads(first_line)

            # Check required fields
            required_fields = ["task_id", "Collection", "input", "contexts", "predictions"]
            missing = [f for f in required_fields if f not in sample]

            if missing:
                print(f"\n❌ MISSING REQUIRED FIELDS: {', '.join(missing)}")
                print("\n⚠️ File structure:")
                print(f"   Found fields: {list(sample.keys())}")
                print("\n💡 Expected Task C format:")
                print('   {"task_id": str, "Collection": str, "input": [...], "contexts": [...], "predictions": [{"text": str}]}')
                exit(1)

            # Verify predictions structure
            if "predictions" in sample:
                preds = sample["predictions"]
                if not isinstance(preds, list):
                    print(f"\n❌ ERROR: 'predictions' must be a list, got {type(preds)}")
                    exit(1)
                if len(preds) == 0:
                    print(f"\n⚠️ WARNING: 'predictions' is empty on first task")
                elif "text" not in preds[0]:
                    print(f"\n❌ ERROR: predictions[0] missing 'text' field")
                    print(f"   Found keys: {list(preds[0].keys())}")
                    exit(1)
                else:
                    print(f"✅ Pre-check passed: All required fields present")
                    print(f"   Sample prediction: {preds[0]['text'][:60]}...")

except json.JSONDecodeError as e:
    print(f"\n❌ ERROR: Invalid JSON in prediction file")
    print(f"   Error: {e}")
    exit(1)
except Exception as e:
    print(f"\n⚠️ Pre-check warning: {e}")


# Run format checker
print("\n🔍 Running official format checker...\n")


cmd = [
    "python",
    FORMAT_CHECKER_SCRIPT,
    "--input_file", INPUT_FILE,
    "--prediction_file", PREDICTION_FILE,
    "--mode", "rag_taskc"  # ✅ TASK C MODE
]


try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=120  # ✅ Longer timeout (Task C file is larger)
    )

    # Print output
    print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    # Check exit code
    if result.returncode == 0:
        print("\n" + "=" * 80)
        print("✅ TASK C FORMAT CHECKER COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print("\n🎉 Your file is ready for Task C submission!")
        print(f"\n📁 Submission file: {PREDICTION_FILE}")

    else:
        print("\n" + "=" * 80)
        print(f"❌ FORMAT CHECKER FAILED (exit code: {result.returncode})")
        print("=" * 80)
        print("\n⚠️ Fix the errors above before submitting!")

except subprocess.TimeoutExpired:
    print("\n❌ ERROR: Format checker timed out after 120 seconds")
    print("   This might indicate the file is too large or has formatting issues")
except Exception as e:
    print(f"\n❌ ERROR: {e}")


# Additional statistics
print("\n" + "=" * 80)
print("📊 FILE STATISTICS")
print("=" * 80)

if os.path.exists(PREDICTION_FILE):
    file_size_mb = os.path.getsize(PREDICTION_FILE) / (1024 * 1024)

    # Count lines and analyze
    num_tasks = 0
    num_idk = 0
    total_words = 0

    with open(PREDICTION_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    num_tasks += 1

                    preds = data.get("predictions", [])
                    if preds and "text" in preds[0]:
                        text = preds[0]["text"]
                        words = len(text.split())
                        total_words += words

                        # Check if IDK
                        if "I do not have specific information" in text:
                            num_idk += 1
                except:
                    pass

    avg_words = total_words / num_tasks if num_tasks > 0 else 0
    idk_pct = (num_idk / num_tasks * 100) if num_tasks > 0 else 0

    print(f"File size:       {file_size_mb:.2f} MB")
    print(f"Total tasks:     {num_tasks}")
    print(f"IDK responses:   {num_idk} ({idk_pct:.1f}%)")
    print(f"Full answers:    {num_tasks - num_idk} ({100-idk_pct:.1f}%)")
    print(f"Avg word count:  {avg_words:.1f} words")

    if file_size_mb > 20:
        print(f"\n⚠️ WARNING: File size exceeds 20 MB limit!")
        print("   Competition may reject your submission")
    else:
        print(f"\n✅ File size is within 20 MB limit")


print("\n" + "=" * 80)
print("📁 FILES CHECKED")
print("=" * 80)
print(f"Input:      {INPUT_FILE}")
print(f"Prediction: {PREDICTION_FILE}")
print("=" * 80)


# Generation Task B

In [ ]:
import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================================
# CONFIGURATION - HYBRID: DeepSeek Extraction + GPT-4o-mini Generation
# ============================================================================
# 🎯 TEST SET SUBMISSION VERSION
# ============================================================================

# DeepSeek V3 (for span extraction & judges - cheaper)
AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# GPT-4o-mini (for natural generation - better instruction following)
AZURE_GPT_ENDPOINT = os.environ["ENDPOINT"]
AZURE_GPT_KEY = os.environ["_KEY"]


# 🧪 EXPERIMENT PARAMETERS
EXPERIMENT_NAME = "test_set_submission"
MAX_SPANS = 8
N_PASSAGES = 5


# 🎯 TEST MODE - FULL RUN
TEST_MODE = False  # Run on full test set


# Pipeline stages
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_DUAL_GENERATION = True
ENABLE_TECHNICAL_JUDGE = True
ENABLE_USER_SATISFACTION_JUDGE = True
ENABLE_FINAL_MICRO_ADJUSTMENTS = True


# 🎯 STRICT ANSWERABILITY RULE
STRICT_ANSWERABILITY_MODE = True


# Span extraction
MIN_SPANS = 1
SPAN_RETRY_ATTEMPTS = 2


# Generation
GENERATION_TEMP_V1 = 0.0
GENERATION_TEMP_V2 = 0.1


# Word count targets
SUGGESTED_MIN = 80
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 105


# Judge settings
JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150


# User satisfaction judge
USER_SAT_JUDGE_SAMPLE_RATE = 0.50


# Conversational and Unanswerable responses
MAX_WORDS_CONV = 50
MAX_WORDS_UNANSWERABLE = 25


# Adjustment criteria
ADJUSTMENT_MIN_WORD_DIFF = 10
ADJUSTMENT_MAX_LENGTH = 150
GENERATION_TEMP_ADJUSTMENT = 0.1


DEBUG_MODE = True
MAX_WORKERS = 5


random.seed(42)


# ============================================================================
# STATISTICS TRACKER
# ============================================================================


class StatsTracker:
    def __init__(self):
        self.total_input_ds = 0
        self.total_output_ds = 0
        self.total_input_gpt = 0
        self.total_output_gpt = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.conversational_count = 0
        self.span_retry_count = 0
        self.dual_generation_count = 0
        self.chose_v1 = 0
        self.chose_v2 = 0
        self.tech_judge_v1_wins = 0
        self.tech_judge_v2_wins = 0
        self.usersat_judge_v1_wins = 0
        self.usersat_judge_v2_wins = 0
        self.micro_adjustments_count = 0
        self.strict_idk_enforced = 0
        self.lock = threading.Lock()


    def add_usage_ds(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input_ds += prompt_tokens
            self.total_output_ds += completion_tokens


    def add_usage_gpt(self, prompt_tokens, completion_tokens):
        with self.lock:
            self.total_input_gpt += prompt_tokens
            self.total_output_gpt += completion_tokens


    def add_prediction(self, is_idk: bool):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1


    def add_conversational(self):
        with self.lock:
            self.conversational_count += 1


    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1


    def add_dual_generation(self):
        with self.lock:
            self.dual_generation_count += 1


    def add_choice(self, chose_v1: bool):
        with self.lock:
            if chose_v1:
                self.chose_v1 += 1
            else:
                self.chose_v2 += 1


    def add_tech_judge_result(self, v1_better: bool):
        with self.lock:
            if v1_better:
                self.tech_judge_v1_wins += 1
            else:
                self.tech_judge_v2_wins += 1


    def add_usersat_judge_result(self, v1_better: bool):
        with self.lock:
            if v1_better:
                self.usersat_judge_v1_wins += 1
            else:
                self.usersat_judge_v2_wins += 1


    def add_micro_adjustment(self):
        with self.lock:
            self.micro_adjustments_count += 1


    def add_strict_idk(self):
        with self.lock:
            self.strict_idk_enforced += 1


    def get_report(self):
        cost_ds = (self.total_input_ds * 0.20 / 1e6) + (self.total_output_ds * 0.50 / 1e6)
        cost_gpt = (self.total_input_gpt * 0.15 / 1e6) + (self.total_output_gpt * 0.60 / 1e6)
        total_cost = cost_ds + cost_gpt

        report = "=" * 70 + "\n"
        report += "🎯 TEST SET SUBMISSION\n"
        report += "=" * 70 + "\n"
        report += f"Token Usage:\n"
        report += f"  DeepSeek V3:\n"
        report += f"    Input: {self.total_input_ds:,} (${cost_ds:.4f})\n"
        report += f"    Output: {self.total_output_ds:,}\n"
        report += f"  GPT-4o-mini:\n"
        report += f"    Input: {self.total_input_gpt:,} (${cost_gpt:.4f})\n"
        report += f"    Output: {self.total_output_gpt:,}\n"
        report += f"  Total Cost: ${total_cost:.4f}\n"

        total_preds = self.idk_count + self.full_answer_count + self.conversational_count
        report += f"\nPredictions (Total: {total_preds}):\n"

        if total_preds > 0:
            idk_pct = (self.idk_count / total_preds * 100)
            ans_pct = (self.full_answer_count / total_preds * 100)
            conv_pct = (self.conversational_count / total_preds * 100)

            report += f"  Full Answers: {self.full_answer_count:4d} ({ans_pct:5.1f}%)\n"
            report += f"  IDK Responses: {self.idk_count:4d} ({idk_pct:5.1f}%)\n"
            report += f"  Conversational: {self.conversational_count:4d} ({conv_pct:5.1f}%)\n"

        if self.strict_idk_enforced > 0:
            report += f"\n🛡️ Strict IDK Enforced: {self.strict_idk_enforced}\n"

        if self.span_retry_count > 0:
            report += f"\nSpan Extraction Retries: {self.span_retry_count}\n"

        if self.dual_generation_count > 0:
            report += f"\n🆕 Dual Generation:\n"
            report += f"  Generated Both Versions: {self.dual_generation_count:4d}\n"

            total_choices = self.chose_v1 + self.chose_v2
            if total_choices > 0:
                v1_pct = (self.chose_v1 / total_choices * 100)
                v2_pct = (self.chose_v2 / total_choices * 100)
                report += f"\n  Final Choice:\n"
                report += f"    V1 (temp=0.0): {self.chose_v1:4d} ({v1_pct:5.1f}%)\n"
                report += f"    V2 (temp=0.1): {self.chose_v2:4d} ({v2_pct:5.1f}%)\n"

        report += "\n" + "=" * 70
        return report


tracker = StatsTracker()


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================


def word_count(text: str) -> int:
    return len([w for w in (text or "").strip().split() if w])


def safe_strip_quotes(s: str) -> str:
    s = (s or "").strip()
    if len(s) >= 2 and ((s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'")):
        return s[1:-1].strip()
    return s


def clean_model_output(text: str) -> str:
    t = safe_strip_quotes((text or "").strip())
    t = re.sub(r'(Agent|Assistant):', '', t.strip(), flags=re.IGNORECASE)
    t = re.sub(r'\s{3,}', ' ', t)
    t = re.sub(r'\n+', '\n', t)
    return t.strip()


def is_pure_idk(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    idk_indicators = [
        "i don't know", "i do not know", "do not have enough information",
        "don't have enough information", "do not have specific information",
        "don't have specific information", "insufficient information",
        "i cannot answer", "unable to answer"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 30:
        return True
    return False


def normalize_idk_partial_safe(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    if is_pure_idk(t):
        return t
    return t


def get_ngrams(text: str, n: int) -> set:
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))


def calculate_extractiveness(answer: str, contexts: List[Dict]) -> float:
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams) if answer_grams else 0.0


# ============================================================================
# MAIN GENERATION CLASS - HYBRID
# ============================================================================


class HybridGeneration:
    def __init__(self):
        print("=" * 70)
        print("🎯 TEST SET SUBMISSION")
        print("=" * 70)
        print("Pipeline:")
        print("  1️⃣  DeepSeek V3 → Span Extraction")
        print("  2️⃣  GPT-4o-mini → Dual Generation")
        print("  3️⃣  DeepSeek V3 → Technical Judge")
        print("  4️⃣  GPT-4o-mini → User Satisfaction Judge")
        print("  5️⃣  GPT-4o-mini → Micro-adjustments")
        print(f"\n⚙ Workers: {MAX_WORKERS}")
        print(f"⚙ MAX_SPANS: {MAX_SPANS}")
        print("=" * 70)

    # ========================================================================
    # LOADING & SAVING (Test set - no ground truth)
    # ========================================================================

    def load_tasks(self, input_file: str, max_tasks: int = None) -> List[Dict]:
        """Load test set (no ground truth)"""
        print(f"📂 Loading {input_file}...")
        tasks = []
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))

        if max_tasks:
            tasks = tasks[:max_tasks]

        print(f"✓ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks: List[Dict], output_file: str):
        """Save predictions for test set (clean format)"""
        print(f"💾 Saving {output_file}...")
        os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)

        with open(output_file, 'w', encoding='utf-8') as f:
            for task in tasks:
                # Clean output - only keep required fields
                output_task = {
                    "conversation_id": task.get("conversation_id", ""),
                    "task_id": task.get("task_id", ""),
                    "Collection": task.get("Collection", ""),
                    "input": task.get("input", []),
                    "contexts": task.get("contexts", []),
                    "predictions": task.get("predictions", [])
                }

                # Remove debug fields
                for key in list(output_task.keys()):
                    if key.startswith('_') or key == 'debug_info' or key == 'contexts_used':
                        if key in output_task:
                            del output_task[key]

                f.write(json.dumps(output_task, ensure_ascii=False) + '\n')

        print(f"✓ Saved {len(tasks)} predictions")

    def get_current_question(self, task: Dict) -> str:
        inputs = task.get("input", []) or []
        if not inputs:
            return ""
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def is_conversational(self, question: str) -> bool:
        q_low = question.lower().strip()
        conversational_patterns = [
            r'\bthanks?\b', r'\bthank you\b', r'\bok\b',
            r'\balright\b', r'^yes\??$', r'^no\??$',
            r'\bwhat about\b', r'\bhow about\b',
            r'\bgot it\b', r'\bi see\b', r'\bgreat\b'
        ]
        return any(re.search(p, q_low) for p in conversational_patterns)


    def build_history_lines(self, inputs: List[Dict], current_question: str) -> List[str]:
        history_lines = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()

            if not text:
                continue

            if speaker == "user" and text == current_question:
                break

            if speaker == "user":
                history_lines.append(f"User: {text}")
            elif speaker == "agent":
                history_lines.append(f"Assistant: {text}")

        return history_lines


    # ========================================================================
    # API CALLS - HYBRID
    # ========================================================================

    def generate_response_deepseek(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        """DeepSeek V3 API call"""
        headers = {
            "Content-Type": "application/json",
            "api-key": AZURE_DS_KEY
        }

        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        for attempt in range(7):
            try:
                response = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage_ds(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                if attempt < 6:
                    time.sleep(5)
                    continue
                else:
                    if DEBUG_MODE:
                        print(f"  ⚠ DeepSeek API error: {e}")
                    return ""

        return ""

    def generate_response_gpt(
        self,
        system: str,
        user: str,
        temperature: float = 0.0,
        max_tokens: int = 360
    ) -> str:
        """GPT-4o-mini API call"""
        headers = {
            "Content-Type": "application/json",
            "api-key": AZURE_GPT_KEY
        }

        payload = {
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        for attempt in range(7):
            try:
                response = requests.post(AZURE_GPT_ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    res_json = response.json()
                    usage = res_json.get("usage", {})
                    tracker.add_usage_gpt(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                    return res_json["choices"][0]["message"]["content"].strip()
                elif response.status_code == 429:
                    time.sleep(5 * (2 ** attempt))
                    continue
            except Exception as e:
                if attempt < 6:
                    time.sleep(5)
                    continue
                else:
                    if DEBUG_MODE:
                        print(f"  ⚠ GPT API error: {e}")
                    return ""

        return ""


    # ========================================================================
    # STAGE 0: STRICT TRIAGE
    # ========================================================================

    def generate_conversational_response(self, question: str, history_lines: List[str]) -> str:
        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-6:])
            history_block = f"\nCONVERSATION HISTORY:\n{recent}\n"

        prompt = f"""You are having a natural conversation with a user.

{history_block}USER: {question}

Generate a brief, friendly, conversational response (under {MAX_WORDS_CONV} words).
- Match the tone of the conversation
- Be natural and human-like
- Acknowledge what they said appropriately

RESPONSE:"""

        return self.generate_response_gpt("You are a helpful assistant.", prompt, temperature=0.3, max_tokens=160)

    def generate_strict_idk_response(self, question: str) -> str:
        """STRICT IDK: Short, clear response when NO contexts available"""
        prompt = f"""NO information is available to answer this question.

USER QUESTION: {question}

Generate a SHORT, clear response (under {MAX_WORDS_UNANSWERABLE} words) that:
- States clearly that the information is not available
- Is concise and direct
- Uses SIMPLE language

🚫 STRICTLY FORBIDDEN - DO NOT USE THESE PHRASES:
- "I don't know"
- "I do not know"
- "I'm not sure"
- "I am not sure"
- "I'm uncertain"
- "I cannot say"

✅ USE INSTEAD:
- "The information is not available."
- "No information found."
- "This information is not accessible."

RESPONSE:"""

        return self.generate_response_gpt("You are a helpful assistant.", prompt, temperature=0.0, max_tokens=100)


    # ========================================================================
    # STAGE 1: SPAN EXTRACTION (DeepSeek V3)
    # ========================================================================

    def extract_relevant_spans_with_retry(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> List[Dict]:
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                if DEBUG_MODE:
                    print(f"  ├─ Span retry {attempt}/{SPAN_RETRY_ATTEMPTS}")
                tracker.add_span_retry()

            spans = self.extract_relevant_spans_cot(question, contexts, history_lines, attempt)
            if spans and len(spans) >= MIN_SPANS:
                return spans

        if DEBUG_MODE:
            print("  ├─ Fallback to top 3 passages")
        return contexts[:3]

    def extract_relevant_spans_cot(
        self,
        question: str,
        contexts: List[Dict],
        history_lines: List[str],
        attempt: int = 1
    ) -> List[Dict]:
        passages_block = ""
        for i, ctx in enumerate(contexts[:N_PASSAGES], 1):
            text = ctx.get("text", "").strip()
            relevance_note = " [MOST RELEVANT]" if i == 1 else " [HIGHLY RELEVANT]" if i == 2 else ""
            passages_block += f"\n\nPASSAGE {i}{relevance_note}:\n{text}"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-6:])
            history_block = f"\n\nCONVERSATION HISTORY:\n{recent}\n"

        urgency = "⚠️ CRITICAL: The answer MUST be in the passages - extract it!" if attempt > 1 else ""

        span_prompt = f"""TASK: Extract sentences that answer the question.

🎯 CERTAINTY RULE: Since passages are provided, the answer MUST exist in them.

{urgency}
{history_block}QUESTION: {question}
{passages_block}

STRICT EXTRACTION RULES:
1. The answer IS in the passages - find it
2. Extract VERBATIM sentences (copy exactly)
3. PRIORITIZE earlier passages
4. Include ALL sentences with:
   - Proper nouns (names, places, organizations)
   - Numbers (dates, quantities, measurements)
   - Key facts needed for complete answer
5. Better to include 1 extra sentence than miss key entity
6. Max {MAX_SPANS} sentences
7. Empty extractedSpans ONLY if truly nothing relevant exists

Return JSON:
{{
  "reasoning": "Brief explanation",
  "extractedSpans": [
    {{"passageId": 1, "sentence": "exact sentence"}}
  ]
}}"""

        try:
            response = self.generate_response_deepseek("You are a helpful assistant.", span_prompt, temperature=0.0, max_tokens=800)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                spans = result.get("extractedSpans", [])

                if DEBUG_MODE and attempt == 1:
                    print(f"  ├─ Extracted {len(spans)} spans (DeepSeek)")

                if spans:
                    filtered = []
                    for span in spans[:MAX_SPANS]:
                        filtered.append({
                            "text": span["sentence"],
                            "document_id": f"passage_{span['passageId']}",
                            "source": "cot_extracted",
                            "reference": True
                        })
                    return filtered
        except Exception as e:
            if DEBUG_MODE:
                print(f"  ⚠ Span extraction failed: {e}")

        return []

    # ========================================================================
    # STAGE 2: NATURAL GENERATION (GPT-4o-mini)
    # ========================================================================

    def generate_natural_answer(
        self,
        spans: List[Dict],
        question: str,
        history_lines: List[str],
        temperature: float
    ) -> str:
        """Natural generation with GPT-4o-mini"""
        spans_block = "\n\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans, 1):
            text = span.get("text", "").strip()
            spans_block += f"{i}. {text}\n"

        history_block = ""
        if history_lines:
            recent = "\n".join(history_lines[-6:])
            history_block = f"\nCONVERSATION HISTORY:\n{recent}\n"

        instruction = f"""Generate a natural, accurate answer using ONLY the EXTRACTED FACTS below.

🛡️ CRITICAL RULES - GROUNDING:
✅ USE ONLY information from EXTRACTED FACTS
✅ Answer MUST be 100% based on the facts provided
✅ If facts don't fully answer the question, acknowledge limitation briefly
❌ ABSOLUTELY NO information from general knowledge
❌ ABSOLUTELY NO assumptions beyond the facts
❌ ABSOLUTELY NO inferences not directly stated

🎯 QUALITY GUIDELINES:
❌ FORBIDDEN PHRASES - DO NOT USE:
   - "based on", "according to", "the documents state", "the information shows"
   - "I don't know", "I'm not sure", "I cannot say"
   - "It appears", "It seems", "Possibly", "Maybe"

✅ WRITE DIRECTLY: State facts naturally as definite statements (but ONLY from EXTRACTED FACTS)

📝 EXTRACTIVENESS REQUIREMENTS:
✅ COPY VERBATIM: Proper nouns, numbers, dates, technical terms, specific claims
✅ Use EXACT 4-6 word phrases for key facts
✅ Paraphrase ONLY connecting words and transitions
✅ Target: ~30-40% verbatim overlap with facts
   - Below 20%: Risk of hallucination
   - Above 60%: Sounds robotic
   - Sweet spot (30-40%): Natural + safe

📏 LENGTH & COMPLETENESS:
- Target: {SUGGESTED_TARGET}w (range: {SUGGESTED_MIN}-{SUGGESTED_MAX}w)
- Include ALL key facts from EXTRACTED FACTS
- Be complete, natural, and conversational

{history_block}{spans_block}
QUESTION: {question}

ANSWER (using ONLY the EXTRACTED FACTS above):"""

        answer = self.generate_response_gpt("You are a helpful assistant.", instruction, temperature=temperature, max_tokens=450)
        return normalize_idk_partial_safe(clean_model_output(answer))


    # ========================================================================
    # STAGE 3: JUDGES
    # ========================================================================

    def technical_judge_comparison(
        self,
        v1: str,
        v2: str,
        question: str,
        spans: List[Dict]
    ) -> Dict:
        """Technical Judge with DeepSeek V3 (cheaper)"""
        v1_wc = word_count(v1)
        v2_wc = word_count(v2)
        v1_extr = calculate_extractiveness(v1, spans)
        v2_extr = calculate_extractiveness(v2, spans)

        spans_block = "\nEXTRACTED FACTS:\n"
        for i, span in enumerate(spans[:5], 1):
            spans_block += f"{i}. {span.get('text', '')[:100]}...\n"

        judge_prompt = f"""Technical quality comparison.

QUESTION: {question}
{spans_block}

VERSION A:
{v1}
Stats: {v1_wc}w, {v1_extr:.0%} extractiveness

VERSION B:
{v2}
Stats: {v2_wc}w, {v2_extr:.0%} extractiveness

EVALUATE:
1. Completeness (all key facts present?)
2. Length (50-150w ideal)
3. Naturalness (sounds human?)
4. Factual grounding (no hallucinations? uses exact phrases for key facts?)
5. Extractiveness balance (30-40% is ideal)

Return JSON:
{{
  "winner": "A|B",
  "reason": "Brief explanation",
  "score_A": 0-10,
  "score_B": 0-10
}}"""

        try:
            response = self.generate_response_deepseek("You are a helpful assistant.", judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                winner = result.get("winner", "B").upper()
                tracker.add_tech_judge_result(v1_better=(winner == "A"))
                return {
                    "winner": winner,
                    "reason": result.get("reason", ""),
                    "score_A": result.get("score_A", 5),
                    "score_B": result.get("score_B", 5)
                }
        except:
            pass

        tracker.add_tech_judge_result(v1_better=False)
        return {"winner": "B", "reason": "Default", "score_A": 5, "score_B": 6}

    def user_satisfaction_judge_comparison(
        self,
        v1: str,
        v2: str,
        question: str,
        contexts: List[Dict],
        history_lines: List[str]
    ) -> Dict:
        """User Satisfaction Judge with GPT-4o-mini (better at user perspective)"""
        history_block = ""
        if history_lines:
            history_block = f"\nCONTEXT: {' | '.join(history_lines[-4:])}\n"

        judge_prompt = f"""You are a USER who asked a question. Which answer do you prefer?

{history_block}YOUR QUESTION: {question}

ANSWER A:
{v1}

ANSWER B:
{v2}

As a user, consider:
1. Clarity
2. Completeness
3. Naturalness
4. Directness

Return JSON:
{{
  "preferred": "A|B",
  "reason": "Brief explanation",
  "confidence": "HIGH|MEDIUM|LOW"
}}"""

        try:
            response = self.generate_response_gpt("You are a helpful assistant.", judge_prompt, temperature=0.0, max_tokens=200)
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))
                preferred = result.get("preferred", "A").upper()
                tracker.add_usersat_judge_result(v1_better=(preferred == "A"))
                return {
                    "preferred": preferred,
                    "reason": result.get("reason", ""),
                    "confidence": result.get("confidence", "MEDIUM")
                }
        except:
            pass

        tracker.add_usersat_judge_result(v1_better=True)
        return {"preferred": "A", "reason": "Default", "confidence": "LOW"}


    # ========================================================================
    # STAGE 4: PICK BEST VERSION
    # ========================================================================

    def pick_best_version(
        self,
        v1: str,
        v2: str,
        tech_judgment: Dict,
        usersat_judgment: Optional[Dict]
    ) -> str:
        scores = {"A": 0, "B": 0}

        tech_score_A = tech_judgment.get("score_A", 5)
        tech_score_B = tech_judgment.get("score_B", 5)
        scores["A"] += tech_score_A * 0.4
        scores["B"] += tech_score_B * 0.4

        if usersat_judgment:
            usersat_preferred = usersat_judgment.get("preferred", "A")
            usersat_conf = usersat_judgment.get("confidence", "MEDIUM")

            conf_weight = {"HIGH": 1.0, "MEDIUM": 0.7, "LOW": 0.4}.get(usersat_conf, 0.7)
            if usersat_preferred == "A":
                scores["A"] += 6.0 * conf_weight
            else:
                scores["B"] += 6.0 * conf_weight
        else:
            scores["A"] += 3.0

        chosen = "A" if scores["A"] >= scores["B"] else "B"

        if DEBUG_MODE:
            print(f"  ├─ Scores: A={scores['A']:.1f}, B={scores['B']:.1f} → Chose {chosen}")

        tracker.add_choice(chose_v1=(chosen == "A"))

        return v1 if chosen == "A" else v2

    # ========================================================================
    # STAGE 5: MICRO-ADJUSTMENTS (GPT-4o-mini)
    # ========================================================================

    def apply_micro_adjustments(
        self,
        answer: str,
        question: str,
        spans: List[Dict]
    ) -> str:
        if not ENABLE_FINAL_MICRO_ADJUSTMENTS or is_pure_idk(answer):
            return answer

        wc = word_count(answer)

        needs_adjustment = False
        adjustment_reason = ""

        if wc < JUDGE_WC_MIN:
            needs_adjustment = True
            adjustment_reason = f"Too short ({wc}w) - expand slightly"
        elif wc > ADJUSTMENT_MAX_LENGTH:
            needs_adjustment = True
            adjustment_reason = f"Too long ({wc}w) - trim to {ADJUSTMENT_MAX_LENGTH}w"

        if not needs_adjustment:
            return answer

        if DEBUG_MODE:
            print(f"  ├─ Micro-adjustment: {adjustment_reason}")

        adjustment_prompt = f"""Make SMALL adjustments to this answer.

ISSUE: {adjustment_reason}

CURRENT ANSWER:
{answer}

TASK: Fix the issue with MINIMAL changes.
- Keep all facts
- Maintain naturalness
- Use exact phrases for key facts
- NO meta-phrases
- NO uncertain phrases

ADJUSTED ANSWER:"""

        adjusted = self.generate_response_gpt("You are a helpful assistant.", adjustment_prompt, temperature=GENERATION_TEMP_ADJUSTMENT, max_tokens=400)
        adjusted = normalize_idk_partial_safe(clean_model_output(adjusted))

        if adjusted and not is_pure_idk(adjusted) and word_count(adjusted) >= 30:
            tracker.add_micro_adjustment()
            return adjusted

        return answer


    # ========================================================================
    # MAIN PIPELINE
    # ========================================================================

    def process_task(self, task: Dict) -> Dict:
        """Main pipeline for test set"""
        inputs = task.get("input", []) or []
        question = self.get_current_question(task)
        history_lines = self.build_history_lines(inputs, current_question=question)

        contexts_original = task.get("contexts", [])[:N_PASSAGES]

        # STAGE 0: TRIAGE
        if not contexts_original or len(contexts_original) == 0:
            if self.is_conversational(question):
                raw = self.generate_conversational_response(question, history_lines)
                raw = clean_model_output(raw)
                task["contexts"] = []
                task["predictions"] = [{"text": raw}]
                tracker.add_conversational()
                return task
            else:
                idk_response = self.generate_strict_idk_response(question)
                idk_response = clean_model_output(idk_response)
                task["contexts"] = []
                task["predictions"] = [{"text": idk_response}]
                tracker.add_prediction(is_idk=True)
                tracker.add_strict_idk()
                return task

        # STAGE 1: Extract spans
        if ENABLE_SPAN_EXTRACTION:
            spans = self.extract_relevant_spans_with_retry(question, contexts_original, history_lines)
        else:
            spans = contexts_original[:3]

        if not spans:
            spans = contexts_original[:3]

        # STAGE 2: Dual generation
        if ENABLE_DUAL_GENERATION:
            v1 = self.generate_natural_answer(spans, question, history_lines, GENERATION_TEMP_V1)
            v2 = self.generate_natural_answer(spans, question, history_lines, GENERATION_TEMP_V2)

            tracker.add_dual_generation()

            if is_pure_idk(v1) and is_pure_idk(v2):
                force_answer = f"Based on the available information: {spans[0].get('text', '')[:100]}"
                task["contexts"] = contexts_original
                task["predictions"] = [{"text": force_answer}]
                tracker.add_prediction(is_idk=False)
                return task

            # STAGE 3: Judges
            tech_judgment = None
            usersat_judgment = None

            if ENABLE_TECHNICAL_JUDGE:
                tech_judgment = self.technical_judge_comparison(v1, v2, question, spans)

            if ENABLE_USER_SATISFACTION_JUDGE and random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
                usersat_judgment = self.user_satisfaction_judge_comparison(
                    v1, v2, question, contexts_original, history_lines
                )

            # STAGE 4: Pick best
            final_answer = self.pick_best_version(v1, v2, tech_judgment or {}, usersat_judgment)
        else:
            final_answer = self.generate_natural_answer(spans, question, history_lines, GENERATION_TEMP_V1)

        # STAGE 5: Micro-adjustments
        if ENABLE_FINAL_MICRO_ADJUSTMENTS:
            adjusted = self.apply_micro_adjustments(final_answer, question, spans)
            if adjusted != final_answer:
                final_answer = adjusted

        # Finalize
        final_answer = final_answer.strip()
        is_idk = is_pure_idk(final_answer)

        task["contexts"] = contexts_original
        task["predictions"] = [{"text": final_answer}]

        tracker.add_prediction(is_idk=is_idk)
        return task

    # ========================================================================
    # EXECUTION
    # ========================================================================

    def run_generation(self, input_file: str, output_file: str, max_tasks: int = None):
        print("=" * 70)
        print("🎯 TEST SET SUBMISSION - Starting...")
        print("=" * 70)

        tasks = self.load_tasks(input_file, max_tasks)

        print(f"\n🚀 Processing {len(tasks)} tasks with {MAX_WORKERS} workers...\n")

        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_idx = {executor.submit(self.process_task, task): idx for idx, task in enumerate(tasks)}

            for future in tqdm(as_completed(future_to_idx), total=len(tasks), desc="Test Set Pipeline"):
                idx = future_to_idx[future]
                try:
                    out_task = future.result()
                    processed[idx] = out_task

                    pred = out_task["predictions"][0]
                    wc = word_count(pred["text"])
                    tqdm.write(f"{idx+1}: {wc}w")
                except Exception as e:
                    tqdm.write(f"⚠ Task {idx+1} error: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)

        print("=" * 70)
        print("✅ TEST SET SUBMISSION COMPLETE!")
        print("=" * 70)

        return output_file


# ============================================================================
# MAIN ENTRY POINT - TEST SET
# ============================================================================

if __name__ == "__main__":
    # 🎯 TEST SET PATHS
    INPUT_FILE = "/content/drive/MyDrive/Project/test_set/reference_taskB.jsonl"
    PREDICTIONS_OUTPUT = "/content/drive/MyDrive/Project/test_set/submission_taskB.jsonl"

    print("=" * 70)
    print("🚀 TEST SET SUBMISSION")
    print("=" * 70)
    print(f"Input:  {INPUT_FILE}")
    print(f"Output: {PREDICTIONS_OUTPUT}")
    print("=" * 70)

    generator = HybridGeneration()
    generator.run_generation(INPUT_FILE, PREDICTIONS_OUTPUT, max_tasks=None)

    print("=" * 70)
    print("🎉 READY FOR SUBMISSION!")
    print(f"📁 Output file: {PREDICTIONS_OUTPUT}")
    print("=" * 70)

In [ ]:
import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================================
# FINAL SAFE v2 + GPT-4o GENERATION - TEST SET SUBMISSION
# ============================================================================
# Based on Final Safe v2 (HM=0.7233)
# GPT-4o for answer generation (Stage 2)
# All other stages: DeepSeek extraction, GPT-mini for other tasks
# Adapted for test set (no ground truth)
# ============================================================================

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

# GPT-4o-mini for conversational, IDK, judges, micro-adjustments
AZURE_GPT_MINI_ENDPOINT = os.environ["ENDPOINT"]
AZURE_GPT_MINI_KEY = os.environ["_KEY"]

# GPT-4o for answer generation ONLY
AZURE_GPT4O_ENDPOINT = os.environ["ENDPOINT"]
AZURE_GPT4O_KEY = os.environ["_KEY"]

# Parameters
EXPERIMENT_NAME = "final_safe_gpt4o_testset"
MAX_SPANS = 8
N_PASSAGES = 5
TEST_MODE = False  # Full test set run

# Pipeline - SAFE configuration
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_DUAL_GENERATION = True
ENABLE_TECHNICAL_JUDGE = True
ENABLE_USER_SATISFACTION_JUDGE = True
ENABLE_FINAL_MICRO_ADJUSTMENTS = True

# DISABLED - These broke unanswerable detection!
ENABLE_VALIDATION_JUDGE = False
ENABLE_QUALITY_BOOST = False

# Answerability - SAFE settings
STRICT_ANSWERABILITY_MODE = True
FORCE_ANSWER_WHEN_CONTEXTS_EXIST = True

# Same as Final Safe
MIN_SPANS = 1
SPAN_RETRY_ATTEMPTS = 2
GENERATION_TEMP_V1 = 0.0
GENERATION_TEMP_V2 = 0.1
SUGGESTED_MIN = 75
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 110

JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
USER_SAT_JUDGE_SAMPLE_RATE = 0.60

# Extractiveness targets
TARGET_EXTRACTIVENESS_MIN = 0.28
TARGET_EXTRACTIVENESS_IDEAL = 0.38
TARGET_EXTRACTIVENESS_MAX = 0.50

MAX_WORDS_CONV = 50
MAX_WORDS_UNANSWERABLE = 25
ADJUSTMENT_MAX_LENGTH = 150
GENERATION_TEMP_ADJUSTMENT = 0.1

DEBUG_MODE = True
MAX_WORKERS = 5
random.seed(42)

# Forbidden phrases
FORBIDDEN_PHRASES_CRITICAL = [
    "i don't know", "i do not know", "i'm not sure", "i am not sure",
    "i'm uncertain", "i cannot say", "it's unclear", "it is unclear",
    "i cannot answer", "unable to answer", "cannot find information",
]


class StatsTracker:
    def __init__(self):
        self.total_input_ds = 0
        self.total_output_ds = 0
        self.total_input_gpt_mini = 0
        self.total_output_gpt_mini = 0
        self.total_input_gpt4o = 0
        self.total_output_gpt4o = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.conversational_count = 0
        self.span_retry_count = 0
        self.dual_generation_count = 0
        self.chose_v1 = 0
        self.chose_v2 = 0
        self.tech_judge_v1_wins = 0
        self.tech_judge_v2_wins = 0
        self.usersat_judge_v1_wins = 0
        self.usersat_judge_v2_wins = 0
        self.micro_adjustments_count = 0
        self.strict_idk_enforced = 0
        self.forced_answer_count = 0
        self.extractiveness_boosted = 0
        self.question_type_counts = {}
        self.lock = threading.Lock()

    def add_usage_ds(self, p, c):
        with self.lock:
            self.total_input_ds += p
            self.total_output_ds += c

    def add_usage_gpt_mini(self, p, c):
        with self.lock:
            self.total_input_gpt_mini += p
            self.total_output_gpt_mini += c

    def add_usage_gpt4o(self, p, c):
        with self.lock:
            self.total_input_gpt4o += p
            self.total_output_gpt4o += c

    def add_prediction(self, is_idk):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1

    def add_conversational(self):
        with self.lock:
            self.conversational_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_dual_generation(self):
        with self.lock:
            self.dual_generation_count += 1

    def add_choice(self, v1):
        with self.lock:
            if v1:
                self.chose_v1 += 1
            else:
                self.chose_v2 += 1

    def add_tech_judge_result(self, v1):
        with self.lock:
            if v1:
                self.tech_judge_v1_wins += 1
            else:
                self.tech_judge_v2_wins += 1

    def add_usersat_judge_result(self, v1):
        with self.lock:
            if v1:
                self.usersat_judge_v1_wins += 1
            else:
                self.usersat_judge_v2_wins += 1

    def add_micro_adjustment(self):
        with self.lock:
            self.micro_adjustments_count += 1

    def add_strict_idk(self):
        with self.lock:
            self.strict_idk_enforced += 1

    def add_forced_answer(self):
        with self.lock:
            self.forced_answer_count += 1

    def add_extractiveness_boost(self):
        with self.lock:
            self.extractiveness_boosted += 1

    def add_question_type(self, qtype):
        with self.lock:
            self.question_type_counts[qtype] = self.question_type_counts.get(qtype, 0) + 1

    def get_report(self):
        cost_ds = (self.total_input_ds * 0.20 / 1e6) + (self.total_output_ds * 0.50 / 1e6)
        cost_gpt_mini = (self.total_input_gpt_mini * 0.15 / 1e6) + (self.total_output_gpt_mini * 0.60 / 1e6)
        cost_gpt4o = (self.total_input_gpt4o * 2.50 / 1e6) + (self.total_output_gpt4o * 10.00 / 1e6)
        total_cost = cost_ds + cost_gpt_mini + cost_gpt4o

        report = "=" * 70 + "\n"
        report += "🎯 FINAL SAFE v2 + GPT-4o - TEST SET\n"
        report += "=" * 70 + "\n"
        report += f"Cost:\n"
        report += f"  DeepSeek: ${cost_ds:.4f}\n"
        report += f"  GPT-mini: ${cost_gpt_mini:.4f}\n"
        report += f"  GPT-4o: ${cost_gpt4o:.4f}\n"
        report += f"  Total: ${total_cost:.4f}\n"

        total = self.idk_count + self.full_answer_count + self.conversational_count
        if total > 0:
            report += f"\nPredictions ({total} total):\n"
            report += f"  Answers: {self.full_answer_count} ({self.full_answer_count/total*100:.1f}%)\n"
            report += f"  IDK: {self.idk_count} ({self.idk_count/total*100:.1f}%)\n"
            report += f"  Conv: {self.conversational_count}\n"

        if self.forced_answer_count > 0:
            report += f"\nForced answers: {self.forced_answer_count}\n"
        if self.extractiveness_boosted > 0:
            report += f"Extractiveness boosts: {self.extractiveness_boosted}\n"

        total_choices = self.chose_v1 + self.chose_v2
        if total_choices > 0:
            report += f"\nVersion selection:\n"
            report += f"  V1: {self.chose_v1} ({self.chose_v1/total_choices*100:.1f}%)\n"
            report += f"  V2: {self.chose_v2} ({self.chose_v2/total_choices*100:.1f}%)\n"

        return report + "=" * 70


tracker = StatsTracker()


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def word_count(text):
    return len([w for w in (text or "").strip().split() if w])


def clean_model_output(text):
    t = (text or "").strip()
    if len(t) >= 2 and ((t[0] == '"' and t[-1] == '"') or (t[0] == "'" and t[-1] == "'")):
        t = t[1:-1].strip()
    t = re.sub(r'^(Agent|Assistant):\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()


def remove_critical_forbidden_phrases(text):
    result = text
    for phrase in FORBIDDEN_PHRASES_CRITICAL:
        pattern = re.compile(re.escape(phrase), re.IGNORECASE)
        result = pattern.sub('', result)
    result = re.sub(r'\s{2,}', ' ', result)
    result = re.sub(r'\s+([.,;:!?])', r'\1', result)
    return result.strip()


def contains_critical_forbidden(text):
    text_lower = (text or "").lower()
    return any(phrase in text_lower for phrase in FORBIDDEN_PHRASES_CRITICAL)


def is_pure_idk(text):
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    idk_indicators = [
        "i don't know", "i do not know", "do not have enough information",
        "don't have enough information", "do not have specific information",
        "don't have specific information", "insufficient information",
        "i cannot answer", "unable to answer", "cannot find",
        "no information available", "information is not available"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 35:
        return True
    return False


def get_ngrams(text, n):
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))


def calculate_extractiveness(answer, contexts):
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams)


def extractiveness_score(extr):
    if TARGET_EXTRACTIVENESS_MIN <= extr <= TARGET_EXTRACTIVENESS_IDEAL:
        return 2.5
    elif TARGET_EXTRACTIVENESS_IDEAL < extr <= TARGET_EXTRACTIVENESS_MAX:
        return 1.5
    elif extr < TARGET_EXTRACTIVENESS_MIN:
        return -1.5
    else:
        return 0.5


def detect_question_type(question):
    q_lower = question.lower().strip()
    if len(q_lower.split()) <= 3 and '?' not in question:
        return "keyword"
    if re.search(r'\bhow (do|can|should|to)\b', q_lower):
        return "how_to"
    if re.search(r'\b(why|explain|reason)\b', q_lower):
        return "explanation"
    if re.search(r'\b(compare|difference|vs|better)\b', q_lower):
        return "comparative"
    if re.search(r'\b(summar|overview)\b', q_lower):
        return "summarization"
    if re.search(r'\b(what|who|when|where|how many|how much)\b', q_lower):
        return "factoid"
    return "default"


QTYPE_CONFIG = {
    "factoid": {"hint": "Direct answer with specific fact.", "length_mod": -5},
    "explanation": {"hint": "Clear explanation.", "length_mod": +10},
    "how_to": {"hint": "Step-by-step.", "length_mod": +5},
    "summarization": {"hint": "Comprehensive summary.", "length_mod": +15},
    "comparative": {"hint": "Compare systematically.", "length_mod": +5},
    "keyword": {"hint": "Interpret and answer.", "length_mod": 0},
    "default": {"hint": "Complete answer.", "length_mod": 0}
}


# ============================================================================
# MAIN CLASS - TEST SET VERSION
# ============================================================================

class FinalSafeGPT4oTestSet:
    def __init__(self):
        print("=" * 70)
        print("🎯 FINAL SAFE v2 + GPT-4o - TEST SET SUBMISSION")
        print("   GPT-4o for answer generation (Stage 2)")
        print("   All other stages: DeepSeek + GPT-mini")
        print("=" * 70)

    # ========================================================================
    # DATA I/O - TEST SET (no ground truth)
    # ========================================================================

    def load_tasks(self, input_file, max_tasks=None):
        """Load test set (no ground truth)"""
        print(f"📂 Loading {input_file}...")
        tasks = []
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))

        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"✓ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks, output_file):
        """Save predictions in clean submission format"""
        print(f"💾 Saving {output_file}...")
        os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)

        with open(output_file, 'w', encoding='utf-8') as f:
            for task in tasks:
                # Clean output - only required fields for submission
                output_task = {
                    "conversation_id": task.get("conversation_id", ""),
                    "task_id": task.get("task_id", ""),
                    "Collection": task.get("Collection", ""),
                    "input": task.get("input", []),
                    "contexts": task.get("contexts", []),
                    "predictions": task.get("predictions", [])
                }

                f.write(json.dumps(output_task, ensure_ascii=False) + '\n')

        print(f"✓ Saved {len(tasks)} predictions")

    def get_question(self, task):
        inputs = task.get("input", []) or []
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def get_history(self, inputs, current_q):
        history = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_q:
                break
            prefix = "User: " if speaker == "user" else "Assistant: "
            history.append(prefix + text)
        return history

    def is_conversational(self, question):
        q = question.lower().strip()
        patterns = [
            r'^\s*thanks?\s*[.!]?\s*$', r'^\s*thank you\s*[.!]?\s*$',
            r'^\s*ok(ay)?\s*[.!]?\s*$', r'^\s*yes\s*[.!?]?\s*$',
            r'^\s*no\s*[.!?]?\s*$', r'^\s*got it\s*[.!]?\s*$',
            r'^\s*(great|cool|nice|alright)\s*[.!]?\s*$',
        ]
        return any(re.search(p, q) for p in patterns)

    # ========================================================================
    # API CALLS
    # ========================================================================

    def call_deepseek(self, system, user, temp=0.0, max_tokens=400):
        headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "max_tokens": max_tokens,
            "temperature": temp
        }
        for attempt in range(5):
            try:
                resp = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if resp.status_code == 200:
                    data = resp.json()
                    tracker.add_usage_ds(data.get("usage", {}).get("prompt_tokens", 0),
                                        data.get("usage", {}).get("completion_tokens", 0))
                    return data["choices"][0]["message"]["content"].strip()
                elif resp.status_code == 429:
                    time.sleep(3 * (2 ** attempt))
            except:
                time.sleep(2)
        return ""

    def call_gpt_mini(self, system, user, temp=0.0, max_tokens=400):
        """GPT-4o-mini for conversational, IDK, judges, micro-adjustments"""
        headers = {"Content-Type": "application/json", "api-key": AZURE_GPT_MINI_KEY}
        payload = {
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "max_tokens": max_tokens,
            "temperature": temp
        }
        for attempt in range(5):
            try:
                resp = requests.post(AZURE_GPT_MINI_ENDPOINT, headers=headers, json=payload, timeout=60)
                if resp.status_code == 200:
                    data = resp.json()
                    tracker.add_usage_gpt_mini(data.get("usage", {}).get("prompt_tokens", 0),
                                         data.get("usage", {}).get("completion_tokens", 0))
                    return data["choices"][0]["message"]["content"].strip()
                elif resp.status_code == 429:
                    time.sleep(3 * (2 ** attempt))
            except:
                time.sleep(2)
        return ""

    def call_gpt4o(self, system, user, temp=0.0, max_tokens=450):
        """GPT-4o for answer generation (Stage 2)"""
        headers = {"Content-Type": "application/json", "api-key": AZURE_GPT4O_KEY}
        payload = {
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "max_tokens": max_tokens,
            "temperature": temp
        }
        for attempt in range(5):
            try:
                resp = requests.post(AZURE_GPT4O_ENDPOINT, headers=headers, json=payload, timeout=60)
                if resp.status_code == 200:
                    data = resp.json()
                    tracker.add_usage_gpt4o(data.get("usage", {}).get("prompt_tokens", 0),
                                           data.get("usage", {}).get("completion_tokens", 0))
                    return data["choices"][0]["message"]["content"].strip()
                elif resp.status_code == 429:
                    time.sleep(3 * (2 ** attempt))
            except:
                time.sleep(2)
        return ""

    # ========================================================================
    # STAGE 0: TRIAGE
    # ========================================================================

    def gen_conversational(self, question, history):
        hist_block = "\n".join(history[-4:]) if history else ""
        prompt = f"""Brief friendly response to conversational message.

{f"History: {hist_block}" if hist_block else ""}
User: {question}

Response (under {MAX_WORDS_CONV} words):"""
        return self.call_gpt_mini("Friendly assistant.", prompt, temp=0.3, max_tokens=100)

    def gen_idk(self, question):
        prompt = f"""No information available for this question.

Question: {question}

Short response (under {MAX_WORDS_UNANSWERABLE} words) stating information is not available.
Do NOT say "I don't know" - say "The information is not available" or similar.

Response:"""
        return self.call_gpt_mini("Helpful assistant.", prompt, temp=0.0, max_tokens=80)

    # ========================================================================
    # STAGE 1: SPAN EXTRACTION
    # ========================================================================

    def extract_spans(self, question, contexts, history):
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                tracker.add_span_retry()
            spans = self._extract_spans_impl(question, contexts, history, attempt)
            if spans and len(spans) >= MIN_SPANS:
                return spans
        return contexts[:3]

    def _extract_spans_impl(self, question, contexts, history, attempt):
        passages = ""
        for i, ctx in enumerate(contexts[:N_PASSAGES], 1):
            text = ctx.get("text", "").strip()
            passages += f"\n\nPASSAGE {i}:\n{text}"

        hist_block = "\n".join(history[-4:]) if history else ""
        urgency = "⚠️ Answer MUST exist - find it!" if attempt > 1 else ""

        prompt = f"""Extract sentences answering the question.

{urgency}
{f"History: {hist_block}" if hist_block else ""}
Question: {question}
{passages}

Rules:
1. Copy EXACT sentences
2. Include: names, numbers, dates, key facts
3. Max {MAX_SPANS} sentences
4. Prioritize earlier passages

JSON format:
{{"extractedSpans": [{{"passageId": 1, "sentence": "exact text"}}]}}"""

        try:
            resp = self.call_deepseek("Extract relevant sentences.", prompt, temp=0.0, max_tokens=700)
            match = re.search(r'\{.*\}', resp, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                spans = data.get("extractedSpans", [])
                if spans:
                    return [{"text": s["sentence"], "source": f"p{s['passageId']}"} for s in spans[:MAX_SPANS]]
        except:
            pass
        return []

    # ========================================================================
    # STAGE 2: GENERATION (GPT-4o)
    # ========================================================================

    def generate_answer(self, spans, question, history, temp, qtype="default"):
        """Uses GPT-4o for answer generation"""
        facts = "\n".join([f"• {s.get('text', '')}" for s in spans])
        hist_block = "\n".join(history[-4:]) if history else ""

        config = QTYPE_CONFIG.get(qtype, QTYPE_CONFIG["default"])
        target_wc = SUGGESTED_TARGET + config["length_mod"]

        prompt = f"""Generate a natural answer using ONLY the facts below.

FACTS:
{facts}

{f"Conversation context: {hist_block}" if hist_block else ""}

Question: {question}
Type: {qtype} - {config["hint"]}

CRITICAL RULES:
✅ Use ONLY information from FACTS above
✅ Copy exact phrases for: names, numbers, dates, technical terms
✅ Aim for ~35% verbatim overlap with facts
✅ Make it sound natural and complete
❌ NO outside knowledge
❌ NO hedging (seems, possibly, maybe)
❌ NO meta-phrases (based on, according to)

Length: ~{target_wc} words

Answer:"""

        # Using GPT-4o for generation
        answer = self.call_gpt4o("Helpful, accurate assistant.", prompt, temp=temp, max_tokens=450)
        return clean_model_output(answer)

    # ========================================================================
    # STAGE 3: JUDGES
    # ========================================================================

    def technical_judge(self, v1, v2, question, spans):
        v1_wc, v2_wc = word_count(v1), word_count(v2)
        v1_ex, v2_ex = calculate_extractiveness(v1, spans), calculate_extractiveness(v2, spans)

        facts = "\n".join([f"• {s.get('text', '')[:120]}" for s in spans[:5]])

        prompt = f"""Compare two answers for quality.

Question: {question}
Facts: {facts}

A: {v1}
   ({v1_wc}w, {v1_ex:.0%} extractiveness)

B: {v2}
   ({v2_wc}w, {v2_ex:.0%} extractiveness)

Evaluate: Faithfulness, Completeness, Naturalness
Ideal extractiveness: 28-45%

JSON: {{"winner": "A|B", "score_A": 0-10, "score_B": 0-10, "reason": "brief"}}"""

        try:
            resp = self.call_deepseek("Judge assistant.", prompt, temp=0.0, max_tokens=150)
            match = re.search(r'\{.*\}', resp, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                winner = data.get("winner", "B").upper()
                tracker.add_tech_judge_result(winner == "A")
                return {"winner": winner, "score_A": data.get("score_A", 5), "score_B": data.get("score_B", 5)}
        except:
            pass
        tracker.add_tech_judge_result(False)
        return {"winner": "B", "score_A": 5, "score_B": 6}

    def user_judge(self, v1, v2, question, history):
        hist = " | ".join(history[-3:]) if history else ""

        prompt = f"""As a user, which answer do you prefer?

{f"Context: {hist}" if hist else ""}
Question: {question}

A: {v1}

B: {v2}

JSON: {{"preferred": "A|B", "confidence": "HIGH|MEDIUM|LOW"}}"""

        try:
            resp = self.call_gpt_mini("User perspective.", prompt, temp=0.0, max_tokens=100)
            match = re.search(r'\{.*\}', resp, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                pref = data.get("preferred", "A").upper()
                tracker.add_usersat_judge_result(pref == "A")
                return {"preferred": pref, "confidence": data.get("confidence", "MEDIUM")}
        except:
            pass
        tracker.add_usersat_judge_result(True)
        return {"preferred": "A", "confidence": "LOW"}

    # ========================================================================
    # STAGE 4: SELECTION
    # ========================================================================

    def select_best(self, v1, v2, tech, user, spans):
        scores = {"A": 0, "B": 0}

        scores["A"] += tech.get("score_A", 5) * 0.35
        scores["B"] += tech.get("score_B", 5) * 0.35

        if user:
            pref = user.get("preferred", "A")
            conf = {"HIGH": 1.0, "MEDIUM": 0.7, "LOW": 0.4}.get(user.get("confidence", "MEDIUM"), 0.7)
            if pref == "A":
                scores["A"] += 5.0 * conf
            else:
                scores["B"] += 5.0 * conf
        else:
            scores["A"] += 2.0

        v1_ex = calculate_extractiveness(v1, spans)
        v2_ex = calculate_extractiveness(v2, spans)
        scores["A"] += extractiveness_score(v1_ex)
        scores["B"] += extractiveness_score(v2_ex)

        if contains_critical_forbidden(v1):
            scores["A"] -= 2.0
        if contains_critical_forbidden(v2):
            scores["B"] -= 2.0

        chosen = "A" if scores["A"] >= scores["B"] else "B"
        tracker.add_choice(chosen == "A")

        if DEBUG_MODE:
            print(f"  ├─ A={scores['A']:.1f} ({v1_ex:.0%}), B={scores['B']:.1f} ({v2_ex:.0%}) → {chosen}")

        return v1 if chosen == "A" else v2

    # ========================================================================
    # STAGE 5: MICRO ADJUSTMENTS
    # ========================================================================

    def micro_adjust(self, answer, question, spans):
        if is_pure_idk(answer):
            return answer

        wc = word_count(answer)
        extr = calculate_extractiveness(answer, spans)

        needs_fix = False
        reason = ""

        if wc < JUDGE_WC_MIN:
            needs_fix, reason = True, f"Too short ({wc}w)"
        elif wc > ADJUSTMENT_MAX_LENGTH:
            needs_fix, reason = True, f"Too long ({wc}w)"
        elif extr < TARGET_EXTRACTIVENESS_MIN:
            needs_fix, reason = True, f"Low extractiveness ({extr:.0%})"
            tracker.add_extractiveness_boost()

        if not needs_fix:
            return answer

        if DEBUG_MODE:
            print(f"  ├─ Adjusting: {reason}")

        facts = "\n".join([f"• {s.get('text', '')}" for s in spans[:5]])

        prompt = f"""Fix this answer: {reason}

Facts:
{facts}

Current: {answer}

Fix with minimal changes. Use exact phrases from facts.

Fixed:"""

        fixed = self.call_gpt_mini("Editor.", prompt, temp=0.1, max_tokens=350)
        fixed = clean_model_output(fixed)

        if fixed and not is_pure_idk(fixed) and word_count(fixed) >= 30:
            tracker.add_micro_adjustment()
            return fixed
        return answer

    # ========================================================================
    # MAIN PIPELINE
    # ========================================================================

    def process_task(self, task):
        inputs = task.get("input", []) or []
        question = self.get_question(task)
        history = self.get_history(inputs, question)
        contexts = task.get("contexts", [])[:N_PASSAGES]

        qtype = detect_question_type(question)
        tracker.add_question_type(qtype)

        # ===== TRIAGE =====
        if not contexts:
            if self.is_conversational(question):
                resp = clean_model_output(self.gen_conversational(question, history))
                task["contexts"] = []
                task["predictions"] = [{"text": resp}]
                tracker.add_conversational()
                return task
            else:
                # NO CONTEXTS = UNANSWERABLE
                resp = clean_model_output(self.gen_idk(question))
                task["contexts"] = []
                task["predictions"] = [{"text": resp}]
                tracker.add_prediction(True)
                tracker.add_strict_idk()
                return task

        # ===== HAS CONTEXTS - Generate answer =====

        # Extract spans
        spans = self.extract_spans(question, contexts, history) if ENABLE_SPAN_EXTRACTION else contexts[:3]
        if not spans:
            spans = contexts[:3]

        # Dual generation (GPT-4o)
        v1 = self.generate_answer(spans, question, history, GENERATION_TEMP_V1, qtype)
        v2 = self.generate_answer(spans, question, history, GENERATION_TEMP_V2, qtype)
        tracker.add_dual_generation()

        if DEBUG_MODE:
            print(f"  ├─ V1 (GPT-4o): {word_count(v1)}w, {calculate_extractiveness(v1, spans):.0%}")
            print(f"  ├─ V2 (GPT-4o): {word_count(v2)}w, {calculate_extractiveness(v2, spans):.0%}")

        # Handle both IDK case
        if is_pure_idk(v1) and is_pure_idk(v2) and FORCE_ANSWER_WHEN_CONTEXTS_EXIST:
            if DEBUG_MODE:
                print(f"  ⚠️ Both IDK but contexts exist - forcing answer")

            facts = "\n".join([f"• {s.get('text', '')}" for s in spans[:5]])
            force_prompt = f"""Answer using ONLY these facts. Do NOT say you cannot answer.

Facts:
{facts}

Question: {question}

Direct answer using the facts:"""

            forced = self.call_gpt_mini("Always answers when facts exist.", force_prompt, temp=0.2, max_tokens=400)
            forced = clean_model_output(forced)

            if forced and not is_pure_idk(forced):
                tracker.add_forced_answer()
                task["contexts"] = contexts
                task["predictions"] = [{"text": forced}]
                tracker.add_prediction(False)
                return task

        # Judges
        tech = self.technical_judge(v1, v2, question, spans) if ENABLE_TECHNICAL_JUDGE else {}
        user = None
        if ENABLE_USER_SATISFACTION_JUDGE and random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
            user = self.user_judge(v1, v2, question, history)

        # Select best
        answer = self.select_best(v1, v2, tech, user, spans)

        # Micro adjustments
        if ENABLE_FINAL_MICRO_ADJUSTMENTS:
            answer = self.micro_adjust(answer, question, spans)

        # Final cleanup
        answer = remove_critical_forbidden_phrases(answer.strip())

        is_idk = is_pure_idk(answer)

        task["contexts"] = contexts
        task["predictions"] = [{"text": answer}]

        tracker.add_prediction(is_idk)
        return task

    # ========================================================================
    # EXECUTION
    # ========================================================================

    def run(self, input_file, output_file, max_tasks=None):
        print("=" * 70)
        print("🎯 STARTING FINAL SAFE v2 + GPT-4o - TEST SET")
        print("=" * 70)

        tasks = self.load_tasks(input_file, max_tasks)
        print(f"\n🚀 Processing {len(tasks)} tasks...\n")

        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(self.process_task, t): i for i, t in enumerate(tasks)}

            for future in tqdm(as_completed(futures), total=len(tasks), desc="Test Set + GPT-4o"):
                idx = futures[future]
                try:
                    result = future.result()
                    processed[idx] = result

                    pred = result["predictions"][0]["text"]
                    wc = word_count(pred)
                    tqdm.write(f"{idx+1}: {wc}w")
                except Exception as e:
                    tqdm.write(f"⚠ Task {idx+1}: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)

        print("=" * 70)
        print("✅ TEST SET SUBMISSION COMPLETE!")
        print(f"📁 Output: {output_file}")
        print("=" * 70)

        return output_file


# ============================================================================
# MAIN - TEST SET
# ============================================================================

if __name__ == "__main__":
    # 🎯 TEST SET PATHS - CHANGE THESE TO YOUR PATHS
    INPUT_FILE = "/content/drive/MyDrive/Project/test_set/reference_taskB.jsonl"
    OUTPUT_FILE = "/content/drive/MyDrive/Project/test_set/submission_taskB_FINAL_SAFE_GPT4o.jsonl"

    print("=" * 70)
    print("🚀 FINAL SAFE v2 + GPT-4o - TEST SET SUBMISSION")
    print("=" * 70)
    print(f"Input:  {INPUT_FILE}")
    print(f"Output: {OUTPUT_FILE}")
    print("=" * 70)

    generator = FinalSafeGPT4oTestSet()
    generator.run(INPUT_FILE, OUTPUT_FILE, max_tasks=None)

    print("=" * 70)
    print("🎉 READY FOR SUBMISSION!")
    print("=" * 70)

In [ ]:
import os
import sys
import json
import re
import time
import random
import requests
import threading
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================================
# FINAL SAFE v2 - TEST SET SUBMISSION
# ============================================================================
# Based on FINAL_SAFE_v2 which achieved:
#   - Answerability: 100% (148/148)
#   - Harmonic Mean: 0.7233
#   - RL_F_idk: 0.9250
#   - RB_llm_idk: 0.7723
#
# Adapted for test set (no ground truth)
# ============================================================================

AZURE_DS_ENDPOINT = os.environ["ENDPOINT"]
AZURE_DS_KEY = os.environ["_KEY"]

AZURE_GPT_ENDPOINT = os.environ["ENDPOINT"]
AZURE_GPT_KEY = os.environ["_KEY"]

# Parameters - EXACTLY as FINAL_SAFE_v2
EXPERIMENT_NAME = "final_safe_v2_testset"
MAX_SPANS = 8
N_PASSAGES = 5
TEST_MODE = False  # Full test set run

# Pipeline - SAFE configuration (same as v2)
ENABLE_SPAN_EXTRACTION = True
ENABLE_SPAN_RETRY = True
ENABLE_DUAL_GENERATION = True
ENABLE_TECHNICAL_JUDGE = True
ENABLE_USER_SATISFACTION_JUDGE = True
ENABLE_FINAL_MICRO_ADJUSTMENTS = True

# DISABLED - These broke unanswerable detection!
ENABLE_VALIDATION_JUDGE = False
ENABLE_QUALITY_BOOST = False

# Answerability - SAFE settings
STRICT_ANSWERABILITY_MODE = True
FORCE_ANSWER_WHEN_CONTEXTS_EXIST = True

# Same as FINAL_SAFE_v2
MIN_SPANS = 1
SPAN_RETRY_ATTEMPTS = 2
GENERATION_TEMP_V1 = 0.0
GENERATION_TEMP_V2 = 0.1
SUGGESTED_MIN = 75
SUGGESTED_TARGET = 90
SUGGESTED_MAX = 110

JUDGE_WC_MIN = 50
JUDGE_WC_MAX = 150
USER_SAT_JUDGE_SAMPLE_RATE = 0.60  # Same as v2

# Extractiveness targets - key for RL_F
TARGET_EXTRACTIVENESS_MIN = 0.28
TARGET_EXTRACTIVENESS_IDEAL = 0.38
TARGET_EXTRACTIVENESS_MAX = 0.50

MAX_WORDS_CONV = 50
MAX_WORDS_UNANSWERABLE = 25
ADJUSTMENT_MAX_LENGTH = 150
GENERATION_TEMP_ADJUSTMENT = 0.1

DEBUG_MODE = True
MAX_WORKERS = 5
random.seed(42)

# Forbidden phrases
FORBIDDEN_PHRASES_CRITICAL = [
    "i don't know", "i do not know", "i'm not sure", "i am not sure",
    "i'm uncertain", "i cannot say", "it's unclear", "it is unclear",
    "i cannot answer", "unable to answer", "cannot find information",
]


class StatsTracker:
    def __init__(self):
        self.total_input_ds = 0
        self.total_output_ds = 0
        self.total_input_gpt = 0
        self.total_output_gpt = 0
        self.idk_count = 0
        self.full_answer_count = 0
        self.conversational_count = 0
        self.span_retry_count = 0
        self.dual_generation_count = 0
        self.chose_v1 = 0
        self.chose_v2 = 0
        self.tech_judge_v1_wins = 0
        self.tech_judge_v2_wins = 0
        self.usersat_judge_v1_wins = 0
        self.usersat_judge_v2_wins = 0
        self.micro_adjustments_count = 0
        self.strict_idk_enforced = 0
        self.forced_answer_count = 0
        self.extractiveness_boosted = 0
        self.question_type_counts = {}
        self.lock = threading.Lock()

    def add_usage_ds(self, p, c):
        with self.lock:
            self.total_input_ds += p
            self.total_output_ds += c

    def add_usage_gpt(self, p, c):
        with self.lock:
            self.total_input_gpt += p
            self.total_output_gpt += c

    def add_prediction(self, is_idk):
        with self.lock:
            if is_idk:
                self.idk_count += 1
            else:
                self.full_answer_count += 1

    def add_conversational(self):
        with self.lock:
            self.conversational_count += 1

    def add_span_retry(self):
        with self.lock:
            self.span_retry_count += 1

    def add_dual_generation(self):
        with self.lock:
            self.dual_generation_count += 1

    def add_choice(self, v1):
        with self.lock:
            if v1:
                self.chose_v1 += 1
            else:
                self.chose_v2 += 1

    def add_tech_judge_result(self, v1):
        with self.lock:
            if v1:
                self.tech_judge_v1_wins += 1
            else:
                self.tech_judge_v2_wins += 1

    def add_usersat_judge_result(self, v1):
        with self.lock:
            if v1:
                self.usersat_judge_v1_wins += 1
            else:
                self.usersat_judge_v2_wins += 1

    def add_micro_adjustment(self):
        with self.lock:
            self.micro_adjustments_count += 1

    def add_strict_idk(self):
        with self.lock:
            self.strict_idk_enforced += 1

    def add_forced_answer(self):
        with self.lock:
            self.forced_answer_count += 1

    def add_extractiveness_boost(self):
        with self.lock:
            self.extractiveness_boosted += 1

    def add_question_type(self, qtype):
        with self.lock:
            self.question_type_counts[qtype] = self.question_type_counts.get(qtype, 0) + 1

    def get_report(self):
        cost_ds = (self.total_input_ds * 0.20 / 1e6) + (self.total_output_ds * 0.50 / 1e6)
        cost_gpt = (self.total_input_gpt * 0.15 / 1e6) + (self.total_output_gpt * 0.60 / 1e6)
        total_cost = cost_ds + cost_gpt

        report = "=" * 70 + "\n"
        report += "🎯 FINAL SAFE v2 - TEST SET SUBMISSION\n"
        report += "=" * 70 + "\n"
        report += f"Cost: ${total_cost:.4f}\n"

        total = self.idk_count + self.full_answer_count + self.conversational_count
        if total > 0:
            report += f"\nPredictions ({total} total):\n"
            report += f"  Answers: {self.full_answer_count} ({self.full_answer_count/total*100:.1f}%)\n"
            report += f"  IDK: {self.idk_count} ({self.idk_count/total*100:.1f}%)\n"
            report += f"  Conv: {self.conversational_count}\n"

        if self.forced_answer_count > 0:
            report += f"\nForced answers: {self.forced_answer_count}\n"
        if self.extractiveness_boosted > 0:
            report += f"Extractiveness boosts: {self.extractiveness_boosted}\n"

        total_choices = self.chose_v1 + self.chose_v2
        if total_choices > 0:
            report += f"\nVersion selection:\n"
            report += f"  V1: {self.chose_v1} ({self.chose_v1/total_choices*100:.1f}%)\n"
            report += f"  V2: {self.chose_v2} ({self.chose_v2/total_choices*100:.1f}%)\n"

        return report + "=" * 70


tracker = StatsTracker()


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def word_count(text):
    return len([w for w in (text or "").strip().split() if w])


def clean_model_output(text):
    t = (text or "").strip()
    if len(t) >= 2 and ((t[0] == '"' and t[-1] == '"') or (t[0] == "'" and t[-1] == "'")):
        t = t[1:-1].strip()
    t = re.sub(r'^(Agent|Assistant):\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()


def remove_critical_forbidden_phrases(text):
    result = text
    for phrase in FORBIDDEN_PHRASES_CRITICAL:
        pattern = re.compile(re.escape(phrase), re.IGNORECASE)
        result = pattern.sub('', result)
    result = re.sub(r'\s{2,}', ' ', result)
    result = re.sub(r'\s+([.,;:!?])', r'\1', result)
    return result.strip()


def contains_critical_forbidden(text):
    text_lower = (text or "").lower()
    return any(phrase in text_lower for phrase in FORBIDDEN_PHRASES_CRITICAL)


def is_pure_idk(text):
    t = (text or "").strip()
    if not t:
        return False
    low = t.lower()
    idk_indicators = [
        "i don't know", "i do not know", "do not have enough information",
        "don't have enough information", "do not have specific information",
        "don't have specific information", "insufficient information",
        "i cannot answer", "unable to answer", "cannot find",
        "no information available", "information is not available"
    ]
    if any(ind in low for ind in idk_indicators) and word_count(t) <= 35:
        return True
    return False


def get_ngrams(text, n):
    words = text.lower().split()
    return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))


def calculate_extractiveness(answer, contexts):
    answer_grams = get_ngrams(answer, 4)
    if not answer_grams:
        return 0.0
    passage_text = " ".join(c.get("text", "") for c in contexts)
    passage_grams = get_ngrams(passage_text, 4)
    overlap = len(answer_grams & passage_grams)
    return overlap / len(answer_grams)


def extractiveness_score(extr):
    if TARGET_EXTRACTIVENESS_MIN <= extr <= TARGET_EXTRACTIVENESS_IDEAL:
        return 2.5
    elif TARGET_EXTRACTIVENESS_IDEAL < extr <= TARGET_EXTRACTIVENESS_MAX:
        return 1.5
    elif extr < TARGET_EXTRACTIVENESS_MIN:
        return -1.5
    else:
        return 0.5


def detect_question_type(question):
    q_lower = question.lower().strip()
    if len(q_lower.split()) <= 3 and '?' not in question:
        return "keyword"
    if re.search(r'\bhow (do|can|should|to)\b', q_lower):
        return "how_to"
    if re.search(r'\b(why|explain|reason)\b', q_lower):
        return "explanation"
    if re.search(r'\b(compare|difference|vs|better)\b', q_lower):
        return "comparative"
    if re.search(r'\b(summar|overview)\b', q_lower):
        return "summarization"
    if re.search(r'\b(what|who|when|where|how many|how much)\b', q_lower):
        return "factoid"
    return "default"


QTYPE_CONFIG = {
    "factoid": {"hint": "Direct answer with specific fact.", "length_mod": -5},
    "explanation": {"hint": "Clear explanation.", "length_mod": +10},
    "how_to": {"hint": "Step-by-step.", "length_mod": +5},
    "summarization": {"hint": "Comprehensive summary.", "length_mod": +15},
    "comparative": {"hint": "Compare systematically.", "length_mod": +5},
    "keyword": {"hint": "Interpret and answer.", "length_mod": 0},
    "default": {"hint": "Complete answer.", "length_mod": 0}
}


# ============================================================================
# MAIN CLASS - TEST SET VERSION
# ============================================================================

class FinalSafeTestSet:
    def __init__(self):
        print("=" * 70)
        print("🎯 FINAL SAFE v2 - TEST SET SUBMISSION")
        print("   Based on validation results: HM=0.7233, Answerability=100%")
        print("=" * 70)

    # ========================================================================
    # DATA I/O - TEST SET (no ground truth)
    # ========================================================================

    def load_tasks(self, input_file, max_tasks=None):
        """Load test set (no ground truth)"""
        print(f"📂 Loading {input_file}...")
        tasks = []
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    tasks.append(json.loads(line))

        if max_tasks:
            tasks = tasks[:max_tasks]
        print(f"✓ Loaded {len(tasks)} tasks")
        return tasks

    def save_predictions(self, tasks, output_file):
        """Save predictions in clean submission format"""
        print(f"💾 Saving {output_file}...")
        os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)

        with open(output_file, 'w', encoding='utf-8') as f:
            for task in tasks:
                # Clean output - only required fields for submission
                output_task = {
                    "conversation_id": task.get("conversation_id", ""),
                    "task_id": task.get("task_id", ""),
                    "Collection": task.get("Collection", ""),
                    "input": task.get("input", []),
                    "contexts": task.get("contexts", []),
                    "predictions": task.get("predictions", [])
                }

                f.write(json.dumps(output_task, ensure_ascii=False) + '\n')

        print(f"✓ Saved {len(tasks)} predictions")

    def get_question(self, task):
        inputs = task.get("input", []) or []
        for inp in reversed(inputs):
            if inp.get("speaker", "").lower() == "user":
                return (inp.get("text", "") or "").strip()
        return ""

    def get_history(self, inputs, current_q):
        history = []
        for inp in inputs:
            speaker = inp.get("speaker", "").lower()
            text = inp.get("text", "").strip()
            if not text:
                continue
            if speaker == "user" and text == current_q:
                break
            prefix = "User: " if speaker == "user" else "Assistant: "
            history.append(prefix + text)
        return history

    def is_conversational(self, question):
        q = question.lower().strip()
        patterns = [
            r'^\s*thanks?\s*[.!]?\s*$', r'^\s*thank you\s*[.!]?\s*$',
            r'^\s*ok(ay)?\s*[.!]?\s*$', r'^\s*yes\s*[.!?]?\s*$',
            r'^\s*no\s*[.!?]?\s*$', r'^\s*got it\s*[.!]?\s*$',
            r'^\s*(great|cool|nice|alright)\s*[.!]?\s*$',
        ]
        return any(re.search(p, q) for p in patterns)

    # ========================================================================
    # API CALLS
    # ========================================================================

    def call_deepseek(self, system, user, temp=0.0, max_tokens=400):
        headers = {"Content-Type": "application/json", "api-key": AZURE_DS_KEY}
        payload = {
            "model": "DeepSeek-V3.2",
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "max_tokens": max_tokens,
            "temperature": temp
        }
        for attempt in range(5):
            try:
                resp = requests.post(AZURE_DS_ENDPOINT, headers=headers, json=payload, timeout=60)
                if resp.status_code == 200:
                    data = resp.json()
                    tracker.add_usage_ds(data.get("usage", {}).get("prompt_tokens", 0),
                                        data.get("usage", {}).get("completion_tokens", 0))
                    return data["choices"][0]["message"]["content"].strip()
                elif resp.status_code == 429:
                    time.sleep(3 * (2 ** attempt))
            except:
                time.sleep(2)
        return ""

    def call_gpt(self, system, user, temp=0.0, max_tokens=400):
        headers = {"Content-Type": "application/json", "api-key": AZURE_GPT_KEY}
        payload = {
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "max_tokens": max_tokens,
            "temperature": temp
        }
        for attempt in range(5):
            try:
                resp = requests.post(AZURE_GPT_ENDPOINT, headers=headers, json=payload, timeout=60)
                if resp.status_code == 200:
                    data = resp.json()
                    tracker.add_usage_gpt(data.get("usage", {}).get("prompt_tokens", 0),
                                         data.get("usage", {}).get("completion_tokens", 0))
                    return data["choices"][0]["message"]["content"].strip()
                elif resp.status_code == 429:
                    time.sleep(3 * (2 ** attempt))
            except:
                time.sleep(2)
        return ""

    # ========================================================================
    # STAGE 0: TRIAGE
    # ========================================================================

    def gen_conversational(self, question, history):
        hist_block = "\n".join(history[-4:]) if history else ""
        prompt = f"""Brief friendly response to conversational message.

{f"History: {hist_block}" if hist_block else ""}
User: {question}

Response (under {MAX_WORDS_CONV} words):"""
        return self.call_gpt("Friendly assistant.", prompt, temp=0.3, max_tokens=100)

    def gen_idk(self, question):
        prompt = f"""No information available for this question.

Question: {question}

Short response (under {MAX_WORDS_UNANSWERABLE} words) stating information is not available.
Do NOT say "I don't know" - say "The information is not available" or similar.

Response:"""
        return self.call_gpt("Helpful assistant.", prompt, temp=0.0, max_tokens=80)

    # ========================================================================
    # STAGE 1: SPAN EXTRACTION
    # ========================================================================

    def extract_spans(self, question, contexts, history):
        for attempt in range(1, SPAN_RETRY_ATTEMPTS + 1):
            if attempt > 1:
                tracker.add_span_retry()
            spans = self._extract_spans_impl(question, contexts, history, attempt)
            if spans and len(spans) >= MIN_SPANS:
                return spans
        return contexts[:3]

    def _extract_spans_impl(self, question, contexts, history, attempt):
        passages = ""
        for i, ctx in enumerate(contexts[:N_PASSAGES], 1):
            text = ctx.get("text", "").strip()
            passages += f"\n\nPASSAGE {i}:\n{text}"

        hist_block = "\n".join(history[-4:]) if history else ""
        urgency = "⚠️ Answer MUST exist - find it!" if attempt > 1 else ""

        prompt = f"""Extract sentences answering the question.

{urgency}
{f"History: {hist_block}" if hist_block else ""}
Question: {question}
{passages}

Rules:
1. Copy EXACT sentences
2. Include: names, numbers, dates, key facts
3. Max {MAX_SPANS} sentences
4. Prioritize earlier passages

JSON format:
{{"extractedSpans": [{{"passageId": 1, "sentence": "exact text"}}]}}"""

        try:
            resp = self.call_deepseek("Extract relevant sentences.", prompt, temp=0.0, max_tokens=700)
            match = re.search(r'\{.*\}', resp, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                spans = data.get("extractedSpans", [])
                if spans:
                    return [{"text": s["sentence"], "source": f"p{s['passageId']}"} for s in spans[:MAX_SPANS]]
        except:
            pass
        return []

    # ========================================================================
    # STAGE 2: GENERATION
    # ========================================================================

    def generate_answer(self, spans, question, history, temp, qtype="default"):
        facts = "\n".join([f"• {s.get('text', '')}" for s in spans])
        hist_block = "\n".join(history[-4:]) if history else ""

        config = QTYPE_CONFIG.get(qtype, QTYPE_CONFIG["default"])
        target_wc = SUGGESTED_TARGET + config["length_mod"]

        prompt = f"""Generate a natural answer using ONLY the facts below.

FACTS:
{facts}

{f"Conversation context: {hist_block}" if hist_block else ""}

Question: {question}
Type: {qtype} - {config["hint"]}

CRITICAL RULES:
✅ Use ONLY information from FACTS above
✅ Copy exact phrases for: names, numbers, dates, technical terms
✅ Aim for ~35% verbatim overlap with facts
✅ Make it sound natural and complete
❌ NO outside knowledge
❌ NO hedging (seems, possibly, maybe)
❌ NO meta-phrases (based on, according to)

Length: ~{target_wc} words

Answer:"""

        answer = self.call_gpt("Helpful, accurate assistant.", prompt, temp=temp, max_tokens=450)
        return clean_model_output(answer)

    # ========================================================================
    # STAGE 3: JUDGES
    # ========================================================================

    def technical_judge(self, v1, v2, question, spans):
        v1_wc, v2_wc = word_count(v1), word_count(v2)
        v1_ex, v2_ex = calculate_extractiveness(v1, spans), calculate_extractiveness(v2, spans)

        facts = "\n".join([f"• {s.get('text', '')[:120]}" for s in spans[:5]])

        prompt = f"""Compare two answers for quality.

Question: {question}
Facts: {facts}

A: {v1}
   ({v1_wc}w, {v1_ex:.0%} extractiveness)

B: {v2}
   ({v2_wc}w, {v2_ex:.0%} extractiveness)

Evaluate: Faithfulness, Completeness, Naturalness
Ideal extractiveness: 28-45%

JSON: {{"winner": "A|B", "score_A": 0-10, "score_B": 0-10, "reason": "brief"}}"""

        try:
            resp = self.call_deepseek("Judge assistant.", prompt, temp=0.0, max_tokens=150)
            match = re.search(r'\{.*\}', resp, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                winner = data.get("winner", "B").upper()
                tracker.add_tech_judge_result(winner == "A")
                return {"winner": winner, "score_A": data.get("score_A", 5), "score_B": data.get("score_B", 5)}
        except:
            pass
        tracker.add_tech_judge_result(False)
        return {"winner": "B", "score_A": 5, "score_B": 6}

    def user_judge(self, v1, v2, question, history):
        hist = " | ".join(history[-3:]) if history else ""

        prompt = f"""As a user, which answer do you prefer?

{f"Context: {hist}" if hist else ""}
Question: {question}

A: {v1}

B: {v2}

JSON: {{"preferred": "A|B", "confidence": "HIGH|MEDIUM|LOW"}}"""

        try:
            resp = self.call_gpt("User perspective.", prompt, temp=0.0, max_tokens=100)
            match = re.search(r'\{.*\}', resp, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                pref = data.get("preferred", "A").upper()
                tracker.add_usersat_judge_result(pref == "A")
                return {"preferred": pref, "confidence": data.get("confidence", "MEDIUM")}
        except:
            pass
        tracker.add_usersat_judge_result(True)
        return {"preferred": "A", "confidence": "LOW"}

    # ========================================================================
    # STAGE 4: SELECTION
    # ========================================================================

    def select_best(self, v1, v2, tech, user, spans):
        scores = {"A": 0, "B": 0}

        scores["A"] += tech.get("score_A", 5) * 0.35
        scores["B"] += tech.get("score_B", 5) * 0.35

        if user:
            pref = user.get("preferred", "A")
            conf = {"HIGH": 1.0, "MEDIUM": 0.7, "LOW": 0.4}.get(user.get("confidence", "MEDIUM"), 0.7)
            if pref == "A":
                scores["A"] += 5.0 * conf
            else:
                scores["B"] += 5.0 * conf
        else:
            scores["A"] += 2.0

        v1_ex = calculate_extractiveness(v1, spans)
        v2_ex = calculate_extractiveness(v2, spans)
        scores["A"] += extractiveness_score(v1_ex)
        scores["B"] += extractiveness_score(v2_ex)

        if contains_critical_forbidden(v1):
            scores["A"] -= 2.0
        if contains_critical_forbidden(v2):
            scores["B"] -= 2.0

        chosen = "A" if scores["A"] >= scores["B"] else "B"
        tracker.add_choice(chosen == "A")

        if DEBUG_MODE:
            print(f"  ├─ A={scores['A']:.1f} ({v1_ex:.0%}), B={scores['B']:.1f} ({v2_ex:.0%}) → {chosen}")

        return v1 if chosen == "A" else v2

    # ========================================================================
    # STAGE 5: MICRO ADJUSTMENTS
    # ========================================================================

    def micro_adjust(self, answer, question, spans):
        if is_pure_idk(answer):
            return answer

        wc = word_count(answer)
        extr = calculate_extractiveness(answer, spans)

        needs_fix = False
        reason = ""

        if wc < JUDGE_WC_MIN:
            needs_fix, reason = True, f"Too short ({wc}w)"
        elif wc > ADJUSTMENT_MAX_LENGTH:
            needs_fix, reason = True, f"Too long ({wc}w)"
        elif extr < TARGET_EXTRACTIVENESS_MIN:
            needs_fix, reason = True, f"Low extractiveness ({extr:.0%})"
            tracker.add_extractiveness_boost()

        if not needs_fix:
            return answer

        if DEBUG_MODE:
            print(f"  ├─ Adjusting: {reason}")

        facts = "\n".join([f"• {s.get('text', '')}" for s in spans[:5]])

        prompt = f"""Fix this answer: {reason}

Facts:
{facts}

Current: {answer}

Fix with minimal changes. Use exact phrases from facts.

Fixed:"""

        fixed = self.call_gpt("Editor.", prompt, temp=0.1, max_tokens=350)
        fixed = clean_model_output(fixed)

        if fixed and not is_pure_idk(fixed) and word_count(fixed) >= 30:
            tracker.add_micro_adjustment()
            return fixed
        return answer

    # ========================================================================
    # MAIN PIPELINE
    # ========================================================================

    def process_task(self, task):
        inputs = task.get("input", []) or []
        question = self.get_question(task)
        history = self.get_history(inputs, question)
        contexts = task.get("contexts", [])[:N_PASSAGES]

        qtype = detect_question_type(question)
        tracker.add_question_type(qtype)

        # ===== TRIAGE =====
        # KEY: If no contexts, this is UNANSWERABLE - say IDK!
        if not contexts:
            if self.is_conversational(question):
                resp = clean_model_output(self.gen_conversational(question, history))
                task["contexts"] = []
                task["predictions"] = [{"text": resp}]
                tracker.add_conversational()
                return task
            else:
                # NO CONTEXTS = UNANSWERABLE
                resp = clean_model_output(self.gen_idk(question))
                task["contexts"] = []
                task["predictions"] = [{"text": resp}]
                tracker.add_prediction(True)
                tracker.add_strict_idk()
                return task

        # ===== HAS CONTEXTS - Generate answer =====

        # Extract spans
        spans = self.extract_spans(question, contexts, history) if ENABLE_SPAN_EXTRACTION else contexts[:3]
        if not spans:
            spans = contexts[:3]

        # Dual generation
        v1 = self.generate_answer(spans, question, history, GENERATION_TEMP_V1, qtype)
        v2 = self.generate_answer(spans, question, history, GENERATION_TEMP_V2, qtype)
        tracker.add_dual_generation()

        if DEBUG_MODE:
            print(f"  ├─ V1: {word_count(v1)}w, {calculate_extractiveness(v1, spans):.0%}")
            print(f"  ├─ V2: {word_count(v2)}w, {calculate_extractiveness(v2, spans):.0%}")

        # Handle both IDK case - force answer since contexts exist
        if is_pure_idk(v1) and is_pure_idk(v2) and FORCE_ANSWER_WHEN_CONTEXTS_EXIST:
            if DEBUG_MODE:
                print(f"  ⚠️ Both IDK but contexts exist - forcing answer")

            facts = "\n".join([f"• {s.get('text', '')}" for s in spans[:5]])
            force_prompt = f"""Answer using ONLY these facts. Do NOT say you cannot answer.

Facts:
{facts}

Question: {question}

Direct answer using the facts:"""

            forced = self.call_gpt("Always answers when facts exist.", force_prompt, temp=0.2, max_tokens=400)
            forced = clean_model_output(forced)

            if forced and not is_pure_idk(forced):
                tracker.add_forced_answer()
                task["contexts"] = contexts
                task["predictions"] = [{"text": forced}]
                tracker.add_prediction(False)
                return task

        # Judges
        tech = self.technical_judge(v1, v2, question, spans) if ENABLE_TECHNICAL_JUDGE else {}
        user = None
        if ENABLE_USER_SATISFACTION_JUDGE and random.random() < USER_SAT_JUDGE_SAMPLE_RATE:
            user = self.user_judge(v1, v2, question, history)

        # Select best
        answer = self.select_best(v1, v2, tech, user, spans)

        # Micro adjustments
        if ENABLE_FINAL_MICRO_ADJUSTMENTS:
            answer = self.micro_adjust(answer, question, spans)

        # Final cleanup
        answer = remove_critical_forbidden_phrases(answer.strip())

        is_idk = is_pure_idk(answer)

        task["contexts"] = contexts
        task["predictions"] = [{"text": answer}]

        tracker.add_prediction(is_idk)
        return task

    # ========================================================================
    # EXECUTION
    # ========================================================================

    def run(self, input_file, output_file, max_tasks=None):
        print("=" * 70)
        print("🎯 STARTING FINAL SAFE v2 - TEST SET")
        print("=" * 70)

        tasks = self.load_tasks(input_file, max_tasks)
        print(f"\n🚀 Processing {len(tasks)} tasks...\n")

        processed = [None] * len(tasks)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(self.process_task, t): i for i, t in enumerate(tasks)}

            for future in tqdm(as_completed(futures), total=len(tasks), desc="Test Set"):
                idx = futures[future]
                try:
                    result = future.result()
                    processed[idx] = result

                    pred = result["predictions"][0]["text"]
                    wc = word_count(pred)
                    tqdm.write(f"{idx+1}: {wc}w")
                except Exception as e:
                    tqdm.write(f"⚠ Task {idx+1}: {e}")
                    processed[idx] = tasks[idx]

        print(tracker.get_report())
        self.save_predictions(processed, output_file)

        print("=" * 70)
        print("✅ TEST SET SUBMISSION COMPLETE!")
        print(f"📁 Output: {output_file}")
        print("=" * 70)

        return output_file


# ============================================================================
# MAIN - TEST SET
# ============================================================================

if __name__ == "__main__":
    # 🎯 TEST SET PATHS - CHANGE THESE TO YOUR PATHS
    INPUT_FILE = "/content/drive/MyDrive/Project/test_set/reference_taskB.jsonl"
    OUTPUT_FILE = "/content/drive/MyDrive/Project/test_set/submission_taskB_SAFE_v2.jsonl"

    print("=" * 70)
    print("🚀 FINAL SAFE v2 - TEST SET SUBMISSION")
    print("=" * 70)
    print(f"Input:  {INPUT_FILE}")
    print(f"Output: {OUTPUT_FILE}")
    print("=" * 70)

    generator = FinalSafeTestSet()
    generator.run(INPUT_FILE, OUTPUT_FILE, max_tasks=None)

    print("=" * 70)
    print("🎉 READY FOR SUBMISSION!")
    print("=" * 70)

# Format checker

In [ ]:
# ==============================================================================
# RUN OFFICIAL FORMAT CHECKER - TASK B (GENERATION)
# Validates final predictions file for generation task
# ==============================================================================

import os
import subprocess
import json

BASE_PATH = "/content/drive/MyDrive/Project"

# Paths
FORMAT_CHECKER_SCRIPT = f"{BASE_PATH}/test_set/format_checker.py"
INPUT_FILE = f"{BASE_PATH}/test_set/reference_taskB.jsonl"  # ✅ Input για Task B (με contexts)
PREDICTION_FILE = f"{BASE_PATH}/test_set/submission_taskB.jsonl"  # ✅ Your generation output

print("=" * 80)
print("🔍 RUNNING OFFICIAL MT-RAG FORMAT CHECKER - TASK B")
print("=" * 80)
print(f"Format checker: {FORMAT_CHECKER_SCRIPT}")
print(f"Input file:     {INPUT_FILE}")
print(f"Prediction:     {PREDICTION_FILE}")
print(f"Mode:           generation_taskb")
print(f"Generator:      Hybrid DeepSeek + GPT-4o-mini")
print("=" * 80)

# Verify files exist
files_to_check = {
    "Format checker": FORMAT_CHECKER_SCRIPT,
    "Input file": INPUT_FILE,
    "Prediction file": PREDICTION_FILE
}

missing_files = []
for name, path in files_to_check.items():
    if not os.path.exists(path):
        missing_files.append(f"❌ {name} not found: {path}")

if missing_files:
    print("\n⚠️ MISSING FILES:")
    for msg in missing_files:
        print(f"   {msg}")

    if not os.path.exists(FORMAT_CHECKER_SCRIPT):
        print("\n💡 Make sure you have downloaded the MT-RAG evaluation scripts:")
        print("   https://github.com/primeqa/mtrag")

    if not os.path.exists(PREDICTION_FILE):
        print("\n💡 Make sure you've run the generation script first!")
        print(f"   Expected output: {PREDICTION_FILE}")

    exit(1)

print("\n✅ All required files found")

# Quick pre-check: Verify predictions field exists
print("\n🔍 Pre-checking file structure...")
try:
    with open(PREDICTION_FILE, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        if first_line.strip():
            sample = json.loads(first_line)

            # Check required fields for Task B
            required_fields = ["task_id", "input", "contexts", "predictions"]
            missing = [f for f in required_fields if f not in sample]

            if missing:
                print(f"\n❌ MISSING REQUIRED FIELDS: {', '.join(missing)}")
                print("\n⚠️ File structure:")
                print(f"   Found fields: {list(sample.keys())}")
                print("\n💡 Expected Task B format:")
                print('   {"task_id": str, "Collection": str, "input": [...], "contexts": [...], "predictions": [{"text": str}]}')
                exit(1)

            # Verify predictions structure
            if "predictions" in sample:
                preds = sample["predictions"]
                if not isinstance(preds, list):
                    print(f"\n❌ ERROR: 'predictions' must be a list, got {type(preds)}")
                    exit(1)
                if len(preds) == 0:
                    print(f"\n⚠️ WARNING: 'predictions' is empty on first task")
                elif "text" not in preds[0]:
                    print(f"\n❌ ERROR: predictions[0] missing 'text' field")
                    print(f"   Found keys: {list(preds[0].keys())}")
                    exit(1)
                else:
                    print(f"✅ Pre-check passed: All required fields present")
                    pred_text = preds[0]['text'][:80].replace('\n', ' ')
                    print(f"   Sample prediction: {pred_text}...")
                    print(f"   Word count: {len(preds[0]['text'].split())}w")

            # Check contexts
            if "contexts" in sample:
                contexts = sample["contexts"]
                if isinstance(contexts, list):
                    print(f"   Contexts: {len(contexts)} passages")
                    if len(contexts) == 0:
                        print(f"   ⚠️ WARNING: Empty contexts list")

except json.JSONDecodeError as e:
    print(f"\n❌ ERROR: Invalid JSON in prediction file")
    print(f"   Error: {e}")
    exit(1)
except Exception as e:
    print(f"\n⚠️ Pre-check warning: {e}")

# Run format checker
print("\n🔍 Running official format checker...\n")

cmd = [
    "python",
    FORMAT_CHECKER_SCRIPT,
    "--input_file", INPUT_FILE,
    "--prediction_file", PREDICTION_FILE,
    "--mode", "generation_taskb"  # ✅ TASK B MODE
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=120
    )

    # Print output
    print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    # Check exit code
    if result.returncode == 0:
        print("\n" + "=" * 80)
        print("✅ TASK B FORMAT CHECKER COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print("\n🎉 Your file is ready for Task B submission!")
        print(f"\n📁 Submission file: {PREDICTION_FILE}")

    else:
        print("\n" + "=" * 80)
        print(f"❌ FORMAT CHECKER FAILED (exit code: {result.returncode})")
        print("=" * 80)
        print("\n⚠️ Fix the errors above before submitting!")

except subprocess.TimeoutExpired:
    print("\n❌ ERROR: Format checker timed out after 120 seconds")
    print("   This might indicate the file is too large or has formatting issues")
except Exception as e:
    print(f"\n❌ ERROR: {e}")

# Additional statistics
print("\n" + "=" * 80)
print("📊 FILE STATISTICS")
print("=" * 80)

if os.path.exists(PREDICTION_FILE):
    file_size_mb = os.path.getsize(PREDICTION_FILE) / (1024 * 1024)

    # Count lines and analyze
    num_tasks = 0
    num_idk = 0
    num_conversational = 0
    total_words = 0
    word_counts = []

    with open(PREDICTION_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    num_tasks += 1

                    preds = data.get("predictions", [])
                    if preds and "text" in preds[0]:
                        text = preds[0]["text"]
                        words = len(text.split())
                        total_words += words
                        word_counts.append(words)

                        # Check response type
                        text_lower = text.lower()
                        if "information is not available" in text_lower or \
                           "no information found" in text_lower or \
                           "information is not accessible" in text_lower:
                            num_idk += 1
                        elif words < 30:
                            num_conversational += 1
                except:
                    pass

    avg_words = total_words / num_tasks if num_tasks > 0 else 0
    idk_pct = (num_idk / num_tasks * 100) if num_tasks > 0 else 0
    conv_pct = (num_conversational / num_tasks * 100) if num_tasks > 0 else 0
    full_answers = num_tasks - num_idk - num_conversational

    print(f"File size:       {file_size_mb:.2f} MB")
    print(f"Total tasks:     {num_tasks}")
    print(f"\n📝 Response Breakdown:")
    print(f"  Full answers:    {full_answers} ({(full_answers/num_tasks*100):.1f}%)")
    print(f"  IDK responses:   {num_idk} ({idk_pct:.1f}%)")
    print(f"  Conversational:  {num_conversational} ({conv_pct:.1f}%)")
    print(f"\n📏 Word Count Stats:")
    print(f"  Average:         {avg_words:.1f} words")
    if word_counts:
        print(f"  Min:             {min(word_counts)} words")
        print(f"  Max:             {max(word_counts)} words")
        print(f"  Median:          {sorted(word_counts)[len(word_counts)//2]} words")

    if file_size_mb > 20:
        print(f"\n⚠️ WARNING: File size exceeds 20 MB limit!")
        print("   Competition may reject your submission")
    else:
        print(f"\n✅ File size is within 20 MB limit")

print("\n" + "=" * 80)
print("📁 FILES CHECKED")
print("=" * 80)
print(f"Input:      {INPUT_FILE}")
print(f"Prediction: {PREDICTION_FILE}")
print("=" * 80)


In [ ]:
# ==============================================================================
# RUN OFFICIAL FORMAT CHECKER - TASK B (GENERATION)
# Validates final predictions file for generation task
# ==============================================================================

import os
import subprocess
import json

BASE_PATH = "/content/drive/MyDrive/Project"

# Paths
FORMAT_CHECKER_SCRIPT = f"{BASE_PATH}/test_set/format_checker.py"
INPUT_FILE = f"{BASE_PATH}/test_set/reference_taskB.jsonl"  # ✅ Input για Task B (με contexts)
PREDICTION_FILE = f"{BASE_PATH}/test_set/submission_taskB_FINAL_SAFE_GPT4o.jsonl"  # ✅ Your generation output

print("=" * 80)
print("🔍 RUNNING OFFICIAL MT-RAG FORMAT CHECKER - TASK B")
print("=" * 80)
print(f"Format checker: {FORMAT_CHECKER_SCRIPT}")
print(f"Input file:     {INPUT_FILE}")
print(f"Prediction:     {PREDICTION_FILE}")
print(f"Mode:           generation_taskb")
print(f"Generator:      Hybrid DeepSeek + GPT-4o-mini")
print("=" * 80)

# Verify files exist
files_to_check = {
    "Format checker": FORMAT_CHECKER_SCRIPT,
    "Input file": INPUT_FILE,
    "Prediction file": PREDICTION_FILE
}

missing_files = []
for name, path in files_to_check.items():
    if not os.path.exists(path):
        missing_files.append(f"❌ {name} not found: {path}")

if missing_files:
    print("\n⚠️ MISSING FILES:")
    for msg in missing_files:
        print(f"   {msg}")

    if not os.path.exists(FORMAT_CHECKER_SCRIPT):
        print("\n💡 Make sure you have downloaded the MT-RAG evaluation scripts:")
        print("   https://github.com/primeqa/mtrag")

    if not os.path.exists(PREDICTION_FILE):
        print("\n💡 Make sure you've run the generation script first!")
        print(f"   Expected output: {PREDICTION_FILE}")

    exit(1)

print("\n✅ All required files found")

# Quick pre-check: Verify predictions field exists
print("\n🔍 Pre-checking file structure...")
try:
    with open(PREDICTION_FILE, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        if first_line.strip():
            sample = json.loads(first_line)

            # Check required fields for Task B
            required_fields = ["task_id", "input", "contexts", "predictions"]
            missing = [f for f in required_fields if f not in sample]

            if missing:
                print(f"\n❌ MISSING REQUIRED FIELDS: {', '.join(missing)}")
                print("\n⚠️ File structure:")
                print(f"   Found fields: {list(sample.keys())}")
                print("\n💡 Expected Task B format:")
                print('   {"task_id": str, "Collection": str, "input": [...], "contexts": [...], "predictions": [{"text": str}]}')
                exit(1)

            # Verify predictions structure
            if "predictions" in sample:
                preds = sample["predictions"]
                if not isinstance(preds, list):
                    print(f"\n❌ ERROR: 'predictions' must be a list, got {type(preds)}")
                    exit(1)
                if len(preds) == 0:
                    print(f"\n⚠️ WARNING: 'predictions' is empty on first task")
                elif "text" not in preds[0]:
                    print(f"\n❌ ERROR: predictions[0] missing 'text' field")
                    print(f"   Found keys: {list(preds[0].keys())}")
                    exit(1)
                else:
                    print(f"✅ Pre-check passed: All required fields present")
                    pred_text = preds[0]['text'][:80].replace('\n', ' ')
                    print(f"   Sample prediction: {pred_text}...")
                    print(f"   Word count: {len(preds[0]['text'].split())}w")

            # Check contexts
            if "contexts" in sample:
                contexts = sample["contexts"]
                if isinstance(contexts, list):
                    print(f"   Contexts: {len(contexts)} passages")
                    if len(contexts) == 0:
                        print(f"   ⚠️ WARNING: Empty contexts list")

except json.JSONDecodeError as e:
    print(f"\n❌ ERROR: Invalid JSON in prediction file")
    print(f"   Error: {e}")
    exit(1)
except Exception as e:
    print(f"\n⚠️ Pre-check warning: {e}")

# Run format checker
print("\n🔍 Running official format checker...\n")

cmd = [
    "python",
    FORMAT_CHECKER_SCRIPT,
    "--input_file", INPUT_FILE,
    "--prediction_file", PREDICTION_FILE,
    "--mode", "generation_taskb"  # ✅ TASK B MODE
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=120
    )

    # Print output
    print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    # Check exit code
    if result.returncode == 0:
        print("\n" + "=" * 80)
        print("✅ TASK B FORMAT CHECKER COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print("\n🎉 Your file is ready for Task B submission!")
        print(f"\n📁 Submission file: {PREDICTION_FILE}")

    else:
        print("\n" + "=" * 80)
        print(f"❌ FORMAT CHECKER FAILED (exit code: {result.returncode})")
        print("=" * 80)
        print("\n⚠️ Fix the errors above before submitting!")

except subprocess.TimeoutExpired:
    print("\n❌ ERROR: Format checker timed out after 120 seconds")
    print("   This might indicate the file is too large or has formatting issues")
except Exception as e:
    print(f"\n❌ ERROR: {e}")

# Additional statistics
print("\n" + "=" * 80)
print("📊 FILE STATISTICS")
print("=" * 80)

if os.path.exists(PREDICTION_FILE):
    file_size_mb = os.path.getsize(PREDICTION_FILE) / (1024 * 1024)

    # Count lines and analyze
    num_tasks = 0
    num_idk = 0
    num_conversational = 0
    total_words = 0
    word_counts = []

    with open(PREDICTION_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    num_tasks += 1

                    preds = data.get("predictions", [])
                    if preds and "text" in preds[0]:
                        text = preds[0]["text"]
                        words = len(text.split())
                        total_words += words
                        word_counts.append(words)

                        # Check response type
                        text_lower = text.lower()
                        if "information is not available" in text_lower or \
                           "no information found" in text_lower or \
                           "information is not accessible" in text_lower:
                            num_idk += 1
                        elif words < 30:
                            num_conversational += 1
                except:
                    pass

    avg_words = total_words / num_tasks if num_tasks > 0 else 0
    idk_pct = (num_idk / num_tasks * 100) if num_tasks > 0 else 0
    conv_pct = (num_conversational / num_tasks * 100) if num_tasks > 0 else 0
    full_answers = num_tasks - num_idk - num_conversational

    print(f"File size:       {file_size_mb:.2f} MB")
    print(f"Total tasks:     {num_tasks}")
    print(f"\n📝 Response Breakdown:")
    print(f"  Full answers:    {full_answers} ({(full_answers/num_tasks*100):.1f}%)")
    print(f"  IDK responses:   {num_idk} ({idk_pct:.1f}%)")
    print(f"  Conversational:  {num_conversational} ({conv_pct:.1f}%)")
    print(f"\n📏 Word Count Stats:")
    print(f"  Average:         {avg_words:.1f} words")
    if word_counts:
        print(f"  Min:             {min(word_counts)} words")
        print(f"  Max:             {max(word_counts)} words")
        print(f"  Median:          {sorted(word_counts)[len(word_counts)//2]} words")

    if file_size_mb > 20:
        print(f"\n⚠️ WARNING: File size exceeds 20 MB limit!")
        print("   Competition may reject your submission")
    else:
        print(f"\n✅ File size is within 20 MB limit")

print("\n" + "=" * 80)
print("📁 FILES CHECKED")
print("=" * 80)
print(f"Input:      {INPUT_FILE}")
print(f"Prediction: {PREDICTION_FILE}")
print("=" * 80)
